In [ ]:
# === CELL 0: Install Step 5 dependencies (Moved to top) ===

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q shapely geopandas anndata h5py tqdm

import torch
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ==============================================================================
# CELL 1+2+3 — LOAD CORRECTED STEP 4 EXPORTS + VIEW FILES + VERIFY RAW COUNT MATRIX
# Replaces old Cells 1, 2, and 3.
# ==============================================================================

import numpy as np
import pandas as pd
from scipy import sparse
import anndata as ad
import json, os, gc, warnings
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ------------------------------------------------------------------
# Path setup
# ------------------------------------------------------------------

STEP4_EXPORT_DIR = "/content/drive/MyDrive/diffusion/step4_exports"

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

EXPORT_DIR = STEP4_EXPORT_DIR

if not os.path.exists(STEP4_EXPORT_DIR):
    fallback = "/content/drive/MyDrive/diffusion/latest_run"
    print(f"WARNING: {STEP4_EXPORT_DIR} not found.")
    print(f"Falling back to: {fallback}")
    STEP4_EXPORT_DIR = fallback
    EXPORT_DIR = STEP4_EXPORT_DIR

print("=" * 70)
print("STEP 5: Loading corrected Step 4 exports")
print("=" * 70)
print(f"STEP4_EXPORT_DIR: {STEP4_EXPORT_DIR}")
print(f"CHECKPOINT_DIR  : {CHECKPOINT_DIR}")

# ------------------------------------------------------------------
# Verify required files
# ------------------------------------------------------------------

required_files = [
    "molecules.parquet",
    "cell_data.npz",
    "denoised_adata.h5ad",
    "was_corrected.npy",
    "step4_config.json",
]

file_paths = {}
missing = []

print("\nRequired Step 4 export files:")
for fname in required_files:
    fpath = os.path.join(STEP4_EXPORT_DIR, fname)
    file_paths[fname] = fpath

    if os.path.exists(fpath):
        print(f"  ✓ {fname:35s} ({os.path.getsize(fpath)/1e6:.1f} MB)")
    else:
        print(f"  ✗ MISSING: {fname}")
        missing.append(fname)

if missing:
    raise FileNotFoundError(f"Missing required Step 4 export files: {missing}")

# ------------------------------------------------------------------
# Helper
# ------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

# ==============================================================================
# FILE 1: Load and view step4_config.json
# ==============================================================================

print("\n" + "=" * 70)
print("FILE 1: step4_config.json")
print("=" * 70)

with open(file_paths["step4_config.json"]) as f:
    step4_config = json.load(f)

print("Top-level keys in step4_config:")
print(list(step4_config.keys()))

print("\nStep 4 config preview:")
for key, value in step4_config.items():
    if isinstance(value, list):
        print(f"  {key}: list with {len(value):,} items")
        print(f"    first 10: {value[:10]}")
    elif isinstance(value, dict):
        print(f"  {key}: dictionary")
        for subkey, subval in value.items():
            print(f"    {subkey}: {subval}")
    else:
        print(f"  {key}: {value}")

shared_genes = list(step4_config["shared_genes"])
ct_column = step4_config.get("cell_type_column", "cell_type")
unique_cell_types = sorted([str(x) for x in step4_config.get("cell_types", [])])

print(f"\nShared genes from config: {len(shared_genes):,}")
print(f"Cell-type column from config: {ct_column}")
print(f"Cell types from config: {len(unique_cell_types):,}")

# ==============================================================================
# FILE 2: Load and view denoised_adata.h5ad
# ==============================================================================

print("\n" + "=" * 70)
print("FILE 2: denoised_adata.h5ad")
print("=" * 70)

denoised_adata = ad.read_h5ad(file_paths["denoised_adata.h5ad"])

print(f"denoised_adata shape: {denoised_adata.shape}")
print(f"Number of cells: {denoised_adata.n_obs:,}")
print(f"Number of genes: {denoised_adata.n_vars:,}")

print("\nFirst 10 cell IDs:")
print(list(denoised_adata.obs_names[:10]))

print("\nFirst 10 genes:")
print(list(denoised_adata.var_names[:10]))

print("\nobs columns:")
print(list(denoised_adata.obs.columns))

print("\nlayers:")
print(list(denoised_adata.layers.keys()))

print("\nobs preview:")
display(denoised_adata.obs.head())

print("\nvar preview:")
display(denoised_adata.var.head())

if ct_column not in denoised_adata.obs.columns:
    if "cell_type" in denoised_adata.obs.columns:
        ct_column = "cell_type"
    elif "Assigned_Xenium_Cell_Type" in denoised_adata.obs.columns:
        ct_column = "Assigned_Xenium_Cell_Type"
    else:
        raise KeyError("No valid cell-type column found in denoised_adata.obs.")

cell_types_series = denoised_adata.obs[ct_column].astype(str)

if not unique_cell_types:
    unique_cell_types = sorted(cell_types_series.unique().tolist())

print(f"\nFinal cell-type column used: {ct_column}")
print("\nCell-type counts:")
display(cell_types_series.value_counts().rename_axis("cell_type").reset_index(name="n_cells"))

ct_to_idx = {ct: i for i, ct in enumerate(unique_cell_types)}
cell_type_indices = cell_types_series.map(ct_to_idx).values.astype(np.int64)

X_denoised = ensure_dense(denoised_adata.X).astype(np.float32)

if "raw" not in denoised_adata.layers:
    raise KeyError("denoised_adata.layers['raw'] is missing.")

X_raw = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)

if "uncertainty" in denoised_adata.layers:
    uncertainty = ensure_dense(denoised_adata.layers["uncertainty"]).astype(np.float32)
else:
    uncertainty = None
    print("WARNING: uncertainty layer not found.")

cell_ids_step4 = np.array(denoised_adata.obs_names, dtype=str)

print("\nMatrix summaries:")
print(f"  X_denoised shape: {X_denoised.shape}")
print(f"  X_raw shape     : {X_raw.shape}")
print(f"  X_raw sum       : {X_raw.sum(dtype=np.float64):,.0f}")
print(f"  X_denoised sum  : {X_denoised.sum(dtype=np.float64):,.2f}")

if uncertainty is not None:
    print(f"  uncertainty shape: {uncertainty.shape}")
    print(f"  uncertainty min/mean/max: {uncertainty.min():.4f} / {uncertainty.mean():.4f} / {uncertainty.max():.4f}")

print("\nSmall raw count preview: first 5 cells × first 5 genes")
display(
    pd.DataFrame(
        X_raw[:5, :5],
        index=denoised_adata.obs_names[:5],
        columns=denoised_adata.var_names[:5]
    )
)

print("\nSmall denoised count preview: first 5 cells × first 5 genes")
display(
    pd.DataFrame(
        X_denoised[:5, :5],
        index=denoised_adata.obs_names[:5],
        columns=denoised_adata.var_names[:5]
    )
)

# ==============================================================================
# FILE 3: Load and view was_corrected.npy
# ==============================================================================

print("\n" + "=" * 70)
print("FILE 3: was_corrected.npy")
print("=" * 70)

was_corrected = np.load(file_paths["was_corrected.npy"])

if was_corrected.shape != X_denoised.shape:
    raise ValueError(
        f"was_corrected shape {was_corrected.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

was_corrected = was_corrected.astype(bool)

print(f"was_corrected shape: {was_corrected.shape}")
print(f"Total cell-gene pairs: {was_corrected.size:,}")
print(f"Corrected pairs: {was_corrected.sum():,}")
print(f"Correction rate: {100*was_corrected.sum()/was_corrected.size:.2f}%")

corrected_per_cell = was_corrected.sum(axis=1)
corrected_per_gene = was_corrected.sum(axis=0)

print("\nCorrected genes per cell summary:")
display(pd.Series(corrected_per_cell).describe().to_frame("corrected_genes_per_cell"))

print("\nTop 20 genes by number of corrected cells:")
display(
    pd.DataFrame({
        "gene_id": denoised_adata.var_names,
        "n_corrected_cells": corrected_per_gene
    })
    .sort_values("n_corrected_cells", ascending=False)
    .head(20)
)

# ==============================================================================
# FILE 4: Load and view molecules.parquet
# ==============================================================================

print("\n" + "=" * 70)
print("FILE 4: molecules.parquet")
print("=" * 70)

molecules = pd.read_parquet(file_paths["molecules.parquet"])

print(f"Molecules loaded: {len(molecules):,} rows")
print(f"Columns: {list(molecules.columns)}")

required_mol_cols = [
    "transcript_id",
    "cell_id",
    "gene_id",
    "x",
    "y",
    "z",
    "quality",
    "overlaps_nucleus",
    "Assigned_Xenium_Cell_Type",
]

missing_cols = [c for c in required_mol_cols if c not in molecules.columns]
if missing_cols:
    raise KeyError(f"molecules.parquet is missing required columns: {missing_cols}")

print("\nMolecule table preview:")
display(molecules.head())

print("\nMolecule table dtypes:")
display(molecules.dtypes.to_frame("dtype"))

print("\nClean molecule table checks:")
print(f"  total molecules: {len(molecules):,}")
print(f"  unique cells: {molecules['cell_id'].nunique():,}")
print(f"  unique genes: {molecules['gene_id'].nunique():,}")
print(f"  QV < 20: {(molecules['quality'] < 20).sum():,}")
print(f"  missing cell type: {molecules['Assigned_Xenium_Cell_Type'].isna().sum():,}")
print(f"  missing coordinates: {molecules[['x', 'y', 'z']].isna().any(axis=1).sum():,}")

print("\nMolecule quality summary:")
display(molecules["quality"].describe().to_frame("quality"))

print("\nTop 20 genes by molecule count:")
display(
    molecules["gene_id"]
    .value_counts()
    .head(20)
    .rename_axis("gene_id")
    .reset_index(name="n_molecules")
)

print("\nTop 20 cell types by molecule count:")
display(
    molecules["Assigned_Xenium_Cell_Type"]
    .value_counts()
    .head(20)
    .rename_axis("cell_type")
    .reset_index(name="n_molecules")
)

# ==============================================================================
# FILE 5: Load and view cell_data.npz
# ==============================================================================

print("\n" + "=" * 70)
print("FILE 5: cell_data.npz")
print("=" * 70)

cell_geom = np.load(file_paths["cell_data.npz"], allow_pickle=False)

print("Keys in cell_data.npz:")
print(list(cell_geom.keys()))

cell_ids_geom = cell_geom["cell_ids"]
centroids = cell_geom["centroids"]
cell_offsets = cell_geom["cell_offsets"]
cell_vertices = cell_geom["cell_vertices"]
nuc_offsets = cell_geom["nuc_offsets"]
nuc_vertices = cell_geom["nuc_vertices"]
nuc_present = cell_geom["nuc_present"]

print(f"\ncell_ids_geom shape: {cell_ids_geom.shape}")
print(f"centroids shape    : {centroids.shape}")
print(f"cell_offsets shape : {cell_offsets.shape}")
print(f"cell_vertices shape: {cell_vertices.shape}")
print(f"nuc_offsets shape  : {nuc_offsets.shape}")
print(f"nuc_vertices shape : {nuc_vertices.shape}")
print(f"nuc_present shape  : {nuc_present.shape}")

print(f"\nNumber of cells in geometry: {len(cell_ids_geom):,}")
print(f"Number of cell boundary vertices: {cell_vertices.shape[0]:,}")
print(f"Number of nucleus boundary vertices: {nuc_vertices.shape[0]:,}")
print(f"Cells with nucleus boundary present: {nuc_present.sum():,}")
print(f"Cells without nucleus boundary: {(~nuc_present.astype(bool)).sum():,}")

print("\nFirst 5 geometry rows:")
display(
    pd.DataFrame({
        "cell_id": cell_ids_geom[:5],
        "x_centroid": centroids[:5, 0],
        "y_centroid": centroids[:5, 1],
        "cell_offset_start": cell_offsets[:5],
        "nuc_offset_start": nuc_offsets[:5],
        "nuc_present": nuc_present[:5],
    })
)

del cell_geom
gc.collect()

# ------------------------------------------------------------------
# Lookup dictionaries
# ------------------------------------------------------------------

gene_to_col = {g: j for j, g in enumerate(shared_genes)}
gene_idx_map = gene_to_col

step4_cell_to_row = {int(c): i for i, c in enumerate(cell_ids_step4)}
cell_idx_map = step4_cell_to_row

print("\nLookup dictionaries created:")
print(f"  gene_to_col entries: {len(gene_to_col):,}")
print(f"  step4_cell_to_row entries: {len(step4_cell_to_row):,}")

# ------------------------------------------------------------------
# CHECK: Is X_raw a true molecule-count matrix?
# ------------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECK: Is X_raw a true molecule-count matrix?")
print("=" * 70)

count_source = molecules[
    molecules["cell_id"].isin(step4_cell_to_row.keys())
    & molecules["gene_id"].isin(gene_to_col.keys())
].copy()

if "status" in count_source.columns:
    print("status column found in molecule table; using status == 'observed' for raw count check.")
    count_source = count_source[count_source["status"] == "observed"].copy()

counts = (
    count_source.groupby(["cell_id", "gene_id"])
    .size()
    .reset_index(name="count")
)

print(f"Grouped molecule count table shape: {counts.shape}")
print("Grouped count preview:")
display(counts.head())

X_raw_counts = np.zeros(X_raw.shape, dtype=np.int32)

rows = counts["cell_id"].map(step4_cell_to_row)
cols = counts["gene_id"].map(gene_to_col)

ok = rows.notna() & cols.notna()

X_raw_counts[
    rows[ok].astype(int).to_numpy(),
    cols[ok].astype(int).to_numpy()
] = counts.loc[ok, "count"].to_numpy(dtype=np.int32)

raw_frac = np.abs(X_raw - np.round(X_raw))
diff = X_raw.astype(np.float64) - X_raw_counts.astype(np.float64)
abs_diff = np.abs(diff)

print(f"\nX_raw shape: {X_raw.shape}")
print(f"X_raw sum: {X_raw.sum(dtype=np.float64):,.0f}")
print(f"X_raw_counts sum from molecule table: {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"Max fractional part in X_raw: {raw_frac.max():.6f}")
print(f"Non-integer X_raw entries: {(raw_frac > 1e-6).sum():,} / {X_raw.size:,}")
print(f"Max abs diff X_raw vs molecule counts: {abs_diff.max():.6f}")
print(f"Pairs with abs diff > 1e-6: {(abs_diff > 1e-6).sum():,}")
print(f"Pairs with abs diff > 0.5: {(abs_diff > 0.5).sum():,}")

RAW_COUNTS_MATCH_X_RAW = bool(abs_diff.max() < 1e-6)

# ============================================================
# VIEW RAW-LAYER VS MOLECULE-COUNT MISMATCHES
# Insert before X_raw is replaced by X_raw_counts
# ============================================================

mismatch_rows, mismatch_cols = np.where(abs_diff > 1e-6)

print("=" * 70)
print("MISMATCH EXAMPLES: X_raw vs X_raw_counts")
print("=" * 70)
print(f"Total mismatched positions: {len(mismatch_rows):,}")

mismatch_preview = pd.DataFrame({
    "cell_id": denoised_adata.obs_names[mismatch_rows[:30]],
    "gene_id": denoised_adata.var_names[mismatch_cols[:30]],
    "X_raw_loaded": X_raw[mismatch_rows[:30], mismatch_cols[:30]],
    "X_raw_counts_from_molecules": X_raw_counts[mismatch_rows[:30], mismatch_cols[:30]],
    "abs_diff": abs_diff[mismatch_rows[:30], mismatch_cols[:30]],
})

display(mismatch_preview)

if RAW_COUNTS_MATCH_X_RAW:
    print("\nVERDICT: X_raw matches direct molecule-table counts.")
else:
    print("\nVERDICT: X_raw does NOT match direct molecule-table counts.")
    print("For coherence, Step 5 will use X_raw_counts rebuilt from the molecule table.")
    print("Also updating denoised_adata.layers['raw'] to X_raw_counts.")
    X_raw = X_raw_counts.astype(np.float32)
    denoised_adata.layers["raw"] = X_raw.copy()

# Backward-compatible aliases
X_observed_counts = X_raw_counts
X_raw_for_step5 = X_raw_counts

# ------------------------------------------------------------------
# Quick Step 5 target diagnostics
# ------------------------------------------------------------------

n_need_impute = int((np.round(X_denoised) > X_raw_counts).sum())
n_need_prune = int((np.round(X_denoised) < X_raw_counts).sum())

n_need_impute_corrected = int(((np.round(X_denoised) > X_raw_counts) & was_corrected).sum())
n_need_prune_corrected = int(((np.round(X_denoised) < X_raw_counts) & was_corrected).sum())

print("\nStep 5 count-direction diagnostics:")
print(f"  Pairs where round(X_denoised) > X_raw_counts: {n_need_impute:,}")
print(f"  Pairs where round(X_denoised) < X_raw_counts: {n_need_prune:,}")
print(f"  Corrected pairs needing imputation: {n_need_impute_corrected:,}")
print(f"  Corrected pairs needing pruning: {n_need_prune_corrected:,}")

print("\n" + "=" * 70)
print("All corrected Step 4 inputs loaded, viewed, and verified. Ready for Step 5.")
print("=" * 70)

In [ ]:
# ============================================================
# VIEW X_raw_counts MATRIX
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("VIEW X_raw_counts")
print("=" * 70)

# Convert a small part of X_raw_counts into a readable dataframe
X_raw_counts_preview = pd.DataFrame(
    X_raw_counts[:10, :10],
    index=denoised_adata.obs_names[:10],
    columns=denoised_adata.var_names[:10]
)

print("First 10 cells × first 10 genes from X_raw_counts:")
display(X_raw_counts_preview)

print("\nMatrix summary:")
print(f"Shape: {X_raw_counts.shape}")
print(f"Total molecule counts: {X_raw_counts.sum():,}")
print(f"Minimum value: {X_raw_counts.min()}")
print(f"Maximum value: {X_raw_counts.max()}")
print(f"Nonzero entries: {np.count_nonzero(X_raw_counts):,}")
print(f"Sparsity: {100 * (1 - np.count_nonzero(X_raw_counts) / X_raw_counts.size):.2f}%")

# Per-cell total counts
cell_total_counts = X_raw_counts.sum(axis=1)

print("\nPer-cell total count summary:")
display(pd.Series(cell_total_counts).describe().to_frame("raw_counts_per_cell"))

# Per-gene total counts
gene_total_counts = X_raw_counts.sum(axis=0)

gene_count_df = pd.DataFrame({
    "gene_id": denoised_adata.var_names,
    "total_raw_counts": gene_total_counts
}).sort_values("total_raw_counts", ascending=False)

print("\nTop 20 genes by total raw molecule count:")
display(gene_count_df.head(20))

In [ ]:
# ==============================================================================
# FIX AND SAVE denoised_adata.h5ad WITH CORRECT RAW GENE ORDER
# ==============================================================================

import os
import shutil
import numpy as np
import pandas as pd
import anndata as ad

print("=" * 70)
print("FIXING RAW LAYERS USING MOLECULE-DERIVED X_raw_counts")
print("=" * 70)

# ------------------------------------------------------------------
# 1. Sanity checks
# ------------------------------------------------------------------

assert "denoised_adata" in globals(), "denoised_adata is missing."
assert "X_raw_counts" in globals(), "X_raw_counts is missing."
assert "molecules" in globals(), "molecules dataframe is missing."
assert "STEP4_EXPORT_DIR" in globals(), "STEP4_EXPORT_DIR is missing."

if X_raw_counts.shape != denoised_adata.shape:
    raise ValueError(
        f"Shape mismatch: X_raw_counts {X_raw_counts.shape} "
        f"vs denoised_adata {denoised_adata.shape}"
    )

print(f"denoised_adata shape: {denoised_adata.shape}")
print(f"X_raw_counts shape  : {X_raw_counts.shape}")
print(f"X_raw_counts sum    : {X_raw_counts.sum(dtype=np.float64):,.0f}")

# ------------------------------------------------------------------
# 2. Rebuild one more molecule-count matrix for strict verification
# ------------------------------------------------------------------

gene_to_col_check = {g: j for j, g in enumerate(denoised_adata.var_names.astype(str))}
cell_to_row_check = {int(c): i for i, c in enumerate(denoised_adata.obs_names.astype(str))}

counts_check = (
    molecules[
        molecules["cell_id"].isin(cell_to_row_check.keys())
        & molecules["gene_id"].isin(gene_to_col_check.keys())
    ]
    .groupby(["cell_id", "gene_id"])
    .size()
    .reset_index(name="count")
)

X_check = np.zeros(denoised_adata.shape, dtype=np.int32)

rows = counts_check["cell_id"].map(cell_to_row_check)
cols = counts_check["gene_id"].map(gene_to_col_check)
ok = rows.notna() & cols.notna()

X_check[
    rows[ok].astype(int).to_numpy(),
    cols[ok].astype(int).to_numpy()
] = counts_check.loc[ok, "count"].to_numpy(dtype=np.int32)

max_diff_check = np.abs(X_check - X_raw_counts).max()

print(f"Verification max diff between X_check and X_raw_counts: {max_diff_check}")

if max_diff_check != 0:
    raise ValueError("X_raw_counts does not match molecule-table reconstruction. Do not save.")

# ------------------------------------------------------------------
# 3. Overwrite raw layers with corrected gene-order matrix
# ------------------------------------------------------------------

X_raw_fixed = X_raw_counts.astype(np.float32)

denoised_adata.layers["raw"] = X_raw_fixed.copy()
denoised_adata.layers["raw_molecule_counts_clean"] = X_raw_fixed.copy()

# Optional: keep old official raw layer as-is if present.
# Do not use raw_official_10x for Step 5 unless separately verified.
print("Updated:")
print("  denoised_adata.layers['raw']")
print("  denoised_adata.layers['raw_molecule_counts_clean']")

# ------------------------------------------------------------------
# 4. Verify fixed raw layer now matches X_raw_counts
# ------------------------------------------------------------------

fixed_raw = np.asarray(denoised_adata.layers["raw"])
abs_diff_fixed = np.abs(fixed_raw.astype(np.float64) - X_raw_counts.astype(np.float64))

print("\nPost-fix verification:")
print(f"  fixed raw sum: {fixed_raw.sum(dtype=np.float64):,.0f}")
print(f"  X_raw_counts sum: {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"  max abs diff: {abs_diff_fixed.max():.6f}")
print(f"  mismatched pairs: {(abs_diff_fixed > 1e-6).sum():,}")

if abs_diff_fixed.max() > 1e-6:
    raise ValueError("Fix failed: raw layer still does not match X_raw_counts.")

# ------------------------------------------------------------------
# 5. Backup old h5ad and save corrected h5ad
# ------------------------------------------------------------------

old_h5ad_path = os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad")
backup_h5ad_path = os.path.join(STEP4_EXPORT_DIR, "denoised_adata_BEFORE_RAW_FIX.h5ad")
fixed_h5ad_path = os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad")

if os.path.exists(old_h5ad_path) and not os.path.exists(backup_h5ad_path):
    print("\nCreating backup of old denoised_adata.h5ad...")
    shutil.copy2(old_h5ad_path, backup_h5ad_path)
    print(f"Backup saved to: {backup_h5ad_path}")
elif os.path.exists(backup_h5ad_path):
    print("\nBackup already exists, not overwriting it:")
    print(backup_h5ad_path)

print("\nSaving corrected denoised_adata.h5ad...")
denoised_adata.write_h5ad(fixed_h5ad_path)

print("\nSaved fixed h5ad:")
print(fixed_h5ad_path)

# ------------------------------------------------------------------
# 6. Reload fixed file and verify from disk
# ------------------------------------------------------------------

print("\nReloading fixed h5ad from disk for final verification...")
adata_test = ad.read_h5ad(fixed_h5ad_path)

raw_test = np.asarray(adata_test.layers["raw"])
abs_diff_disk = np.abs(raw_test.astype(np.float64) - X_raw_counts.astype(np.float64))

print("\nDisk verification:")
print(f"  raw layer sum: {raw_test.sum(dtype=np.float64):,.0f}")
print(f"  X_raw_counts sum: {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"  max abs diff: {abs_diff_disk.max():.6f}")
print(f"  mismatched pairs: {(abs_diff_disk > 1e-6).sum():,}")

if abs_diff_disk.max() < 1e-6:
    print("\nSUCCESS: Future Step 5 loads should no longer show raw-layer mismatch.")
else:
    raise ValueError("Disk verification failed: saved h5ad still mismatches.")

In [ ]:
# === CELL 4: Build cell geometry lookup ===

from shapely.geometry import Polygon, Point
from shapely import prepare
import time

print("=" * 60)
print("SUBSTEP 5A-prep: Building cell geometry lookup")
print("=" * 60)
t0 = time.time()

# Build mapping: cell_id -> index in geometry arrays
geom_id_to_idx = {int(cid): i for i, cid in enumerate(cell_ids_geom)}

# Build Shapely polygons for all cells (batched)
cell_polygons = {}
nuc_polygons = {}
cell_areas = {}
nuc_areas = {}

for i, cid in enumerate(cell_ids_geom):
    cid = int(cid)
    # Cell boundary
    cv = cell_vertices[cell_offsets[i]:cell_offsets[i+1]]
    if len(cv) >= 3:
        poly = Polygon(cv)
        if not poly.is_valid:
            poly = poly.buffer(0)
        cell_polygons[cid] = poly
        cell_areas[cid] = poly.area
    # Nucleus boundary
    if nuc_present[i]:
        nv = nuc_vertices[nuc_offsets[i]:nuc_offsets[i+1]]
        if len(nv) >= 3:
            npoly = Polygon(nv)
            if not npoly.is_valid:
                npoly = npoly.buffer(0)
            nuc_polygons[cid] = npoly
            nuc_areas[cid] = npoly.area

print(f"Built {len(cell_polygons):,} cell polygons, "
      f"{len(nuc_polygons):,} nucleus polygons in {time.time()-t0:.1f}s")


In [ ]:
#Cell 6: coordinate normalization
import numpy as np
from shapely.geometry import Point, Polygon, MultiPolygon, LineString
from tqdm import tqdm
import time

print("=" * 60)
print("SUBSTEP 5A: Cell-relative coordinate normalization")
print("=" * 60)
t0 = time.time()

# Helper function to get minimum distance to polygon exterior, handling MultiPolygon
def get_min_exterior_distance(geom, point):
    if isinstance(geom, Polygon):
        return geom.exterior.distance(point)
    elif isinstance(geom, MultiPolygon):
        min_dist = float('inf')
        for poly in geom.geoms:
            min_dist = min(min_dist, poly.exterior.distance(point))
        return min_dist
    return float('inf') # Should not happen if data is clean

# Build centroid lookup
centroid_lookup = {}
for i, cid in enumerate(cell_ids_geom):
    centroid_lookup[int(cid)] = centroids[i]

# Filter molecules to only cells with geometry AND in denoised_adata
valid_cell_ids = set(cell_polygons.keys()) & set(int(c) for c in cell_ids_step4)
mol_mask = molecules['cell_id'].isin(valid_cell_ids)
mol = molecules[mol_mask].copy()
print(f"Molecules in valid cells: {len(mol):,} / {len(molecules):,}")

# Pre-compute nucleus centroid for each cell (use nucleus polygon centroid if available,
# otherwise fall back to cell centroid from cells.csv)
nuc_centroids = {}
for cid in valid_cell_ids:
    if cid in nuc_polygons:
        nc = nuc_polygons[cid].centroid
        nuc_centroids[cid] = np.array([nc.x, nc.y], dtype=np.float32)
    else:
        nuc_centroids[cid] = centroid_lookup[cid]

# Vectorized coordinate normalization (process in chunks for memory)
CHUNK = 500_000
n_mol = len(mol)
r_norm_all = np.zeros(n_mol, dtype=np.float32)
theta_all = np.zeros(n_mol, dtype=np.float32)
z_rel_all = np.zeros(n_mol, dtype=np.float32)
p_nuclear_all = np.zeros(n_mol, dtype=np.float32)

# Pre-compute per-cell z range
cell_z_range = mol.groupby('cell_id')['z'].agg(['min', 'max'])

for start in tqdm(range(0, n_mol, CHUNK), desc="Normalizing coordinates"):
    end = min(start + CHUNK, n_mol)
    chunk = mol.iloc[start:end]

    cids = chunk['cell_id'].values
    x_m = chunk['x'].values.astype(np.float32)
    y_m = chunk['y'].values.astype(np.float32)
    z_m = chunk['z'].values.astype(np.float32)
    overlaps_nuc = chunk['overlaps_nucleus'].values.astype(np.int8)

    for j in range(len(chunk)):
        cid = int(cids[j])
        nc = nuc_centroids[cid]

        # Radial distance from nucleus centroid
        dx = x_m[j] - nc[0]
        dy = y_m[j] - nc[1]
        d_nuc = np.sqrt(dx*dx + dy*dy)

        # Distance to cell boundary
        pt = Point(x_m[j], y_m[j])
        cell_poly = cell_polygons[cid]
        d_edge = get_min_exterior_distance(cell_poly, pt)

        # Normalized radial position
        denom = d_nuc + d_edge
        r_norm_all[start+j] = d_nuc / denom if denom > 0 else 0.0

        # Angular position
        theta_all[start+j] = np.arctan2(dy, dx)

        # Z-relative
        zr = cell_z_range.loc[cid] if cid in cell_z_range.index else None
        if zr is not None and (zr['max'] - zr['min']) > 0:
            z_rel_all[start+j] = (z_m[j] - zr['min']) / (zr['max'] - zr['min'])
        else:
            z_rel_all[start+j] = 0.5

        # Compartment probability
        if overlaps_nuc[j] == 1:
            p_nuclear_all[start+j] = 1.0
        elif cid in nuc_polygons:
            d_nuc_boundary = get_min_exterior_distance(nuc_polygons[cid], pt)
            # Sigmoid with calibrated steepness
            p_nuclear_all[start+j] = 1.0 / (1.0 + np.exp(2.0 * d_nuc_boundary))
        else:
            p_nuclear_all[start+j] = 0.0

# Assign to molecule table
mol = mol.copy()
mol['r_norm'] = r_norm_all
mol['theta'] = theta_all
mol['z_rel'] = z_rel_all
mol['p_nuclear'] = p_nuclear_all
mol['status'] = 'observed'
mol['weight'] = 1.0
mol['is_imputed'] = False

elapsed = time.time() - t0
print(f"\nCoordinate normalization complete in {elapsed/60:.1f} min")
print(f"r_norm range: [{r_norm_all.min():.3f}, {r_norm_all.max():.3f}]")
print(f"z_rel range: [{z_rel_all.min():.3f}, {z_rel_all.max():.3f}]")
print(f"p_nuclear > 0.5: {(p_nuclear_all > 0.5).sum():,} / {n_mol:,} "
      f"({100*(p_nuclear_all > 0.5).sum()/n_mol:.1f}%) ")


In [ ]:
# ==============================================================================
# CELL 7 — CHECKPOINT AFTER COORDINATE NORMALIZATION
# Saves mol and geometry objects after nuc_centroids exists.
# ==============================================================================

import os
import pickle
import gc

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("=" * 70)
print("CHECKPOINT A: Save molecule table after coordinate normalization")
print("=" * 70)

# Save processed molecule table
mol_path = os.path.join(CHECKPOINT_DIR, "checkpoint_mol_after_5A.parquet")
mol.to_parquet(mol_path, index=False)

print(
    f"Saved: {mol_path}\n"
    f"  rows: {len(mol):,}\n"
    f"  memory: {mol.memory_usage(deep=True).sum()/1e9:.2f} GB"
)

# Save geometry objects that are expensive to rebuild
objects_to_save = {
    "cell_polygons.pkl": cell_polygons,
    "nuc_polygons.pkl": nuc_polygons,
    "cell_areas.pkl": cell_areas,
    "nuc_areas.pkl": nuc_areas,
    "nuc_centroids.pkl": nuc_centroids,
}

for fname, obj in objects_to_save.items():
    fpath = os.path.join(CHECKPOINT_DIR, fname)
    with open(fpath, "wb") as f:
        pickle.dump(obj, f)
    print(f"Saved: {fpath}")

gc.collect()

print("Checkpoint A complete.")

In [ ]:
# === CELL 8: FREE MEMORY — run this before S5-5 ===

import gc
import sys

def sizeof_fmt(num):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if abs(num) < 1024:
            return f"{num:.1f} {unit}"
        num /= 1024
    return f"{num:.1f} TB"

# Show what's using memory before cleanup
print("=" * 60)
print("MEMORY CLEANUP")
print("=" * 60)

big_vars = []
for name, obj in list(globals().items()):
    if name.startswith('_'):
        continue
    try:
        size = sys.getsizeof(obj)
        big_vars.append((name, size, type(obj).__name__))
    except:
        pass

big_vars.sort(key=lambda x: -x[1])
print("Top memory consumers before cleanup:")
for name, size, typ in big_vars[:20]:
    print(f"  {name:40s} {sizeof_fmt(size):>10s}  ({typ})")

# --- Delete variables no longer needed ---

# The original unfiltered molecules DataFrame — mol already has everything we need
if 'molecules' in dir():
    del molecules
    print("\n  Deleted: molecules (original DataFrame)")

# merged_transcripts_df — was used to create molecules.parquet, no longer needed
if 'merged_transcripts_df' in dir():
    del merged_transcripts_df
    print("  Deleted: merged_transcripts_df")

# filtered_transcripts_df
if 'filtered_transcripts_df' in dir():
    del filtered_transcripts_df
    print("  Deleted: filtered_transcripts_df")

# molecules_csv_df
if 'molecules_csv_df' in dir():
    del molecules_csv_df
    print("  Deleted: molecules_csv_df")

# molecules_df
if 'molecules_df' in dir():
    del molecules_df
    print("  Deleted: molecules_df")

# transcripts_df — the raw CSV load
if 'transcripts_df' in dir():
    del transcripts_df
    print("  Deleted: transcripts_df")

# filtered_molecules_df
if 'filtered_molecules_df' in dir():
    del filtered_molecules_df
    print("  Deleted: filtered_molecules_df")

# The full scRNA-seq reference — we only need the denoised outputs now
if 'adata_ref' in dir():
    del adata_ref
    print("  Deleted: adata_ref")

if 'annotated_ref_adata' in dir():
    del annotated_ref_adata
    print("  Deleted: annotated_ref_adata")

# The raw spatial adata — we have denoised_adata which is what we need
if 'annotated_spatial_adata' in dir():
    del annotated_spatial_adata
    print("  Deleted: annotated_spatial_adata")

# cell_by_gene_adata — raw cell×gene matrix, already captured in X_raw
if 'cell_by_gene_adata' in dir():
    del cell_by_gene_adata
    print("  Deleted: cell_by_gene_adata")

# Various intermediate DataFrames from exploration cells
for varname in ['diff_exp_df', 'clusters_df', 'cells_df',
                'cell_boundaries_df', 'nucleus_boundaries_df',
                'filtered_molecules_csv_df', 'count_pivot', 'conf_pivot',
                'imp_pivot', 'xenium_annotations_df']:
    if varname in dir():
        exec(f'del {varname}')
        print(f"  Deleted: {varname}")

# The raw cell geometry arrays — we already built polygons from them
if 'cell_geom' in dir():
    del cell_geom
    print("  Deleted: cell_geom (raw npz)")

if 'cell_vertices' in dir():
    del cell_vertices
    print("  Deleted: cell_vertices")

if 'nuc_vertices' in dir():
    del nuc_vertices
    print("  Deleted: nuc_vertices")

# Force garbage collection (multiple passes)
gc.collect()
gc.collect()
gc.collect()

# Check memory after cleanup
import psutil
mem = psutil.virtual_memory()
print(f"\nSystem RAM after cleanup: {mem.used/1e9:.1f} / {mem.total/1e9:.1f} GB "
      f"({mem.percent}%)")
print(f"Available: {mem.available/1e9:.1f} GB")


In [ ]:
# ==============================================================================
# CELL 10 — CONSERVATIVE CONTAMINATION PRUNING
# Only prune where Step 4 actually corrected AND reduction >= 2 molecules.
# ==============================================================================

print("=" * 70)
print("SUBSTEP 5B: Contamination pruning — conservative corrected-count version")
print("=" * 70)

import numpy as np
import pandas as pd
import gc, time, os
from collections import defaultdict

t0 = time.time()

# ------------------------------------------------------------------
# Required variables
# ------------------------------------------------------------------
required_vars = [
    "mol",
    "X_raw_counts",
    "X_denoised",
    "was_corrected",
    "cell_ids_step4",
    "shared_genes",
    "valid_cell_ids",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Required variable missing: {v}")

if was_corrected.shape != X_denoised.shape:
    raise ValueError(
        f"was_corrected shape {was_corrected.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

if X_raw_counts.shape != X_denoised.shape:
    raise ValueError(
        f"X_raw_counts shape {X_raw_counts.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

# Make sure status/weight columns exist
if "status" not in mol.columns:
    mol["status"] = "observed"

if "weight" not in mol.columns:
    mol["weight"] = 1.0

if "is_imputed" not in mol.columns:
    mol["is_imputed"] = False

# Convert status to string to avoid categorical assignment issues
mol["status"] = mol["status"].astype(str)

# Reset previous pruning if this cell is re-run
mol.loc[mol["status"] == "pruned", "status"] = "observed"
mol.loc[mol["weight"] == 0.0, "weight"] = 1.0

# ------------------------------------------------------------------
# Build conservative overcounted-pair list
# ------------------------------------------------------------------
step4_cell_to_row = {int(c): i for i, c in enumerate(cell_ids_step4)}
gene_to_col = {g: j for j, g in enumerate(shared_genes)}

raw_int = X_raw_counts.astype(np.int32)
den_round = np.rint(X_denoised).astype(np.int32)

reduction = raw_int - den_round

# Required condition:
#   Step 4 actually corrected this cell-gene pair
#   and denoising reduced the count by at least 2 molecules
candidate_mask = (was_corrected.astype(bool)) & (reduction >= 2)

# Optional extra conservativeness:
# avoid pruning from tiny raw counts.
# Set MIN_RAW_FOR_PRUNE = 0 if you want exactly only the two conditions above.
MIN_RAW_FOR_PRUNE = 4
candidate_mask &= raw_int >= MIN_RAW_FOR_PRUNE

# Only consider cells with geometry and molecules
valid_row_mask = np.array(
    [int(cid) in valid_cell_ids for cid in cell_ids_step4],
    dtype=bool
)
candidate_mask &= valid_row_mask[:, None]

candidate_positions = np.argwhere(candidate_mask)

overcounted_pairs = {
    (int(cell_ids_step4[i]), shared_genes[j]): int(reduction[i, j])
    for i, j in candidate_positions
}

print(f"Overcounted pairs eligible for pruning: {len(overcounted_pairs):,}")
print(f"Conditions:")
print(f"  was_corrected == True")
print(f"  X_raw_counts - round(X_denoised) >= 2")
print(f"  X_raw_counts >= {MIN_RAW_FOR_PRUNE}")
print(f"  cell has valid geometry")

if len(overcounted_pairs) == 0:
    print("No pruning needed.")
    n_pruned_total = 0

else:
    # ------------------------------------------------------------------
    # Find candidate molecules only from affected cell-gene pairs
    # ------------------------------------------------------------------
    oc_cells = set(cid for cid, gid in overcounted_pairs.keys())
    oc_genes = set(gid for cid, gid in overcounted_pairs.keys())

    mask_candidates = (
        mol["cell_id"].isin(oc_cells)
        & mol["gene_id"].isin(oc_genes)
        & (mol["status"] == "observed")
    )

    candidate_idx = mol.index[mask_candidates]

    print(f"Candidate molecule rows from affected cells/genes: {len(candidate_idx):,}")

    # Pull arrays once for speed
    cids_arr = mol.loc[candidate_idx, "cell_id"].values
    gids_arr = mol.loc[candidate_idx, "gene_id"].astype(str).values
    rnorm_arr = mol.loc[candidate_idx, "r_norm"].values.astype(np.float32)
    qual_arr = mol.loc[candidate_idx, "quality"].values.astype(np.float32)

    pair_to_molecules = defaultdict(list)

    for k in range(len(candidate_idx)):
        key = (int(cids_arr[k]), gids_arr[k])
        if key in overcounted_pairs:
            pair_to_molecules[key].append((k, candidate_idx[k]))

    # ------------------------------------------------------------------
    # Choose specific molecules to prune
    # Suspicion score:
    #   higher r_norm = closer to cell boundary
    #   lower QV = less confident detection
    # ------------------------------------------------------------------
    prune_indices = []

    for key, n_to_prune in overcounted_pairs.items():
        mol_list = pair_to_molecules.get(key, [])

        if not mol_list:
            continue

        local_k = np.array([m[0] for m in mol_list], dtype=int)
        local_idx = np.array([m[1] for m in mol_list])

        quality_scaled = np.clip(qual_arr[local_k] / 40.0, 0.0, 1.0)

        suspicion = (
            0.6 * rnorm_arr[local_k]
            + 0.4 * (1.0 - quality_scaled)
        )

        order = np.argsort(-suspicion)
        n_actual = min(int(n_to_prune), len(order))

        prune_indices.extend(local_idx[order[:n_actual]].tolist())

    if prune_indices:
        mol.loc[prune_indices, "status"] = "pruned"
        mol.loc[prune_indices, "weight"] = 0.0

    n_pruned_total = len(prune_indices)

    del candidate_idx, cids_arr, gids_arr, rnorm_arr, qual_arr, pair_to_molecules
    gc.collect()

pct_pruned = 100 * n_pruned_total / len(mol)

print(f"\nPruned: {n_pruned_total:,} molecules ({pct_pruned:.4f}%)")
print(f"Remaining observed: {(mol['status'] == 'observed').sum():,}")
print(f"Pruned status count: {(mol['status'] == 'pruned').sum():,}")
print(f"Completed in {time.time() - t0:.1f}s")

if pct_pruned > 10:
    print(f"WARNING: {pct_pruned:.2f}% pruned is high. Check Step 4 correction settings.")
elif pct_pruned < 5:
    print(f"Pruning rate {pct_pruned:.2f}% is conservative/healthy.")

In [ ]:
# ==============================================================================
# CELL 11 — SAVE POST-PRUNING MOLECULE TABLE FOR RECOVERY
# ==============================================================================

import os

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

post_prune_path = os.path.join(CHECKPOINT_DIR, "step5_mol_processed.parquet")

mol.to_parquet(post_prune_path, index=False)

print(f"Saved post-pruning molecule table:")
print(f"  {post_prune_path}")
print(f"  rows: {len(mol):,}")
print(f"  observed: {(mol['status'] == 'observed').sum():,}")
print(f"  pruned: {(mol['status'] == 'pruned').sum():,}")
print(f"  imputed: {(mol['status'] == 'imputed').sum() if 'imputed' in set(mol['status']) else 0:,}")

# session crash1


In [ ]:
# ==============================================================================
# RECOVERY CELL — LOAD STEP 5 STATE AFTER RUNTIME DISCONNECT
# Use this if you already ran through:
# CELL 11 — SAVE POST-PRUNING MOLECULE TABLE FOR RECOVERY
# ==============================================================================

import os
import gc
import json
import pickle
import warnings
import numpy as np
import pandas as pd
from scipy import sparse
import anndata as ad

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

print("=" * 80)
print("STEP 5 RECOVERY: Loading post-pruning checkpoint and required Step 4 state")
print("=" * 80)

# ------------------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------------------

STEP4_EXPORT_DIR = "/content/drive/MyDrive/diffusion/step4_exports"
CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"

POST_PRUNE_MOL_PATH = os.path.join(CHECKPOINT_DIR, "step5_mol_processed.parquet")
MOL_AFTER_5A_PATH = os.path.join(CHECKPOINT_DIR, "checkpoint_mol_after_5A.parquet")

print(f"STEP4_EXPORT_DIR: {STEP4_EXPORT_DIR}")
print(f"CHECKPOINT_DIR  : {CHECKPOINT_DIR}")

# ------------------------------------------------------------------------------
# Helper
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

# ------------------------------------------------------------------------------
# Check required files
# ------------------------------------------------------------------------------

required_files = {
    "Step 4 config": os.path.join(STEP4_EXPORT_DIR, "step4_config.json"),
    "Step 4 denoised AnnData": os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad"),
    "Step 4 was_corrected": os.path.join(STEP4_EXPORT_DIR, "was_corrected.npy"),
    "Step 4 molecules": os.path.join(STEP4_EXPORT_DIR, "molecules.parquet"),
    "Step 4 cell geometry": os.path.join(STEP4_EXPORT_DIR, "cell_data.npz"),
}

# Prefer Cell 11 checkpoint. If unavailable, fall back to Cell 7 checkpoint.
if os.path.exists(POST_PRUNE_MOL_PATH):
    mol_checkpoint_path = POST_PRUNE_MOL_PATH
    print(f"\nUsing Cell 11 post-pruning checkpoint:")
    print(f"  {mol_checkpoint_path}")
elif os.path.exists(MOL_AFTER_5A_PATH):
    mol_checkpoint_path = MOL_AFTER_5A_PATH
    print(f"\nWARNING: Cell 11 checkpoint not found.")
    print(f"Falling back to Cell 7 coordinate-normalization checkpoint:")
    print(f"  {mol_checkpoint_path}")
else:
    raise FileNotFoundError(
        "Neither step5_mol_processed.parquet nor checkpoint_mol_after_5A.parquet was found. "
        "You need to rerun Cells 1–7 or restore the checkpoint files."
    )

required_files["Step 5 molecule checkpoint"] = mol_checkpoint_path

missing = []
print("\nChecking required files:")
for name, path in required_files.items():
    if os.path.exists(path):
        print(f"  ✓ {name:30s}: {path}")
    else:
        print(f"  ✗ MISSING {name:30s}: {path}")
        missing.append((name, path))

if missing:
    raise FileNotFoundError(f"Missing required recovery files: {missing}")

# ------------------------------------------------------------------------------
# Load Step 4 config
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("1. Loading Step 4 config")
print("=" * 80)

with open(os.path.join(STEP4_EXPORT_DIR, "step4_config.json"), "r") as f:
    step4_config = json.load(f)

shared_genes = list(step4_config["shared_genes"])
ct_column = step4_config.get("cell_type_column", "cell_type")
unique_cell_types = sorted([str(x) for x in step4_config.get("cell_types", [])])

print(f"Shared genes: {len(shared_genes):,}")
print(f"Cell-type column: {ct_column}")
print(f"Cell types: {len(unique_cell_types):,}")

# ------------------------------------------------------------------------------
# Load denoised AnnData, X_denoised, uncertainty, cell types
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. Loading denoised_adata and Step 4 matrices")
print("=" * 80)

denoised_adata = ad.read_h5ad(os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad"))

print(f"denoised_adata shape: {denoised_adata.shape}")
print(f"obs columns: {list(denoised_adata.obs.columns)}")
print(f"layers: {list(denoised_adata.layers.keys())}")

if ct_column not in denoised_adata.obs.columns:
    if "cell_type" in denoised_adata.obs.columns:
        ct_column = "cell_type"
    elif "Assigned_Xenium_Cell_Type" in denoised_adata.obs.columns:
        ct_column = "Assigned_Xenium_Cell_Type"
    else:
        raise KeyError("No valid cell-type column found in denoised_adata.obs.")

cell_types_series = denoised_adata.obs[ct_column].astype(str)

if not unique_cell_types:
    unique_cell_types = sorted(cell_types_series.unique().tolist())

ct_to_idx = {ct: i for i, ct in enumerate(unique_cell_types)}
cell_type_indices = cell_types_series.map(ct_to_idx).values.astype(np.int64)

if np.isnan(cell_type_indices).any():
    raise ValueError("Some cell types could not be mapped to integer indices.")

X_denoised = ensure_dense(denoised_adata.X).astype(np.float32)

if "uncertainty" in denoised_adata.layers:
    uncertainty = ensure_dense(denoised_adata.layers["uncertainty"]).astype(np.float32)
else:
    uncertainty = None
    print("WARNING: uncertainty layer not found.")

cell_ids_step4 = np.array(denoised_adata.obs_names, dtype=str)

print(f"X_denoised shape: {X_denoised.shape}")
print(f"X_denoised sum  : {X_denoised.sum(dtype=np.float64):,.2f}")
if uncertainty is not None:
    print(f"uncertainty shape: {uncertainty.shape}")

# ------------------------------------------------------------------------------
# Load was_corrected
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. Loading was_corrected mask")
print("=" * 80)

was_corrected = np.load(os.path.join(STEP4_EXPORT_DIR, "was_corrected.npy")).astype(bool)

if was_corrected.shape != X_denoised.shape:
    raise ValueError(
        f"was_corrected shape {was_corrected.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

print(f"was_corrected shape: {was_corrected.shape}")
print(f"Corrected pairs: {was_corrected.sum():,}")
print(f"Correction rate: {100 * was_corrected.sum() / was_corrected.size:.2f}%")

# ------------------------------------------------------------------------------
# Load recovered molecule table from Cell 11
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. Loading recovered Step 5 molecule table")
print("=" * 80)

mol = pd.read_parquet(mol_checkpoint_path)

print(f"mol loaded: {len(mol):,} rows")
print(f"mol columns: {list(mol.columns)}")

required_mol_cols = [
    "transcript_id",
    "cell_id",
    "gene_id",
    "x",
    "y",
    "z",
    "quality",
    "overlaps_nucleus",
    "Assigned_Xenium_Cell_Type",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "status",
    "weight",
    "is_imputed",
]

missing_cols = [c for c in required_mol_cols if c not in mol.columns]
if missing_cols:
    raise KeyError(f"Recovered mol table is missing required columns: {missing_cols}")

# Make sure status/weight/imputed columns have safe types
mol["status"] = mol["status"].astype(str)
mol["weight"] = mol["weight"].astype(np.float32)
mol["is_imputed"] = mol["is_imputed"].astype(bool)

print("\nRecovered molecule status counts:")
display(mol["status"].value_counts().rename_axis("status").reset_index(name="n_molecules"))

print("\nRecovered coordinate-feature checks:")
print(f"  r_norm range   : [{mol['r_norm'].min():.4f}, {mol['r_norm'].max():.4f}]")
print(f"  theta range    : [{mol['theta'].min():.4f}, {mol['theta'].max():.4f}]")
print(f"  z_rel range    : [{mol['z_rel'].min():.4f}, {mol['z_rel'].max():.4f}]")
print(f"  p_nuclear range: [{mol['p_nuclear'].min():.4f}, {mol['p_nuclear'].max():.4f}]")

# For compatibility with older later cells
molecules = mol

# ------------------------------------------------------------------------------
# Rebuild X_raw_counts from the original clean Step 4 molecule table
# This preserves the original raw observed baseline.
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. Rebuilding X_raw_counts from clean Step 4 molecules")
print("=" * 80)

step4_molecules = pd.read_parquet(os.path.join(STEP4_EXPORT_DIR, "molecules.parquet"))

gene_to_col = {g: j for j, g in enumerate(shared_genes)}
gene_idx_map = gene_to_col

step4_cell_to_row = {int(c): i for i, c in enumerate(cell_ids_step4)}
cell_idx_map = step4_cell_to_row

count_source = step4_molecules[
    step4_molecules["cell_id"].isin(step4_cell_to_row.keys())
    & step4_molecules["gene_id"].isin(gene_to_col.keys())
].copy()

counts = (
    count_source
    .groupby(["cell_id", "gene_id"])
    .size()
    .reset_index(name="count")
)

X_raw_counts = np.zeros(X_denoised.shape, dtype=np.int32)

rows = counts["cell_id"].map(step4_cell_to_row)
cols = counts["gene_id"].map(gene_to_col)
ok = rows.notna() & cols.notna()

X_raw_counts[
    rows[ok].astype(int).to_numpy(),
    cols[ok].astype(int).to_numpy()
] = counts.loc[ok, "count"].to_numpy(dtype=np.int32)

X_raw = X_raw_counts.astype(np.float32)
X_observed_counts = X_raw_counts
X_raw_for_step5 = X_raw_counts

# Update AnnData raw layers in memory to avoid old gene-order mismatch
denoised_adata.layers["raw"] = X_raw.copy()
denoised_adata.layers["raw_molecule_counts_clean"] = X_raw.copy()

print(f"X_raw_counts shape: {X_raw_counts.shape}")
print(f"X_raw_counts sum  : {X_raw_counts.sum(dtype=np.float64):,.0f}")

# Optional status-aware current observed count matrix after pruning
# In your current run pruning should be zero, so this should match X_raw_counts.
observed_mol = mol[mol["status"] == "observed"].copy()

counts_current = (
    observed_mol
    .groupby(["cell_id", "gene_id"])
    .size()
    .reset_index(name="count")
)

X_current_observed_counts = np.zeros(X_denoised.shape, dtype=np.int32)

rows2 = counts_current["cell_id"].map(step4_cell_to_row)
cols2 = counts_current["gene_id"].map(gene_to_col)
ok2 = rows2.notna() & cols2.notna()

X_current_observed_counts[
    rows2[ok2].astype(int).to_numpy(),
    cols2[ok2].astype(int).to_numpy()
] = counts_current.loc[ok2, "count"].to_numpy(dtype=np.int32)

current_diff = np.abs(
    X_current_observed_counts.astype(np.float64) -
    X_raw_counts.astype(np.float64)
)

print("\nStatus-aware observed-count check:")
print(f"  current observed count sum: {X_current_observed_counts.sum(dtype=np.float64):,.0f}")
print(f"  original raw count sum    : {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"  max abs diff              : {current_diff.max():.6f}")
print(f"  mismatched pairs          : {(current_diff > 1e-6).sum():,}")

del step4_molecules, count_source, counts, observed_mol, counts_current
gc.collect()

# ------------------------------------------------------------------------------
# Load geometry objects saved after Cell 6 / Cell 7
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("6. Loading saved geometry objects")
print("=" * 80)

geometry_pkl_files = {
    "cell_polygons": os.path.join(CHECKPOINT_DIR, "cell_polygons.pkl"),
    "nuc_polygons": os.path.join(CHECKPOINT_DIR, "nuc_polygons.pkl"),
    "cell_areas": os.path.join(CHECKPOINT_DIR, "cell_areas.pkl"),
    "nuc_areas": os.path.join(CHECKPOINT_DIR, "nuc_areas.pkl"),
    "nuc_centroids": os.path.join(CHECKPOINT_DIR, "nuc_centroids.pkl"),
}

missing_geom = [name for name, path in geometry_pkl_files.items() if not os.path.exists(path)]

if missing_geom:
    print(f"WARNING: Missing geometry pickle files: {missing_geom}")
    print("Later cells that need polygons may fail unless you rerun Cell 4 and Cell 6.")
else:
    with open(geometry_pkl_files["cell_polygons"], "rb") as f:
        cell_polygons = pickle.load(f)

    with open(geometry_pkl_files["nuc_polygons"], "rb") as f:
        nuc_polygons = pickle.load(f)

    with open(geometry_pkl_files["cell_areas"], "rb") as f:
        cell_areas = pickle.load(f)

    with open(geometry_pkl_files["nuc_areas"], "rb") as f:
        nuc_areas = pickle.load(f)

    with open(geometry_pkl_files["nuc_centroids"], "rb") as f:
        nuc_centroids = pickle.load(f)

    print(f"cell_polygons: {len(cell_polygons):,}")
    print(f"nuc_polygons : {len(nuc_polygons):,}")
    print(f"cell_areas   : {len(cell_areas):,}")
    print(f"nuc_areas    : {len(nuc_areas):,}")
    print(f"nuc_centroids: {len(nuc_centroids):,}")

# ------------------------------------------------------------------------------
# Load compact geometry arrays from cell_data.npz too
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("7. Loading compact cell_data.npz arrays")
print("=" * 80)

cell_geom = np.load(os.path.join(STEP4_EXPORT_DIR, "cell_data.npz"), allow_pickle=False)

cell_ids_geom = cell_geom["cell_ids"]
centroids = cell_geom["centroids"]
cell_offsets = cell_geom["cell_offsets"]
cell_vertices = cell_geom["cell_vertices"]
nuc_offsets = cell_geom["nuc_offsets"]
nuc_vertices = cell_geom["nuc_vertices"]
nuc_present = cell_geom["nuc_present"]

del cell_geom
gc.collect()

print(f"cell_ids_geom shape: {cell_ids_geom.shape}")
print(f"centroids shape    : {centroids.shape}")
print(f"cell_vertices shape: {cell_vertices.shape}")
print(f"nuc_vertices shape : {nuc_vertices.shape}")

# ------------------------------------------------------------------------------
# Recreate valid_cell_ids
# ------------------------------------------------------------------------------

if "cell_polygons" in globals():
    valid_cell_ids = set(cell_polygons.keys()) & set(int(c) for c in cell_ids_step4)
else:
    valid_cell_ids = set(mol["cell_id"].unique()) & set(int(c) for c in cell_ids_step4)

print(f"\nvalid_cell_ids: {len(valid_cell_ids):,}")

# ------------------------------------------------------------------------------
# Step 5 target diagnostics
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("8. Step 5 target diagnostics after recovery")
print("=" * 80)

den_round = np.rint(X_denoised).astype(np.int32)

need_impute_mask = den_round > X_raw_counts
need_prune_mask = den_round < X_raw_counts

n_need_impute = int(need_impute_mask.sum())
n_need_prune = int(need_prune_mask.sum())

n_need_impute_corrected = int((need_impute_mask & was_corrected).sum())
n_need_prune_corrected = int((need_prune_mask & was_corrected).sum())

print(f"Pairs where round(X_denoised) > X_raw_counts: {n_need_impute:,}")
print(f"Pairs where round(X_denoised) < X_raw_counts: {n_need_prune:,}")
print(f"Corrected pairs needing imputation: {n_need_impute_corrected:,}")
print(f"Corrected pairs needing pruning: {n_need_prune_corrected:,}")

# ------------------------------------------------------------------------------
# Final sanity summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("RECOVERY COMPLETE")
print("=" * 80)

print("Recovered variables ready for later Step 5 cells:")
print("  mol")
print("  molecules")
print("  denoised_adata")
print("  X_denoised")
print("  X_raw_counts / X_observed_counts / X_raw_for_step5")
print("  X_current_observed_counts")
print("  was_corrected")
print("  uncertainty")
print("  shared_genes")
print("  cell_ids_step4")
print("  gene_to_col / gene_idx_map")
print("  step4_cell_to_row / cell_idx_map")
print("  cell_type_indices")
print("  valid_cell_ids")
print("  cell_ids_geom, centroids, cell_offsets, cell_vertices")
print("  nuc_offsets, nuc_vertices, nuc_present")
if "cell_polygons" in globals():
    print("  cell_polygons, nuc_polygons, cell_areas, nuc_areas, nuc_centroids")

print("\nYou can now continue with later Step 5 cells for building/training data.")

In [ ]:
# ==============================================================================
# CELL S5-6-DIST — Build training data + empirical baseline pools
# Replaces old PC2PC-based training-data cell.
# ==============================================================================

import os
import gc
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

print("=" * 80)
print("SUBSTEP 5C: Building training data + empirical baselines")
print("=" * 80)
t0 = time.time()

# ------------------------------------------------------------------------------
# Required variables from recovery cell
# ------------------------------------------------------------------------------

required_vars = [
    "mol",
    "X_raw_counts",
    "X_denoised",
    "was_corrected",
    "shared_genes",
    "cell_ids_step4",
    "gene_to_col",
    "step4_cell_to_row",
    "cell_type_indices",
    "unique_cell_types",
    "denoised_adata",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Required variable missing: {v}")

# Geometry dictionaries are needed for area features.
if "cell_areas" not in globals():
    print("WARNING: cell_areas missing; using default cell area.")
    cell_areas = {}

if "nuc_areas" not in globals():
    print("WARNING: nuc_areas missing; using default nucleus area.")
    nuc_areas = {}

# ------------------------------------------------------------------------------
# Use only active observed molecules
# ------------------------------------------------------------------------------

mol["status"] = mol["status"].astype(str)
mol_clean = mol[mol["status"] == "observed"].copy()

print(f"Observed molecules for training/baselines: {len(mol_clean):,}")

required_cols = [
    "cell_id",
    "gene_id",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "Assigned_Xenium_Cell_Type",
]

missing_cols = [c for c in required_cols if c not in mol_clean.columns]
if missing_cols:
    raise KeyError(f"mol is missing required columns: {missing_cols}")

# ------------------------------------------------------------------------------
# Build cell centroid lookup for Spatial-kNN baseline
# ------------------------------------------------------------------------------

cell_xy_lookup = {}

if {"x_centroid", "y_centroid"}.issubset(set(denoised_adata.obs.columns)):
    print("Using denoised_adata.obs x_centroid/y_centroid for cell coordinates.")
    for cid, x, y in zip(
        denoised_adata.obs_names.astype(str),
        denoised_adata.obs["x_centroid"].values,
        denoised_adata.obs["y_centroid"].values,
    ):
        cell_xy_lookup[int(cid)] = np.array([float(x), float(y)], dtype=np.float32)

elif "cell_ids_geom" in globals() and "centroids" in globals():
    print("Using cell_ids_geom/centroids for cell coordinates.")
    for cid, xy in zip(cell_ids_geom, centroids):
        cell_xy_lookup[int(cid)] = np.asarray(xy, dtype=np.float32)

else:
    raise NameError(
        "No cell coordinate source found. Need denoised_adata.obs x/y centroids "
        "or cell_ids_geom + centroids from recovery cell."
    )

print(f"Cell coordinate lookup entries: {len(cell_xy_lookup):,}")

# ------------------------------------------------------------------------------
# Build one training example per eligible (cell, gene) pair
# ------------------------------------------------------------------------------

MIN_MOL_FOR_TRAINING = 6

print("\nBuilding examples from observed molecules...")
examples_by_pair = []

for (cid, gid), group in mol_clean.groupby(["cell_id", "gene_id"], sort=False):
    n = len(group)

    if n < MIN_MOL_FOR_TRAINING:
        continue

    cid_int = int(cid)

    if cid_int not in step4_cell_to_row:
        continue

    if gid not in gene_to_col:
        continue

    if cid_int not in cell_xy_lookup:
        continue

    row = step4_cell_to_row[cid_int]
    gene_idx = gene_to_col[gid]
    ct_idx = int(cell_type_indices[row])

    c_area = float(cell_areas.get(cid_int, 100.0))
    n_area = float(nuc_areas.get(cid_int, 30.0))

    values = group[["r_norm", "theta", "z_rel", "p_nuclear"]].values.astype(np.float32)

    examples_by_pair.append({
        "cell_id": cid_int,
        "gene_id": str(gid),
        "gene_idx": gene_idx,
        "ct_idx": ct_idx,
        "cell_xy": cell_xy_lookup[cid_int],
        "cell_area": c_area,
        "nuc_area": n_area,
        "values": values,
    })

print(f"Eligible observed (cell, gene) pairs: {len(examples_by_pair):,}")

if len(examples_by_pair) == 0:
    raise RuntimeError("No eligible training examples found. Check mol/status/features.")

# ------------------------------------------------------------------------------
# Cell-level train/validation split
# Important: baselines are built only from training cells to avoid validation leakage.
# ------------------------------------------------------------------------------

P_DROP_MIN = 0.15
P_DROP_MAX = 0.40

all_cells = sorted({ex["cell_id"] for ex in examples_by_pair})
rng_split = np.random.default_rng(42)
rng_split.shuffle(all_cells)

n_val = max(1, int(0.10 * len(all_cells)))
val_cells = set(all_cells[:n_val])
train_cells = set(all_cells[n_val:])

training_examples = [ex for ex in examples_by_pair if ex["cell_id"] in train_cells]
validation_examples = [ex for ex in examples_by_pair if ex["cell_id"] in val_cells]

print(f"Train cells: {len(train_cells):,}")
print(f"Validation cells: {len(val_cells):,}")
print(f"Training examples: {len(training_examples):,}")
print(f"Validation examples: {len(validation_examples):,}")
print(f"Dynamic molecule dropout: {P_DROP_MIN*100:.0f}%–{P_DROP_MAX*100:.0f}%")

# ------------------------------------------------------------------------------
# Build empirical baseline pools from training examples only
# ------------------------------------------------------------------------------

print("\nBuilding empirical baseline pools from TRAINING examples only...")

gene_pool_lists = defaultdict(list)
ct_gene_pool_lists = defaultdict(list)

# For spatial-kNN:
# key = (ct_idx, gene_idx)
# value = list of entries with cell_id, xy, and molecule values
spatial_entry_lists = defaultdict(list)

for ex in training_examples:
    gene_idx = int(ex["gene_idx"])
    ct_idx = int(ex["ct_idx"])
    values = ex["values"].astype(np.float32)

    gene_pool_lists[gene_idx].append(values)
    ct_gene_pool_lists[(ct_idx, gene_idx)].append(values)

    spatial_entry_lists[(ct_idx, gene_idx)].append({
        "cell_id": int(ex["cell_id"]),
        "xy": ex["cell_xy"].astype(np.float32),
        "values": values,
    })

baseline_gene_pools = {}
for gene_idx, arrs in gene_pool_lists.items():
    baseline_gene_pools[gene_idx] = np.concatenate(arrs, axis=0).astype(np.float32)

baseline_ct_gene_pools = {}
for key, arrs in ct_gene_pool_lists.items():
    baseline_ct_gene_pools[key] = np.concatenate(arrs, axis=0).astype(np.float32)

baseline_spatial_index = {}
for key, entries in spatial_entry_lists.items():
    cell_ids_arr = np.array([e["cell_id"] for e in entries], dtype=np.int64)
    xy_arr = np.stack([e["xy"] for e in entries], axis=0).astype(np.float32)
    values_list = [e["values"].astype(np.float32) for e in entries]

    baseline_spatial_index[key] = {
        "cell_ids": cell_ids_arr,
        "xy": xy_arr,
        "values": values_list,
    }

print(f"Gene-level empirical pools: {len(baseline_gene_pools):,}")
print(f"Cell-type gene empirical pools: {len(baseline_ct_gene_pools):,}")
print(f"Spatial-kNN empirical index groups: {len(baseline_spatial_index):,}")

# A few diagnostics
pool_size_summary = pd.Series(
    {shared_genes[g]: len(v) for g, v in baseline_gene_pools.items()}
).sort_values(ascending=False)

print("\nTop 10 gene-level baseline pool sizes:")
display(pool_size_summary.head(10).rename("n_molecules").reset_index().rename(columns={"index": "gene"}))

# ------------------------------------------------------------------------------
# Dataset
# ------------------------------------------------------------------------------

class LocalizationDistributionDataset(Dataset):
    """
    Dynamically hides observed molecules and asks the model to recover their
    within-cell localization distribution.

    Baselines are NOT generated here. They are generated during validation from
    training-only empirical pools.
    """

    def __init__(
        self,
        examples,
        max_context=60,
        max_target=20,
        p_drop_min=0.15,
        p_drop_max=0.40,
        training=True,
        seed=42,
    ):
        self.examples = examples
        self.max_context = max_context
        self.max_target = max_target
        self.p_drop_min = p_drop_min
        self.p_drop_max = p_drop_max
        self.training = training
        self.seed = seed

    def __len__(self):
        return len(self.examples)

    def _rng(self, idx):
        if self.training:
            return np.random.default_rng()
        return np.random.default_rng(self.seed + idx)

    @staticmethod
    def _features(vals):
        r = vals[:, 0]
        theta = vals[:, 1]
        z = vals[:, 2]
        p_nuc = vals[:, 3]

        # Use sin/cos so theta wraps correctly around -pi/pi.
        return np.stack(
            [r, np.sin(theta), np.cos(theta), z, p_nuc],
            axis=-1
        ).astype(np.float32)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        vals = ex["values"]
        n = len(vals)
        rng = self._rng(idx)

        p_drop = rng.uniform(self.p_drop_min, self.p_drop_max)
        k = int(rng.binomial(n, p_drop))
        k = max(1, min(n - 3, k, self.max_target))

        perm = rng.permutation(n)
        target_vals = vals[perm[:k]]
        context_vals = vals[perm[k:]]

        if len(context_vals) > self.max_context:
            ctx_idx = rng.choice(len(context_vals), size=self.max_context, replace=False)
            context_used = context_vals[ctx_idx]
        else:
            context_used = context_vals

        n_ctx = min(len(context_used), self.max_context)
        n_tgt = min(len(target_vals), self.max_target)

        ctx_padded = np.zeros((self.max_context, 5), dtype=np.float32)
        ctx_mask = np.zeros(self.max_context, dtype=np.float32)

        if n_ctx > 0:
            ctx_padded[:n_ctx] = self._features(context_used[:n_ctx])
            ctx_mask[:n_ctx] = 1.0

        target_coords = np.zeros((self.max_target, 3), dtype=np.float32)
        target_pnuc = np.zeros(self.max_target, dtype=np.float32)
        target_mask = np.zeros(self.max_target, dtype=np.float32)

        target_coords[:n_tgt] = target_vals[:n_tgt, :3]
        target_pnuc[:n_tgt] = target_vals[:n_tgt, 3]
        target_mask[:n_tgt] = 1.0

        c_area = float(ex["cell_area"])
        n_area = float(ex["nuc_area"])
        c_area = float(ex["cell_area"])
        n_area = float(ex["nuc_area"])

        # Context summary features from visible/context molecules.
        # Feature order in ctx_features:
        #   0 = r_norm
        #   1 = sin(theta)
        #   2 = cos(theta)
        #   3 = z_rel
        #   4 = p_nuclear

        if n_ctx > 0:
            ctx_features = ctx_padded[:n_ctx]

            ctx_r_mean = float(ctx_features[:, 0].mean())
            ctx_r_std = float(ctx_features[:, 0].std())

            ctx_z_mean = float(ctx_features[:, 3].mean())
            ctx_z_std = float(ctx_features[:, 3].std())

            ctx_pnuc_mean = float(ctx_features[:, 4].mean())
        else:
            ctx_r_mean = 0.5
            ctx_r_std = 0.0
            ctx_z_mean = 0.5
            ctx_z_std = 0.0
            ctx_pnuc_mean = 0.0

        geom = np.array([
            c_area / 500.0,
            n_area / 200.0,
            n_area / c_area if c_area > 0 else 0.3,
            n_ctx / 50.0,
            ctx_r_mean,
            ctx_r_std,
            ctx_z_mean,
            ctx_z_std,
            ctx_pnuc_mean,
        ], dtype=np.float32)

        return {
            "context": torch.tensor(ctx_padded),
            "context_mask": torch.tensor(ctx_mask),
            "target_coords": torch.tensor(target_coords),
            "target_pnuc": torch.tensor(target_pnuc),
            "target_mask": torch.tensor(target_mask),
            "gene_idx": torch.tensor(ex["gene_idx"], dtype=torch.long),
            "ct_idx": torch.tensor(ex["ct_idx"], dtype=torch.long),
            "cell_id": torch.tensor(ex["cell_id"], dtype=torch.long),
            "cell_xy": torch.tensor(ex["cell_xy"], dtype=torch.float32),
            "geom": torch.tensor(geom),
        }

train_dataset = LocalizationDistributionDataset(
    training_examples,
    max_context=96,
    max_target=24,
    p_drop_min=P_DROP_MIN,
    p_drop_max=P_DROP_MAX,
    training=True,
)

val_dataset = LocalizationDistributionDataset(
    validation_examples,
    max_context=96,
    max_target=24,
    p_drop_min=P_DROP_MIN,
    p_drop_max=P_DROP_MAX,
    training=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"\nTrain batches: {len(train_loader):,}")
print(f"Val batches: {len(val_loader):,}")
print(f"Cell S5-6-DIST completed in {(time.time() - t0) / 60:.1f} min")

# Clean temporary large objects
del examples_by_pair
del gene_pool_lists, ct_gene_pool_lists, spatial_entry_lists
if "mol_clean" in globals():
    del mol_clean

gc.collect()

In [ ]:
# === CELL S5-7-DIST: Conditional localization distribution model ===

import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 60)
print("SUBSTEP 5D: Conditional Localization Distribution Model")
print("=" * 60)

class PointNetContextEncoder(nn.Module):
    def __init__(self, in_dim=5, hidden=192, out_dim=192):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, out_dim),
        )
        self.out_proj = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim),
            nn.SiLU(),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, points, mask):
        h = self.mlp(points)
        mask_exp = mask.unsqueeze(-1)
        h_masked = h.masked_fill(mask_exp == 0, -1e4)
        max_pool = h_masked.max(dim=1).values
        max_pool = torch.where(torch.isfinite(max_pool), max_pool, torch.zeros_like(max_pool))

        denom = mask_exp.sum(dim=1).clamp_min(1.0)
        mean_pool = (h * mask_exp).sum(dim=1) / denom

        return self.out_proj(torch.cat([max_pool, mean_pool], dim=-1))

class LocalizationDistributionModel(nn.Module):
    def __init__(self, n_genes, n_cell_types, ctx_dim=192, gene_dim=96,
                 ct_dim=32, geom_dim=9, geom_hidden=64, hidden=384):
        super().__init__()
        self.context_encoder = PointNetContextEncoder(in_dim=5, hidden=192, out_dim=ctx_dim)
        self.gene_embed = nn.Embedding(n_genes, gene_dim)
        self.ct_embed = nn.Embedding(n_cell_types, ct_dim)
        self.geom_mlp = nn.Sequential(
            nn.Linear(geom_dim, geom_hidden), nn.SiLU(),
            nn.Linear(geom_hidden, geom_hidden), nn.SiLU(),
        )
        fused = ctx_dim + gene_dim + ct_dim + geom_hidden
        self.backbone = nn.Sequential(
            nn.Linear(fused, hidden), nn.SiLU(), nn.Dropout(0.08),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Dropout(0.08),
            nn.Linear(hidden, hidden), nn.SiLU(),
        )
        self.r_head = nn.Linear(hidden, 2)      # alpha, beta for r_norm
        self.z_head = nn.Linear(hidden, 2)      # alpha, beta for z_rel
        self.pnuc_head = nn.Linear(hidden, 1)   # logit p_nuclear
        self.theta_head = nn.Linear(hidden, 2)  # optional von-Mises-like mean direction proxy

    def forward(self, context, context_mask, gene_idx, ct_idx, geom):
        h_ctx = self.context_encoder(context, context_mask)
        h_gene = self.gene_embed(gene_idx)
        h_ct = self.ct_embed(ct_idx)
        h_geom = self.geom_mlp(geom)
        h = self.backbone(torch.cat([h_ctx, h_gene, h_ct, h_geom], dim=-1))

        r_ab = F.softplus(self.r_head(h)) + 1.05
        z_ab = F.softplus(self.z_head(h)) + 1.05
        pnuc_logit = self.pnuc_head(h).squeeze(-1)
        theta_vec = self.theta_head(h)
        return {
            'r_alpha': r_ab[:, 0], 'r_beta': r_ab[:, 1],
            'z_alpha': z_ab[:, 0], 'z_beta': z_ab[:, 1],
            'pnuc_logit': pnuc_logit,
            'theta_vec': theta_vec,
        }

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LocalizationDistributionModel(
    n_genes=len(shared_genes),
    n_cell_types=len(unique_cell_types),
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,} ({n_params*4/1e6:.1f} MB)")
print(f"Device: {device}")


In [ ]:
# ==============================================================================
# CELL S5-8-DIST — Train localization distribution model
# Baselines:
#   1. Gene-level empirical distribution
#   2. Cell-type gene empirical distribution
#   3. Spatial-kNN empirical distribution
#
# This replaces old Cell 16.
# It removes PC2PC evaluation and compares learned model vs 3 empirical baselines.
# ==============================================================================

import os
import gc
import time
import json
import math
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.distributions import Beta

from sklearn.neighbors import BallTree

print("=" * 80)
print("SUBSTEP 5E: Training localization distribution model")
print("Baselines: gene empirical / cell-type gene empirical / spatial-kNN empirical")
print("=" * 80)

# ------------------------------------------------------------------------------
# 0. Required variable checks
# ------------------------------------------------------------------------------

required_vars = [
    "model",
    "device",
    "train_loader",
    "val_loader",
    "train_dataset",
    "val_dataset",
    "training_examples",
    "validation_examples",
    "shared_genes",
    "unique_cell_types",
    "cell_ids_step4",
    "cell_type_indices",
    "gene_to_col",
    "step4_cell_to_row",
    "centroids",
    "cell_ids_geom",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable: {v}")

# These are optional but strongly recommended.
if "P_DROP_MIN" not in globals():
    P_DROP_MIN = 0.15
if "P_DROP_MAX" not in globals():
    P_DROP_MAX = 0.40

print(f"Training examples: {len(training_examples):,}")
print(f"Validation examples: {len(validation_examples):,}")
print(f"Genes: {len(shared_genes):,}")
print(f"Cell types: {len(unique_cell_types):,}")
print(f"Device: {device}")

# ------------------------------------------------------------------------------
# 1. Training hyperparameters
# ------------------------------------------------------------------------------

RUN_NAME = "attempt_9E_distribution_empirical_baselines"

EPOCHS = 60
LR = 2e-4
ETA_MIN = 1e-5

RECOVERY_EVAL_EVERY = 5
RECOVERY_EVAL_BATCHES = 8
RECOVERY_EVAL_BATCH_SIZE = 256
#RECOVERY_EVAL_MAX_ITEMS = RECOVERY_EVAL_BATCHES * RECOVERY_EVAL_BATCH_SIZE
RECOVERY_EVAL_MAX_ITEMS = 4096

# Baseline pool caps keep memory controlled.
GENE_POOL_CAP = 50_000
CT_GENE_POOL_CAP = 20_000
SPATIAL_KNN_K = 80
SPATIAL_KNN_MIN_POOL = 5

EPS = 1e-4

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

LOCAL_BEST_MODEL_PATH = f"/content/{RUN_NAME}_best_model.pt"

print("\nTraining config:")
print(f"  RUN_NAME: {RUN_NAME}")
print(f"  EPOCHS: {EPOCHS}")
print(f"  LR: {LR}")
print(f"  ETA_MIN: {ETA_MIN}")
print(f"  RECOVERY_EVAL_EVERY: {RECOVERY_EVAL_EVERY}")
print(f"  RECOVERY_EVAL_MAX_ITEMS: {RECOVERY_EVAL_MAX_ITEMS}")
print(f"  GENE_POOL_CAP: {GENE_POOL_CAP}")
print(f"  CT_GENE_POOL_CAP: {CT_GENE_POOL_CAP}")
print(f"  SPATIAL_KNN_K: {SPATIAL_KNN_K}")

# ------------------------------------------------------------------------------
# 2. Build empirical baseline pools from TRAINING examples only
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("Building empirical baseline pools from training examples only")
print("=" * 80)

t_pool = time.time()

rng_pool = np.random.default_rng(123)

# gene_idx -> list of value arrays
_gene_lists = {}

# (ct_idx, gene_idx) -> list of value arrays
_ct_gene_lists = {}

# cell_id -> {gene_idx -> values}
# This stores references to existing arrays, not large duplicated concatenations.
train_cell_gene_values = {}

# Build centroid lookup from compact geometry arrays.
geom_cell_to_centroid = {
    int(cid): centroids[i].astype(np.float32)
    for i, cid in enumerate(cell_ids_geom)
}

for ex in training_examples:
    gid = int(ex["gene_idx"])
    ct = int(ex["ct_idx"])
    cid = int(ex["cell_id"])
    vals = ex["values"].astype(np.float32)

    _gene_lists.setdefault(gid, []).append(vals)
    _ct_gene_lists.setdefault((ct, gid), []).append(vals)

    if cid not in train_cell_gene_values:
        train_cell_gene_values[cid] = {}
    train_cell_gene_values[cid][gid] = vals

def _concat_and_cap(list_of_arrays, cap, rng):
    """
    Concatenate empirical molecule-coordinate arrays and cap them to avoid
    huge memory growth.
    Each row is [r_norm, theta, z_rel, p_nuclear].
    """
    if len(list_of_arrays) == 0:
        return None

    arr = np.concatenate(list_of_arrays, axis=0).astype(np.float32)

    if len(arr) > cap:
        idx = rng.choice(len(arr), size=cap, replace=False)
        arr = arr[idx]

    return arr

gene_empirical_pools = {}
for gid, arr_list in _gene_lists.items():
    gene_empirical_pools[gid] = _concat_and_cap(arr_list, GENE_POOL_CAP, rng_pool)

ct_gene_empirical_pools = {}
for key, arr_list in _ct_gene_lists.items():
    ct_gene_empirical_pools[key] = _concat_and_cap(arr_list, CT_GENE_POOL_CAP, rng_pool)

del _gene_lists, _ct_gene_lists
gc.collect()

print(f"Gene empirical pools: {len(gene_empirical_pools):,}")
print(f"Cell-type gene empirical pools: {len(ct_gene_empirical_pools):,}")
print(f"Training cells with molecule pools: {len(train_cell_gene_values):,}")

# Build BallTree over training-cell centroids for spatial-kNN baseline.
train_spatial_cells = []
train_spatial_coords = []

for cid in train_cell_gene_values.keys():
    if cid in geom_cell_to_centroid:
        train_spatial_cells.append(cid)
        train_spatial_coords.append(geom_cell_to_centroid[cid])

train_spatial_cells = np.array(train_spatial_cells, dtype=np.int64)
train_spatial_coords = np.asarray(train_spatial_coords, dtype=np.float32)

if len(train_spatial_cells) == 0:
    spatial_tree = None
    print("WARNING: No training-cell centroids found. Spatial-kNN baseline will fall back.")
else:
    spatial_tree = BallTree(train_spatial_coords)
    print(f"Spatial BallTree cells: {len(train_spatial_cells):,}")

print(f"Baseline pools built in {(time.time() - t_pool) / 60:.2f} min")

# ------------------------------------------------------------------------------
# 3. Distribution loss
# ------------------------------------------------------------------------------

MEAN_LOSS_WEIGHT = 0.05  # start with 0.05; try 0.10 later if stable
THETA_LOSS_WEIGHT = 0.05

def beta_nll(x, alpha, beta):
    x = x.clamp(EPS, 1 - EPS)
    dist = Beta(alpha.unsqueeze(1), beta.unsqueeze(1))
    return -dist.log_prob(x)


def mean_location_loss(outputs, target_coords, target_mask):
    """
    Auxiliary loss.

    It compares:
      predicted mean r/z from the Beta distributions
    with:
      actual mean r/z of the hidden target molecules.

    This helps align the training objective with coordNN-style recovery.
    """
    r_target = target_coords[:, :, 0]
    z_target = target_coords[:, :, 2]

    denom_per_example = target_mask.sum(dim=1).clamp_min(1.0)

    r_true_mean = (r_target * target_mask).sum(dim=1) / denom_per_example
    z_true_mean = (z_target * target_mask).sum(dim=1) / denom_per_example

    r_pred_mean = outputs["r_alpha"] / (
        outputs["r_alpha"] + outputs["r_beta"] + 1e-8
    )
    z_pred_mean = outputs["z_alpha"] / (
        outputs["z_alpha"] + outputs["z_beta"] + 1e-8
    )

    r_mean_loss = F.smooth_l1_loss(r_pred_mean, r_true_mean)
    z_mean_loss = F.smooth_l1_loss(z_pred_mean, z_true_mean)

    return r_mean_loss + z_mean_loss

def theta_direction_loss(theta_vec, target_coords, target_mask):
    """
    Circular direction loss for theta.

    It compares the model's predicted angular direction vector
    with the target molecule theta values using sin/cos representation.
    """
    target_theta = target_coords[:, :, 1]

    target_vec = torch.stack(
        [torch.sin(target_theta), torch.cos(target_theta)],
        dim=-1
    )

    pred_vec = F.normalize(theta_vec, dim=-1).unsqueeze(1)
    target_vec = F.normalize(target_vec, dim=-1)

    cosine_sim = (pred_vec * target_vec).sum(dim=-1)
    loss = 1.0 - cosine_sim

    denom = target_mask.sum() + 1e-8
    return (loss * target_mask).sum() / denom

def distribution_loss(outputs, target_coords, target_pnuc, target_mask):
    """
    Model learns:
      r_norm distribution through Beta(alpha, beta)
      z_rel distribution through Beta(alpha, beta)
      p_nuclear through Bernoulli/logit BCE

    Added:
      mean_location_loss encourages the predicted r/z distribution center
      to match the hidden target molecules' average r/z location.
    """
    r = target_coords[:, :, 0]
    z = target_coords[:, :, 2]

    r_nll = beta_nll(r, outputs["r_alpha"], outputs["r_beta"])
    z_nll = beta_nll(z, outputs["z_alpha"], outputs["z_beta"])

    pnuc_logits = outputs["pnuc_logit"].unsqueeze(1).expand_as(target_pnuc)
    pnuc_bce = F.binary_cross_entropy_with_logits(
        pnuc_logits,
        target_pnuc,
        reduction="none"
    )

    denom = target_mask.sum() + 1e-8

    r_loss = (r_nll * target_mask).sum() / denom
    z_loss = (z_nll * target_mask).sum() / denom
    pnuc_loss = (pnuc_bce * target_mask).sum() / denom

    mean_loss = mean_location_loss(outputs, target_coords, target_mask)
    theta_loss = theta_direction_loss(
    outputs["theta_vec"],
    target_coords,
    target_mask
    )

    total = (
        r_loss
        + z_loss
        + 0.5 * pnuc_loss
        + MEAN_LOSS_WEIGHT * mean_loss
        + THETA_LOSS_WEIGHT * theta_loss
    )

    return total, r_loss, z_loss, pnuc_loss

# ------------------------------------------------------------------------------
# 4. Learned model sampler
# ------------------------------------------------------------------------------

@torch.no_grad()
def sample_from_learned_distribution(outputs, target_mask, context, context_mask):
    """
    Samples molecule coordinates from the learned localization distribution.

    Outputs:
      r_norm sampled from predicted Beta distribution
      theta sampled around predicted theta direction
      z_rel sampled from predicted Beta distribution

    Shape:
      returns [B, K, 3], where:
        B = batch size
        K = max target molecules
    """

    dev = target_mask.device

    # B = batch size, K = number of target slots
    B, K = target_mask.shape

    # ------------------------------------------------------------
    # Sample r_norm from predicted Beta distribution
    # ------------------------------------------------------------
    r_dist = torch.distributions.Beta(
        outputs["r_alpha"].clamp_min(1e-4),
        outputs["r_beta"].clamp_min(1e-4),
    )

    r = r_dist.sample((K,)).transpose(0, 1).to(dev)
    r = r.clamp(0.0, 1.0)

    # ------------------------------------------------------------
    # Sample z_rel from predicted Beta distribution
    # ------------------------------------------------------------
    z_dist = torch.distributions.Beta(
        outputs["z_alpha"].clamp_min(1e-4),
        outputs["z_beta"].clamp_min(1e-4),
    )

    z = z_dist.sample((K,)).transpose(0, 1).to(dev)
    z = z.clamp(0.0, 1.0)

    # ------------------------------------------------------------
    # Sample theta using model-predicted theta direction
    # ------------------------------------------------------------
    # theta_vec is trained as [sin(theta), cos(theta)].
    theta_vec = F.normalize(outputs["theta_vec"], dim=-1)

    # Convert predicted sin/cos vector back to angle.
    theta_mu = torch.atan2(theta_vec[:, 0], theta_vec[:, 1])  # [B]

    # Repeat one predicted theta direction across target slots.
    theta = theta_mu.unsqueeze(1).expand(B, K).clone()

    # Add small angular jitter so all predicted molecules do not collapse
    # to the exact same angle.
    theta_jitter = 0.20
    theta = theta + torch.randn((B, K), device=dev) * theta_jitter

    # Wrap theta back to [-pi, pi].
    theta = ((theta + torch.pi) % (2 * torch.pi)) - torch.pi

    # ------------------------------------------------------------
    # Return sampled coordinates
    # ------------------------------------------------------------
    return torch.stack([r, theta, z], dim=-1)

# ------------------------------------------------------------------------------
# 5. Empirical baseline samplers
# ------------------------------------------------------------------------------

def _sample_empirical_pool(pool, n, rng):
    """
    Sample n rows from an empirical localization pool.
    Each pool row is [r_norm, theta, z_rel, p_nuclear].
    Returns coordinates [r_norm, theta, z_rel].
    """
    if pool is None or len(pool) == 0:
        r = rng.uniform(0, 1, size=n)
        theta = rng.uniform(-np.pi, np.pi, size=n)
        z = rng.uniform(0, 1, size=n)
        return np.stack([r, theta, z], axis=1).astype(np.float32)

    idx = rng.integers(0, len(pool), size=n)
    vals = pool[idx, :3].astype(np.float32)

    vals[:, 0] = np.clip(vals[:, 0], 0, 1)
    vals[:, 1] = np.arctan2(np.sin(vals[:, 1]), np.cos(vals[:, 1]))
    vals[:, 2] = np.clip(vals[:, 2], 0, 1)

    return vals

def sample_gene_empirical(ex, n, rng):
    """
    Baseline 1:
    Sample from all observed training molecules of the same gene.
    """
    gid = int(ex["gene_idx"])
    pool = gene_empirical_pools.get(gid, None)
    return _sample_empirical_pool(pool, n, rng)

def sample_ct_gene_empirical(ex, n, rng):
    """
    Baseline 2:
    Sample from observed training molecules of the same cell type and same gene.
    Fallback: gene-level empirical.
    """
    gid = int(ex["gene_idx"])
    ct = int(ex["ct_idx"])

    pool = ct_gene_empirical_pools.get((ct, gid), None)

    if pool is None or len(pool) < 3:
        pool = gene_empirical_pools.get(gid, None)

    return _sample_empirical_pool(pool, n, rng)

def sample_spatial_knn_empirical(ex, n, rng):
    """
    Baseline 3:
    Find spatially nearby training cells, collect molecules of the same gene
    from those cells, and sample from their empirical localization distribution.

    Fallback order:
      spatial-kNN same-gene pool
      cell-type gene empirical
      gene empirical
      uniform
    """
    gid = int(ex["gene_idx"])
    cid = int(ex["cell_id"])

    if spatial_tree is None or cid not in geom_cell_to_centroid:
        return sample_ct_gene_empirical(ex, n, rng)

    query_coord = geom_cell_to_centroid[cid].reshape(1, -1)

    k = min(SPATIAL_KNN_K, len(train_spatial_cells))
    _, nn_idx = spatial_tree.query(query_coord, k=k)

    candidate_arrays = []

    for idx in nn_idx[0]:
        nb_cid = int(train_spatial_cells[idx])
        gene_dict = train_cell_gene_values.get(nb_cid, {})
        vals = gene_dict.get(gid, None)

        if vals is not None and len(vals) > 0:
            candidate_arrays.append(vals)

    if len(candidate_arrays) > 0:
        pool = np.concatenate(candidate_arrays, axis=0).astype(np.float32)

        if len(pool) >= SPATIAL_KNN_MIN_POOL:
            return _sample_empirical_pool(pool, n, rng)

    return sample_ct_gene_empirical(ex, n, rng)

# ------------------------------------------------------------------------------
# 6. Held-out recovery metrics
# ------------------------------------------------------------------------------

def _wasserstein_1d(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.nan

    a = np.sort(np.asarray(a, dtype=np.float64))
    b = np.sort(np.asarray(b, dtype=np.float64))

    q = np.linspace(0, 1, max(len(a), len(b)))
    aq = np.interp(q, np.linspace(0, 1, len(a)), a)
    bq = np.interp(q, np.linspace(0, 1, len(b)), b)

    return float(np.mean(np.abs(aq - bq)))

def _theta_set_distance(pred_theta, true_theta):
    if len(pred_theta) == 0 or len(true_theta) == 0:
        return np.nan

    pred_theta = np.asarray(pred_theta, dtype=np.float64)
    true_theta = np.asarray(true_theta, dtype=np.float64)

    # Circular pairwise absolute angular distance.
    d = np.abs(
        np.arctan2(
            np.sin(pred_theta[:, None] - true_theta[None, :]),
            np.cos(pred_theta[:, None] - true_theta[None, :])
        )
    )

    return float(np.mean(np.min(d, axis=1)) / np.pi)

def compute_recovery_metrics_np(pred, target):
    """
    pred and target are lists of arrays.
    Each array has shape [n_molecules, 3]:
      [r_norm, theta, z_rel]
    """
    matched_dists = []
    r_wass = []
    z_wass = []
    theta_dist = []
    hist_l1 = []

    for p, t in zip(pred, target):
        if len(p) == 0 or len(t) == 0:
            continue

        p = np.asarray(p, dtype=np.float32)
        t = np.asarray(t, dtype=np.float32)

        # Pairwise distance in normalized coordinate space.
        dr = p[:, None, 0] - t[None, :, 0]

        dtheta = np.arctan2(
            np.sin(p[:, None, 1] - t[None, :, 1]),
            np.cos(p[:, None, 1] - t[None, :, 1])
        ) / np.pi

        dz = p[:, None, 2] - t[None, :, 2]

        dist = np.sqrt(dr * dr + dtheta * dtheta + dz * dz)

        # Predicted-to-true nearest-neighbor distance.
        matched_dists.append(float(np.mean(np.min(dist, axis=1))))

        r_wass.append(_wasserstein_1d(p[:, 0], t[:, 0]))
        z_wass.append(_wasserstein_1d(p[:, 2], t[:, 2]))
        theta_dist.append(_theta_set_distance(p[:, 1], t[:, 1]))

        p_hist, _ = np.histogram(p[:, 0], bins=np.linspace(0, 1, 11), density=False)
        t_hist, _ = np.histogram(t[:, 0], bins=np.linspace(0, 1, 11), density=False)

        p_hist = p_hist / max(p_hist.sum(), 1)
        t_hist = t_hist / max(t_hist.sum(), 1)

        hist_l1.append(float(np.abs(p_hist - t_hist).sum()))

    return {
        "coord_nn": float(np.nanmean(matched_dists)) if matched_dists else np.nan,
        "r_wasserstein": float(np.nanmean(r_wass)) if r_wass else np.nan,
        "z_wasserstein": float(np.nanmean(z_wass)) if z_wass else np.nan,
        "theta_distance": float(np.nanmean(theta_dist)) if theta_dist else np.nan,
        "radial_hist_l1": float(np.nanmean(hist_l1)) if hist_l1 else np.nan,
    }

# ------------------------------------------------------------------------------
# 7. Deterministic held-out construction for empirical recovery evaluation
# ------------------------------------------------------------------------------

def _features_np(vals):
    """
    Convert [r_norm, theta, z_rel, p_nuclear] into model context features:
      [r_norm, sin(theta), cos(theta), z_rel, p_nuclear]
    """
    r = vals[:, 0]
    th = vals[:, 1]
    z = vals[:, 2]
    pn = vals[:, 3]

    return np.stack(
        [r, np.sin(th), np.cos(th), z, pn],
        axis=-1
    ).astype(np.float32)

def make_eval_item_from_example(ex, idx, max_context=96, max_target=24,
                                p_drop_min=0.15, p_drop_max=0.40,
                                seed=12345):
    """
    Recreates a deterministic held-out split for a validation example.
    This avoids depending on old PC2PC x0 fields.
    """
    vals = ex["values"]
    n = len(vals)

    rng = np.random.default_rng(seed + idx)

    p_drop = rng.uniform(p_drop_min, p_drop_max)

    k = int(rng.binomial(n, p_drop))
    k = max(1, min(n - 3, k, max_target))

    perm = rng.permutation(n)

    target_vals = vals[perm[:k]]
    context_vals = vals[perm[k:]]

    if len(context_vals) > max_context:
        ctx_idx = rng.choice(len(context_vals), size=max_context, replace=False)
        context_used = context_vals[ctx_idx]
    else:
        context_used = context_vals

    n_ctx = min(len(context_used), max_context)
    n_tgt = min(len(target_vals), max_target)

    ctx_padded = np.zeros((max_context, 5), dtype=np.float32)
    if n_ctx > 0:
        ctx_padded[:n_ctx] = _features_np(context_used[:n_ctx])

    ctx_mask = np.zeros(max_context, dtype=np.float32)
    ctx_mask[:n_ctx] = 1.0

    target_coords = np.zeros((max_target, 3), dtype=np.float32)
    target_pnuc = np.zeros(max_target, dtype=np.float32)
    target_mask = np.zeros(max_target, dtype=np.float32)

    target_coords[:n_tgt] = target_vals[:n_tgt, :3]
    target_pnuc[:n_tgt] = target_vals[:n_tgt, 3]
    target_mask[:n_tgt] = 1.0

    c_area = float(ex.get("cell_area", 100.0))
    n_area = float(ex.get("nuc_area", 30.0))

    # Must match training geom_dim=9.
    if n_ctx > 0:
        ctx_features = ctx_padded[:n_ctx]

        ctx_r_mean = float(ctx_features[:, 0].mean())
        ctx_r_std = float(ctx_features[:, 0].std())

        ctx_z_mean = float(ctx_features[:, 3].mean())
        ctx_z_std = float(ctx_features[:, 3].std())

        ctx_pnuc_mean = float(ctx_features[:, 4].mean())
    else:
        ctx_r_mean = 0.5
        ctx_r_std = 0.0
        ctx_z_mean = 0.5
        ctx_z_std = 0.0
        ctx_pnuc_mean = 0.0

    geom = np.array([
        c_area / 500.0,
        n_area / 200.0,
        n_area / c_area if c_area > 0 else 0.3,
        n_ctx / 50.0,
        ctx_r_mean,
        ctx_r_std,
        ctx_z_mean,
        ctx_z_std,
        ctx_pnuc_mean,
    ], dtype=np.float32)

    return {
        "context": ctx_padded,
        "context_mask": ctx_mask,
        "target_coords": target_coords,
        "target_pnuc": target_pnuc,
        "target_mask": target_mask,
        "gene_idx": int(ex["gene_idx"]),
        "ct_idx": int(ex["ct_idx"]),
        "geom": geom,
        "ex": ex,
        "n_tgt": n_tgt,
    }

@torch.no_grad()
def evaluate_heldout_recovery_empirical(model, validation_examples, device,
                                        max_items=2048,
                                        batch_size=256,
                                        seed=12345):
    """
    Evaluates learned model and three empirical baselines on the same held-out
    molecules from validation examples.

    Lower is better for all metrics:
      coord_nn
      r_wasserstein
      z_wasserstein
      theta_distance
      radial_hist_l1
    """
    model.eval()

    n_eval = min(max_items, len(validation_examples))

    learned_preds = []
    gene_preds = []
    ct_gene_preds = []
    spatial_knn_preds = []
    targets = []

    rng_eval = np.random.default_rng(seed)

    for start in range(0, n_eval, batch_size):
        end = min(start + batch_size, n_eval)

        items = [
            make_eval_item_from_example(
                validation_examples[i],
                idx=i,
                p_drop_min=P_DROP_MIN,
                p_drop_max=P_DROP_MAX,
                seed=seed
            )
            for i in range(start, end)
        ]

        ctx = torch.tensor(np.stack([it["context"] for it in items]), dtype=torch.float32, device=device)
        ctx_mask = torch.tensor(np.stack([it["context_mask"] for it in items]), dtype=torch.float32, device=device)
        target_coords_t = torch.tensor(np.stack([it["target_coords"] for it in items]), dtype=torch.float32, device=device)
        target_mask_t = torch.tensor(np.stack([it["target_mask"] for it in items]), dtype=torch.float32, device=device)

        gene_idx_t = torch.tensor([it["gene_idx"] for it in items], dtype=torch.long, device=device)
        ct_idx_t = torch.tensor([it["ct_idx"] for it in items], dtype=torch.long, device=device)
        geom_t = torch.tensor(np.stack([it["geom"] for it in items]), dtype=torch.float32, device=device)

        outputs = model(ctx, ctx_mask, gene_idx_t, ct_idx_t, geom_t)
        learned_sample_t = sample_from_learned_distribution(outputs, target_mask_t, ctx, ctx_mask)

        learned_sample = learned_sample_t.detach().cpu().numpy()
        target_coords_np = target_coords_t.detach().cpu().numpy()
        target_mask_np = target_mask_t.detach().cpu().numpy()

        for local_i, it in enumerate(items):
            m = target_mask_np[local_i] > 0
            n_tgt = int(m.sum())

            if n_tgt <= 0:
                continue

            true_coords = target_coords_np[local_i, m, :]
            learned_coords = learned_sample[local_i, m, :]

            ex = it["ex"]

            gene_coords = sample_gene_empirical(ex, n_tgt, rng_eval)
            ct_gene_coords = sample_ct_gene_empirical(ex, n_tgt, rng_eval)
            spatial_coords = sample_spatial_knn_empirical(ex, n_tgt, rng_eval)

            targets.append(true_coords)
            learned_preds.append(learned_coords)
            gene_preds.append(gene_coords)
            ct_gene_preds.append(ct_gene_coords)
            spatial_knn_preds.append(spatial_coords)

    metrics = {}

    all_methods = {
        "learned": learned_preds,
        "gene_emp": gene_preds,
        "ct_gene_emp": ct_gene_preds,
        "spatial_knn_emp": spatial_knn_preds,
    }

    for name, preds in all_methods.items():
        m = compute_recovery_metrics_np(preds, targets)
        for k, v in m.items():
            metrics[f"{name}_{k}"] = v

    # Improvements relative to each baseline.
    learned_coord = metrics.get("learned_coord_nn", np.nan)

    for base in ["gene_emp", "ct_gene_emp", "spatial_knn_emp"]:
        base_coord = metrics.get(f"{base}_coord_nn", np.nan)
        metrics[f"improvement_vs_{base}_coord_nn"] = float(
            (base_coord - learned_coord) / (abs(base_coord) + 1e-8)
        )

    metrics["n_eval_examples"] = len(targets)
    metrics["n_eval_target_molecules"] = int(sum(len(t) for t in targets))

    return metrics


# ------------------------------------------------------------------------------
# Compatibility aliases / config
# ------------------------------------------------------------------------------

# If the sampling function was defined under the old name, keep compatibility.
if "sample_from_learned_distribution" not in globals():
    if "sample_from_distribution" in globals():
        sample_from_learned_distribution = sample_from_distribution
    else:
        raise NameError(
            "Missing sample_from_learned_distribution or sample_from_distribution. "
            "Run the model/helper-definition cell first."
        )

RUN_NAME = "attempt_9E_distribution_empirical_baselines"

EPOCHS = globals().get("EPOCHS", 60)
LR = globals().get("LR", 2e-4)
ETA_MIN = globals().get("ETA_MIN", 1e-5)

RECOVERY_EVAL_EVERY = globals().get("RECOVERY_EVAL_EVERY", 5)
RECOVERY_EVAL_MAX_ITEMS = globals().get("RECOVERY_EVAL_MAX_ITEMS", 2048)
RECOVERY_EVAL_BATCH_SIZE = globals().get("RECOVERY_EVAL_BATCH_SIZE", 256)

LOCAL_BEST_MODEL_PATH = f"/content/{RUN_NAME}_best_model.pt"

print("\nTraining configuration:")
print(f"  RUN_NAME: {RUN_NAME}")
print(f"  EPOCHS: {EPOCHS}")
print(f"  LR: {LR}")
print(f"  ETA_MIN: {ETA_MIN}")
print(f"  RECOVERY_EVAL_EVERY: {RECOVERY_EVAL_EVERY}")
print(f"  RECOVERY_EVAL_MAX_ITEMS: {RECOVERY_EVAL_MAX_ITEMS}")
print(f"  RECOVERY_EVAL_BATCH_SIZE: {RECOVERY_EVAL_BATCH_SIZE}")
print("  Baselines: gene_emp, ct_gene_emp, spatial_knn_emp")


# ------------------------------------------------------------------------------
# Optimizer / scheduler / history containers
# ------------------------------------------------------------------------------

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler_lr = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=ETA_MIN
)

train_losses, val_losses = [], []
train_r_losses, train_z_losses, train_pnuc_losses = [], [], []
val_r_losses, val_z_losses, val_pnuc_losses = [], [], []

recovery_history = []
best_val_loss = float("inf")

# Also track best model by held-out coordinate recovery.
best_coord_nn = float("inf")
BEST_COORD_MODEL_PATH = f"/content/{RUN_NAME}_best_coordNN_model.pt"

t0_train = time.time()


# ------------------------------------------------------------------------------
# Main training loop
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINING: learned localization distribution model")
print("=" * 80)

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # --------------------------------------------------------------------------
    # Train
    # --------------------------------------------------------------------------
    model.train()

    train_total = 0.0
    train_r = 0.0
    train_z = 0.0
    train_p = 0.0
    n_train_batches = 0

    for batch in train_loader:
        ctx = batch["context"].to(device)
        ctx_mask = batch["context_mask"].to(device)
        target = batch["target_coords"].to(device)
        target_pnuc = batch["target_pnuc"].to(device)
        mask = batch["target_mask"].to(device)

        gene_idx = batch["gene_idx"].to(device)
        ct_idx = batch["ct_idx"].to(device)
        geom = batch["geom"].to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(ctx, ctx_mask, gene_idx, ct_idx, geom)

        loss, r_loss, z_loss, pnuc_loss = distribution_loss(
            outputs,
            target,
            target_pnuc,
            mask
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_total += float(loss.item())
        train_r += float(r_loss.item())
        train_z += float(z_loss.item())
        train_p += float(pnuc_loss.item())
        n_train_batches += 1

    avg_train = train_total / max(n_train_batches, 1)
    avg_train_r = train_r / max(n_train_batches, 1)
    avg_train_z = train_z / max(n_train_batches, 1)
    avg_train_p = train_p / max(n_train_batches, 1)

    train_losses.append(avg_train)
    train_r_losses.append(avg_train_r)
    train_z_losses.append(avg_train_z)
    train_pnuc_losses.append(avg_train_p)

    # --------------------------------------------------------------------------
    # Validation loss
    # --------------------------------------------------------------------------
    model.eval()

    val_total = 0.0
    val_r = 0.0
    val_z = 0.0
    val_p = 0.0
    n_val_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            ctx = batch["context"].to(device)
            ctx_mask = batch["context_mask"].to(device)
            target = batch["target_coords"].to(device)
            target_pnuc = batch["target_pnuc"].to(device)
            mask = batch["target_mask"].to(device)

            gene_idx = batch["gene_idx"].to(device)
            ct_idx = batch["ct_idx"].to(device)
            geom = batch["geom"].to(device)

            outputs = model(ctx, ctx_mask, gene_idx, ct_idx, geom)

            loss, r_loss, z_loss, pnuc_loss = distribution_loss(
                outputs,
                target,
                target_pnuc,
                mask
            )

            val_total += float(loss.item())
            val_r += float(r_loss.item())
            val_z += float(z_loss.item())
            val_p += float(pnuc_loss.item())
            n_val_batches += 1

    avg_val = val_total / max(n_val_batches, 1)
    avg_val_r = val_r / max(n_val_batches, 1)
    avg_val_z = val_z / max(n_val_batches, 1)
    avg_val_p = val_p / max(n_val_batches, 1)

    val_losses.append(avg_val)
    val_r_losses.append(avg_val_r)
    val_z_losses.append(avg_val_z)
    val_pnuc_losses.append(avg_val_p)

    # --------------------------------------------------------------------------
    # LR schedule
    # --------------------------------------------------------------------------
    scheduler_lr.step()

    # --------------------------------------------------------------------------
    # Held-out recovery evaluation against empirical baselines
    # --------------------------------------------------------------------------
    recovery_metrics = None

    should_eval_recovery = (
        epoch == 0
        or (epoch + 1) % RECOVERY_EVAL_EVERY == 0
        or (epoch + 1) == EPOCHS
    )

    if should_eval_recovery:
        recovery_metrics = evaluate_heldout_recovery_empirical(
            model=model,
            validation_examples=validation_examples,
            device=device,
            max_items=RECOVERY_EVAL_MAX_ITEMS,
            batch_size=RECOVERY_EVAL_BATCH_SIZE,
            seed=12345,
        )

        recovery_metrics["epoch"] = epoch + 1
        recovery_history.append(recovery_metrics)

    # --------------------------------------------------------------------------
    # Save best model by held-out coordinate recovery
    # --------------------------------------------------------------------------

    # Track best model by held-out coordinate recovery too

    if recovery_metrics is not None:
        current_coord = recovery_metrics.get("learned_coord_nn", np.nan)

        if np.isfinite(current_coord) and current_coord < best_coord_nn:
            best_coord_nn = current_coord
            torch.save(model.state_dict(), BEST_COORD_MODEL_PATH)
            print(f"  ★ saved best coordNN model: {best_coord_nn:.4f}")
    # --------------------------------------------------------------------------
    # Save best model by validation distribution loss
    # --------------------------------------------------------------------------
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), LOCAL_BEST_MODEL_PATH)
        marker = " ★ saved"
    else:
        marker = ""

    # --------------------------------------------------------------------------
    # Console logging
    # --------------------------------------------------------------------------
    if (epoch + 1) % 5 == 0 or epoch == 0 or recovery_metrics is not None:
        rec_msg = ""

        if recovery_metrics is not None:
            rec_msg = (
                f" | learned coordNN={recovery_metrics.get('learned_coord_nn', np.nan):.4f}"
                f" | gene={recovery_metrics.get('gene_emp_coord_nn', np.nan):.4f}"
                f" | ct_gene={recovery_metrics.get('ct_gene_emp_coord_nn', np.nan):.4f}"
                f" | spatial_kNN={recovery_metrics.get('spatial_knn_emp_coord_nn', np.nan):.4f}"
                f" | imp_vs_gene={100*recovery_metrics.get('improvement_vs_gene_emp_coord_nn', np.nan):.2f}%"
                f" | imp_vs_ct_gene={100*recovery_metrics.get('improvement_vs_ct_gene_emp_coord_nn', np.nan):.2f}%"
                f" | imp_vs_spatial={100*recovery_metrics.get('improvement_vs_spatial_knn_emp_coord_nn', np.nan):.2f}%"
            )

        print(
            f"Epoch {epoch+1:3d}/{EPOCHS} | "
            f"Train total/r/z/pnuc: "
            f"{avg_train:.4f}/{avg_train_r:.4f}/{avg_train_z:.4f}/{avg_train_p:.4f} | "
            f"Val total/r/z/pnuc: "
            f"{avg_val:.4f}/{avg_val_r:.4f}/{avg_val_z:.4f}/{avg_val_p:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.2e} | "
            f"{time.time() - epoch_start:.1f}s{marker}{rec_msg}"
        )

print("\n" + "=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Best local model path: {LOCAL_BEST_MODEL_PATH}")
print(f"Total training time: {(time.time() - t0_train) / 60:.1f} min")

model.load_state_dict(torch.load(LOCAL_BEST_MODEL_PATH, map_location=device))
model.to(device)
model.eval()

print("Loaded best model weights.")

In [ ]:
# ==============================================================================
# REPLACEMENT CHECKPOINT DIST
# Save model/report with empirical-baseline metrics
# ==============================================================================

import os
import json
import torch
import numpy as np
import pandas as pd

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RUN_NAME = "attempt_9E_distribution_empirical_baselines"

LOCAL_BEST_MODEL_PATH = f"/content/{RUN_NAME}_best_model.pt"
DRIVE_FULL_CKPT = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_best_model.pt")
REPORT_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_training_report.txt")
RECOVERY_CSV_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_recovery_history.csv")
CURVES_NPZ_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_training_curves.npz")

print("=" * 80)
print("CHECKPOINT DIST: Saving learned distribution model and baseline report")
print("=" * 80)

# Make sure best local weights are loaded before saving full checkpoint.
if os.path.exists(LOCAL_BEST_MODEL_PATH):
    model.load_state_dict(torch.load(LOCAL_BEST_MODEL_PATH, map_location=device))
    model.to(device)
    model.eval()
    print(f"Loaded best local weights from: {LOCAL_BEST_MODEL_PATH}")
else:
    print("WARNING: Local best model path not found. Saving current model state.")

checkpoint_payload = {
    "model_state_dict": model.state_dict(),

    "n_genes": len(shared_genes),
    "n_cell_types": len(unique_cell_types),
    "gene_names": list(shared_genes),
    "cell_type_names": list(unique_cell_types),

    "architecture": "Conditional_Beta_Localization_Distribution_v1",
    "run_name": RUN_NAME,

    "training_config": {
        "epochs": EPOCHS,
        "batch_size": getattr(train_loader, "batch_size", None),
        "lr": LR,
        "eta_min": ETA_MIN,

        "coord_model": "beta_r_beta_z_bernoulli_pnuc_theta_from_context",
        "p_drop_min": P_DROP_MIN,
        "p_drop_max": P_DROP_MAX,

        "baselines": [
            "gene_empirical_distribution",
            "cell_type_gene_empirical_distribution",
            "spatial_knn_empirical",
        ],

        "n_train_examples": len(training_examples),
        "n_val_examples": len(validation_examples),

        "recovery_eval_every": RECOVERY_EVAL_EVERY,
        "recovery_eval_max_items": RECOVERY_EVAL_MAX_ITEMS,
        "recovery_eval_batch_size": RECOVERY_EVAL_BATCH_SIZE,
    },

    "train_losses": train_losses,
    "val_losses": val_losses,
    "train_r_losses": train_r_losses,
    "train_z_losses": train_z_losses,
    "train_pnuc_losses": train_pnuc_losses,
    "val_r_losses": val_r_losses,
    "val_z_losses": val_z_losses,
    "val_pnuc_losses": val_pnuc_losses,

    "recovery_history": recovery_history,
    "best_val_loss": float(best_val_loss),
}

torch.save(checkpoint_payload, DRIVE_FULL_CKPT)

print(f"Saved full checkpoint:")
print(f"  {DRIVE_FULL_CKPT}")

# ------------------------------------------------------------------
# Also save best model selected by held-out coordNN
# ------------------------------------------------------------------

DRIVE_BEST_COORD_CKPT = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_best_coordNN_model.pt"
)

if "BEST_COORD_MODEL_PATH" in globals() and os.path.exists(BEST_COORD_MODEL_PATH):
    best_coord_payload = checkpoint_payload.copy()

    model.load_state_dict(torch.load(BEST_COORD_MODEL_PATH, map_location=device))
    best_coord_payload["model_state_dict"] = model.state_dict()
    best_coord_payload["selection_metric"] = "best_learned_coord_nn"
    best_coord_payload["best_coord_nn"] = float(best_coord_nn)

    torch.save(best_coord_payload, DRIVE_BEST_COORD_CKPT)

    print(f"Saved best coordNN checkpoint:")
    print(f"  {DRIVE_BEST_COORD_CKPT}")
    print(f"  best_coord_nn: {best_coord_nn:.6f}")
else:
    print("WARNING: No best coordNN local checkpoint found. Skipping coordNN checkpoint save.")

# Save recovery history as CSV.
if recovery_history:
    recovery_df = pd.DataFrame(recovery_history)
    recovery_df.to_csv(RECOVERY_CSV_PATH, index=False)
    print(f"Saved recovery history CSV:")
    print(f"  {RECOVERY_CSV_PATH}")
else:
    recovery_df = pd.DataFrame()
    print("WARNING: recovery_history is empty. No recovery CSV saved.")

# Save curves as NPZ for easy reload.
np.savez_compressed(
    CURVES_NPZ_PATH,
    train_losses=np.array(train_losses, dtype=np.float32),
    val_losses=np.array(val_losses, dtype=np.float32),
    train_r_losses=np.array(train_r_losses, dtype=np.float32),
    train_z_losses=np.array(train_z_losses, dtype=np.float32),
    train_pnuc_losses=np.array(train_pnuc_losses, dtype=np.float32),
    val_r_losses=np.array(val_r_losses, dtype=np.float32),
    val_z_losses=np.array(val_z_losses, dtype=np.float32),
    val_pnuc_losses=np.array(val_pnuc_losses, dtype=np.float32),
)

print(f"Saved training curves NPZ:")
print(f"  {CURVES_NPZ_PATH}")

# Text report.
with open(REPORT_PATH, "w") as f:
    f.write("=" * 80 + "\n")
    f.write("STEP 5 LOCALIZATION DISTRIBUTION MODEL TRAINING REPORT\n")
    f.write("=" * 80 + "\n\n")

    f.write(f"Run: {RUN_NAME}\n")
    f.write(f"Checkpoint directory: {CHECKPOINT_DIR}\n")
    f.write(f"Best validation total loss: {best_val_loss:.6f}\n\n")

    f.write("Training config:\n")
    for k, v in checkpoint_payload["training_config"].items():
        f.write(f"  {k}: {v}\n")

    f.write("\nEpoch losses:\n")
    f.write(
        "epoch\t"
        "train_total\ttrain_r\ttrain_z\ttrain_pnuc\t"
        "val_total\tval_r\tval_z\tval_pnuc\n"
    )

    for idx in range(len(train_losses)):
        f.write(
            f"{idx+1}\t"
            f"{train_losses[idx]:.6f}\t"
            f"{train_r_losses[idx]:.6f}\t"
            f"{train_z_losses[idx]:.6f}\t"
            f"{train_pnuc_losses[idx]:.6f}\t"
            f"{val_losses[idx]:.6f}\t"
            f"{val_r_losses[idx]:.6f}\t"
            f"{val_z_losses[idx]:.6f}\t"
            f"{val_pnuc_losses[idx]:.6f}\n"
        )

    f.write("\nHeld-out recovery metrics:\n")
    f.write("Compared methods:\n")
    f.write("  learned\n")
    f.write("  gene_emp\n")
    f.write("  ct_gene_emp\n")
    f.write("  spatial_knn_emp\n\n")

    if recovery_history:
        keys = sorted([k for k in recovery_history[-1].keys() if k != "epoch"])
        f.write("epoch\t" + "\t".join(keys) + "\n")

        for rec in recovery_history:
            f.write(str(rec.get("epoch", "")))

            for k in keys:
                val = rec.get(k, np.nan)
                if isinstance(val, (int, np.integer)):
                    f.write(f"\t{val}")
                elif isinstance(val, (float, np.floating)):
                    f.write(f"\t{val:.6f}")
                else:
                    f.write(f"\t{val}")

            f.write("\n")
    else:
        f.write("No recovery metrics were recorded.\n")

print(f"Saved training report:")
print(f"  {REPORT_PATH}")

print("\nCheckpoint save complete.")

In [ ]:
# ==============================================================================
# REPLACEMENT S5-DIST TRAINING CURVES
# Plot loss curves + empirical-baseline recovery curves
# ==============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
RUN_NAME = "attempt_9E_distribution_empirical_baselines"

CURVE_PATH = f"/content/{RUN_NAME}_training_curves.png"
DRIVE_CURVE_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_training_curves.png")

RECOVERY_CURVE_PATH = f"/content/{RUN_NAME}_recovery_baseline_curves.png"
DRIVE_RECOVERY_CURVE_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_recovery_baseline_curves.png")

print("=" * 80)
print("S5-DIST: Plotting training curves")
print("=" * 80)

epochs = np.arange(1, len(train_losses) + 1)

# ------------------------------------------------------------------------------
# 1. Training / validation loss curves
# ------------------------------------------------------------------------------

fig, axes = plt.subplots(1, 4, figsize=(24, 4))

axes[0].plot(epochs, train_losses, label="Train total")
axes[0].plot(epochs, val_losses, label="Val total")
axes[0].set_title("Total distribution loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, train_r_losses, label="Train r")
axes[1].plot(epochs, val_r_losses, label="Val r")
axes[1].set_title("r_norm Beta NLL")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, train_z_losses, label="Train z")
axes[2].plot(epochs, val_z_losses, label="Val z")
axes[2].set_title("z_rel Beta NLL")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Loss")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

axes[3].plot(epochs, train_pnuc_losses, label="Train p_nuclear")
axes[3].plot(epochs, val_pnuc_losses, label="Val p_nuclear")
axes[3].set_title("p_nuclear BCE")
axes[3].set_xlabel("Epoch")
axes[3].set_ylabel("Loss")
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(CURVE_PATH, dpi=150)
plt.savefig(DRIVE_CURVE_PATH, dpi=150)
plt.show()

print(f"Saved training loss curves:")
print(f"  {CURVE_PATH}")
print(f"  {DRIVE_CURVE_PATH}")


# ------------------------------------------------------------------------------
# 2. Held-out recovery curves against empirical baselines
# ------------------------------------------------------------------------------

if recovery_history:
    rec_df = pd.DataFrame(recovery_history).sort_values("epoch")

    fig, axes = plt.subplots(1, 5, figsize=(28, 4))

    metric_specs = [
        ("coord_nn", "coordNN ↓"),
        ("r_wasserstein", "r Wasserstein ↓"),
        ("z_wasserstein", "z Wasserstein ↓"),
        ("theta_distance", "Theta distance ↓"),
        ("radial_hist_l1", "Radial hist L1 ↓"),
    ]

    methods = [
        ("learned", "Learned"),
        ("gene_emp", "Gene empirical"),
        ("ct_gene_emp", "Cell-type gene empirical"),
        ("spatial_knn_emp", "Spatial-kNN empirical"),
    ]

    for ax, (metric_key, title) in zip(axes, metric_specs):
        for method_key, method_label in methods:
            col = f"{method_key}_{metric_key}"
            if col in rec_df.columns:
                ax.plot(
                    rec_df["epoch"],
                    rec_df[col],
                    marker="o",
                    label=method_label,
                )

        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Lower is better")
        ax.grid(True, alpha=0.3)

    axes[0].legend(loc="best")

    plt.tight_layout()
    plt.savefig(RECOVERY_CURVE_PATH, dpi=150)
    plt.savefig(DRIVE_RECOVERY_CURVE_PATH, dpi=150)
    plt.show()

    print(f"Saved recovery baseline curves:")
    print(f"  {RECOVERY_CURVE_PATH}")
    print(f"  {DRIVE_RECOVERY_CURVE_PATH}")

    print("\nLatest held-out recovery metrics:")
    latest = rec_df.iloc[-1].to_dict()

    summary_rows = []
    for method_key, method_label in methods:
        row = {"method": method_label}
        for metric_key, _ in metric_specs:
            row[metric_key] = latest.get(f"{method_key}_{metric_key}", np.nan)
        summary_rows.append(row)

    recovery_summary_df = pd.DataFrame(summary_rows)
    display(recovery_summary_df)

    print("\nLatest learned improvement over baselines using coordNN:")
    for base in ["gene_emp", "ct_gene_emp", "spatial_knn_emp"]:
        k = f"improvement_vs_{base}_coord_nn"
        if k in latest:
            print(f"  learned vs {base}: {100 * latest[k]:.2f}%")

else:
    print("No recovery_history found. Skipping empirical-baseline recovery curves.")

In [ ]:
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
RUN_NAME = "attempt_9E_distribution_empirical_baselines"

files_to_check = [
    f"{RUN_NAME}_best_model.pt",
    f"{RUN_NAME}_training_report.txt",
    f"{RUN_NAME}_recovery_history.csv",
    f"{RUN_NAME}_training_curves.npz",
    "step5_mol_processed.parquet",
    "cell_polygons.pkl",
    "nuc_polygons.pkl",
    "cell_areas.pkl",
    "nuc_areas.pkl",
    "nuc_centroids.pkl",
]

print("=" * 80)
print("CHECKING WHETHER IT IS SAFE TO DISCONNECT")
print("=" * 80)

all_ok = True

for fname in files_to_check:
    path = os.path.join(CHECKPOINT_DIR, fname)
    if os.path.exists(path):
        print(f"✓ {fname:70s} {os.path.getsize(path)/1e6:.2f} MB")
    else:
        print(f"✗ MISSING: {fname}")
        all_ok = False

if all_ok:
    print("\nSAFE: Important Step 5 training/recovery files are saved to Drive.")
else:
    print("\nNOT SAFE YET: Some required files are missing. Do not disconnect until saving them.")

===========RAN TILL THIS\=================

In [ ]:
# ==============================================================================
# RECOVERY CELL FOR 9E FINAL LARGE EVALUATION
# Run this in a fresh runtime before running final 10k/20k recovery evaluation.
# ==============================================================================

import os
import gc
import json
import pickle
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Beta

from scipy import sparse
import anndata as ad
from sklearn.neighbors import BallTree

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------------------
# 0. Mount Google Drive
# ------------------------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

print("=" * 90)
print("9E FINAL-EVAL RECOVERY: loading files from Drive")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------------------------

STEP4_EXPORT_DIR = "/content/drive/MyDrive/diffusion/step4_exports"
CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"

RUN_NAME = "attempt_9E_distribution_empirical_baselines"

POST_PRUNE_MOL_PATH = os.path.join(CHECKPOINT_DIR, "step5_mol_processed.parquet")

DENOISED_ADATA_PATH = os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad")
STEP4_CONFIG_PATH = os.path.join(STEP4_EXPORT_DIR, "step4_config.json")
WAS_CORRECTED_PATH = os.path.join(STEP4_EXPORT_DIR, "was_corrected.npy")
CELL_DATA_PATH = os.path.join(STEP4_EXPORT_DIR, "cell_data.npz")

CELL_POLYGONS_PATH = os.path.join(CHECKPOINT_DIR, "cell_polygons.pkl")
NUC_POLYGONS_PATH = os.path.join(CHECKPOINT_DIR, "nuc_polygons.pkl")
CELL_AREAS_PATH = os.path.join(CHECKPOINT_DIR, "cell_areas.pkl")
NUC_AREAS_PATH = os.path.join(CHECKPOINT_DIR, "nuc_areas.pkl")
NUC_CENTROIDS_PATH = os.path.join(CHECKPOINT_DIR, "nuc_centroids.pkl")

BEST_VAL_MODEL_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_best_model.pt")
BEST_COORD_MODEL_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_best_coordNN_model.pt")
RECOVERY_HISTORY_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_recovery_history.csv")
TRAINING_CURVES_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_training_curves.npz")
TRAINING_REPORT_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_training_report.txt")

required_files = {
    "processed Step 5 molecule table": POST_PRUNE_MOL_PATH,
    "Step 4 denoised AnnData": DENOISED_ADATA_PATH,
    "Step 4 config": STEP4_CONFIG_PATH,
    "Step 4 was_corrected": WAS_CORRECTED_PATH,
    "Step 4 compact cell_data": CELL_DATA_PATH,
    "cell polygons": CELL_POLYGONS_PATH,
    "nucleus polygons": NUC_POLYGONS_PATH,
    "cell areas": CELL_AREAS_PATH,
    "nucleus areas": NUC_AREAS_PATH,
    "nucleus centroids": NUC_CENTROIDS_PATH,
    "9E best validation-loss checkpoint": BEST_VAL_MODEL_PATH,
    "9E best coordNN checkpoint": BEST_COORD_MODEL_PATH,
    "9E recovery history CSV": RECOVERY_HISTORY_PATH,
    "9E training curves NPZ": TRAINING_CURVES_PATH,
    "9E training report": TRAINING_REPORT_PATH,
}

print("\nFiles expected from Drive:")
all_ok = True
for label, path in required_files.items():
    exists = os.path.exists(path)
    if exists:
        print(f"  ✓ {label:38s}: {path}  ({os.path.getsize(path) / 1e6:.2f} MB)")
    else:
        print(f"  ✗ MISSING {label:30s}: {path}")
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        "Some required recovery files are missing. Check the paths above before continuing."
    )

# ------------------------------------------------------------------------------
# 2. Helper
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

# ------------------------------------------------------------------------------
# 3. Load Step 5 molecule table
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("1. Loading processed Step 5 molecule table")
print("=" * 90)

print(f"Loading: {POST_PRUNE_MOL_PATH}")
mol = pd.read_parquet(POST_PRUNE_MOL_PATH)

# Compatibility alias, because some older cells may use molecules.
molecules = mol

print(f"mol shape: {mol.shape}")
print(f"mol columns: {list(mol.columns)}")
print("\nMolecule status counts:")
display(mol["status"].value_counts(dropna=False).reset_index().rename(
    columns={"index": "status", "status": "n_molecules"}
))

# ------------------------------------------------------------------------------
# 4. Load Step 4 AnnData and config/state
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("2. Loading Step 4 AnnData/config/state")
print("=" * 90)

print(f"Loading: {DENOISED_ADATA_PATH}")
denoised_adata = ad.read_h5ad(DENOISED_ADATA_PATH)

print(f"Loading: {STEP4_CONFIG_PATH}")
with open(STEP4_CONFIG_PATH, "r") as f:
    step4_config = json.load(f)

print(f"Loading: {WAS_CORRECTED_PATH}")
was_corrected = np.load(WAS_CORRECTED_PATH)

print(f"denoised_adata shape: {denoised_adata.shape}")
print(f"denoised_adata layers: {list(denoised_adata.layers.keys())}")
print(f"was_corrected shape: {was_corrected.shape}")

shared_genes = list(denoised_adata.var_names.astype(str))
cell_ids_step4 = np.array([int(x) for x in denoised_adata.obs_names.astype(str)])

gene_to_col = {g: i for i, g in enumerate(shared_genes)}
step4_cell_to_row = {int(cid): i for i, cid in enumerate(cell_ids_step4)}

# These are not strictly needed for final recovery evaluation, but useful for consistency.
X_denoised = ensure_dense(denoised_adata.X).astype(np.float32)

if "raw_molecule_counts_clean" in denoised_adata.layers:
    X_raw_counts = ensure_dense(denoised_adata.layers["raw_molecule_counts_clean"]).astype(np.float32)
elif "raw" in denoised_adata.layers:
    X_raw_counts = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)
else:
    print("WARNING: no raw count layer found in denoised_adata. Rebuilding from mol.")
    X_raw_counts = None

# Cell type indices.
if "cell_type" not in denoised_adata.obs.columns:
    raise KeyError("denoised_adata.obs must contain 'cell_type'.")

cell_type_labels = denoised_adata.obs["cell_type"].astype(str).values
unique_cell_types = sorted(pd.unique(cell_type_labels))
ct_to_idx = {ct: i for i, ct in enumerate(unique_cell_types)}
cell_type_indices = np.array([ct_to_idx[ct] for ct in cell_type_labels], dtype=np.int64)

print(f"shared_genes: {len(shared_genes)}")
print(f"cell_ids_step4: {len(cell_ids_step4):,}")
print(f"unique_cell_types: {len(unique_cell_types)}")
print(f"X_denoised shape: {X_denoised.shape}")
if X_raw_counts is not None:
    print(f"X_raw_counts shape: {X_raw_counts.shape}")
    print(f"X_raw_counts sum: {X_raw_counts.sum():,.0f}")

# ------------------------------------------------------------------------------
# 5. Load geometry support objects
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("3. Loading geometry objects")
print("=" * 90)

print(f"Loading: {CELL_POLYGONS_PATH}")
with open(CELL_POLYGONS_PATH, "rb") as f:
    cell_polygons = pickle.load(f)

print(f"Loading: {NUC_POLYGONS_PATH}")
with open(NUC_POLYGONS_PATH, "rb") as f:
    nuc_polygons = pickle.load(f)

print(f"Loading: {CELL_AREAS_PATH}")
with open(CELL_AREAS_PATH, "rb") as f:
    cell_areas = pickle.load(f)

print(f"Loading: {NUC_AREAS_PATH}")
with open(NUC_AREAS_PATH, "rb") as f:
    nuc_areas = pickle.load(f)

print(f"Loading: {NUC_CENTROIDS_PATH}")
with open(NUC_CENTROIDS_PATH, "rb") as f:
    nuc_centroids = pickle.load(f)

print(f"cell_polygons: {len(cell_polygons):,}")
print(f"nuc_polygons : {len(nuc_polygons):,}")
print(f"cell_areas   : {len(cell_areas):,}")
print(f"nuc_areas    : {len(nuc_areas):,}")
print(f"nuc_centroids: {len(nuc_centroids):,}")

print(f"Loading: {CELL_DATA_PATH}")
cell_data = np.load(CELL_DATA_PATH, allow_pickle=True)
cell_ids_geom = cell_data["cell_ids"]
centroids = cell_data["centroids"]

print(f"cell_ids_geom shape: {cell_ids_geom.shape}")
print(f"centroids shape    : {centroids.shape}")

# ------------------------------------------------------------------------------
# 6. Rebuild observed examples and empirical baseline pools
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("4. Rebuilding validation examples and empirical baseline pools")
print("=" * 90)

P_DROP_MIN = 0.15
P_DROP_MAX = 0.40
MIN_MOL_FOR_TRAINING = 6

mol["status"] = mol["status"].astype(str)
mol_clean = mol[mol["status"] == "observed"].copy()

required_cols = [
    "cell_id",
    "gene_id",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "Assigned_Xenium_Cell_Type",
]
missing_cols = [c for c in required_cols if c not in mol_clean.columns]
if missing_cols:
    raise KeyError(f"mol is missing required columns: {missing_cols}")

print(f"Observed molecules used for examples/baselines: {len(mol_clean):,}")

# Cell coordinate lookup.
cell_xy_lookup = {}

if {"x_centroid", "y_centroid"}.issubset(set(denoised_adata.obs.columns)):
    print("Using denoised_adata.obs x_centroid/y_centroid for cell coordinates.")
    for cid, x, y in zip(
        denoised_adata.obs_names.astype(str),
        denoised_adata.obs["x_centroid"].values,
        denoised_adata.obs["y_centroid"].values,
    ):
        cell_xy_lookup[int(cid)] = np.array([float(x), float(y)], dtype=np.float32)
else:
    print("Using cell_ids_geom/centroids for cell coordinates.")
    for cid, xy in zip(cell_ids_geom, centroids):
        cell_xy_lookup[int(cid)] = np.asarray(xy, dtype=np.float32)

examples_by_pair = []

print("Building examples from observed molecules...")
for (cid, gid), group in mol_clean.groupby(["cell_id", "gene_id"], sort=False):
    n = len(group)
    if n < MIN_MOL_FOR_TRAINING:
        continue

    cid_int = int(cid)

    if cid_int not in step4_cell_to_row:
        continue
    if gid not in gene_to_col:
        continue
    if cid_int not in cell_xy_lookup:
        continue

    row = step4_cell_to_row[cid_int]
    gene_idx = gene_to_col[gid]
    ct_idx = int(cell_type_indices[row])

    c_area = float(cell_areas.get(cid_int, 100.0))
    n_area = float(nuc_areas.get(cid_int, 30.0))

    values = group[["r_norm", "theta", "z_rel", "p_nuclear"]].values.astype(np.float32)

    examples_by_pair.append({
        "cell_id": cid_int,
        "gene_id": str(gid),
        "gene_idx": gene_idx,
        "ct_idx": ct_idx,
        "cell_xy": cell_xy_lookup[cid_int],
        "cell_area": c_area,
        "nuc_area": n_area,
        "values": values,
    })

print(f"Eligible observed (cell, gene) pairs: {len(examples_by_pair):,}")

# Same deterministic split as 9E training.
all_cells = sorted({ex["cell_id"] for ex in examples_by_pair})
rng_split = np.random.default_rng(42)
rng_split.shuffle(all_cells)

n_val = max(1, int(0.10 * len(all_cells)))
val_cells = set(all_cells[:n_val])
train_cells = set(all_cells[n_val:])

training_examples = [ex for ex in examples_by_pair if ex["cell_id"] in train_cells]
validation_examples = [ex for ex in examples_by_pair if ex["cell_id"] in val_cells]

print(f"Train cells: {len(train_cells):,}")
print(f"Validation cells: {len(val_cells):,}")
print(f"Training examples: {len(training_examples):,}")
print(f"Validation examples: {len(validation_examples):,}")

# Baseline pools from training examples only.
print("Building empirical baseline pools from training examples only...")

gene_pool_lists = defaultdict(list)
ct_gene_pool_lists = defaultdict(list)
spatial_entry_lists = defaultdict(list)

for ex in training_examples:
    gene_idx = int(ex["gene_idx"])
    ct_idx = int(ex["ct_idx"])
    values = ex["values"].astype(np.float32)

    gene_pool_lists[gene_idx].append(values)
    ct_gene_pool_lists[(ct_idx, gene_idx)].append(values)

    spatial_entry_lists[(ct_idx, gene_idx)].append({
        "cell_id": int(ex["cell_id"]),
        "xy": ex["cell_xy"].astype(np.float32),
        "values": values,
    })

baseline_gene_pools = {
    gene_idx: np.concatenate(arrs, axis=0).astype(np.float32)
    for gene_idx, arrs in gene_pool_lists.items()
}

baseline_ct_gene_pools = {
    key: np.concatenate(arrs, axis=0).astype(np.float32)
    for key, arrs in ct_gene_pool_lists.items()
}

baseline_spatial_index = {}
for key, entries in spatial_entry_lists.items():
    cell_ids_arr = np.array([e["cell_id"] for e in entries], dtype=np.int64)
    xy_arr = np.stack([e["xy"] for e in entries], axis=0).astype(np.float32)
    values_list = [e["values"].astype(np.float32) for e in entries]

    baseline_spatial_index[key] = {
        "cell_ids": cell_ids_arr,
        "xy": xy_arr,
        "values": values_list,
    }

print(f"Gene empirical pools: {len(baseline_gene_pools):,}")
print(f"Cell-type gene empirical pools: {len(baseline_ct_gene_pools):,}")
print(f"Spatial-kNN empirical index groups: {len(baseline_spatial_index):,}")

# Free the huge filtered copy.
del mol_clean
gc.collect()

# ------------------------------------------------------------------------------
# 7. Define 9E model architecture
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("5. Defining 9E model architecture")
print("=" * 90)

class PointNetContextEncoder(nn.Module):
    def __init__(self, in_dim=5, hidden=192, out_dim=192):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, out_dim),
        )
        self.out_proj = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim),
            nn.SiLU(),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, points, mask):
        h = self.mlp(points)
        mask_exp = mask.unsqueeze(-1)

        h_masked = h.masked_fill(mask_exp == 0, -1e4)
        max_pool = h_masked.max(dim=1).values
        max_pool = torch.where(torch.isfinite(max_pool), max_pool, torch.zeros_like(max_pool))

        denom = mask_exp.sum(dim=1).clamp_min(1.0)
        mean_pool = (h * mask_exp).sum(dim=1) / denom

        return self.out_proj(torch.cat([max_pool, mean_pool], dim=-1))


class LocalizationDistributionModel(nn.Module):
    def __init__(self, n_genes, n_cell_types, ctx_dim=192, gene_dim=96,
                 ct_dim=32, geom_dim=9, geom_hidden=64, hidden=384):
        super().__init__()

        self.context_encoder = PointNetContextEncoder(in_dim=5, hidden=192, out_dim=ctx_dim)
        self.gene_embed = nn.Embedding(n_genes, gene_dim)
        self.ct_embed = nn.Embedding(n_cell_types, ct_dim)

        self.geom_mlp = nn.Sequential(
            nn.Linear(geom_dim, geom_hidden), nn.SiLU(),
            nn.Linear(geom_hidden, geom_hidden), nn.SiLU(),
        )

        fused = ctx_dim + gene_dim + ct_dim + geom_hidden

        self.backbone = nn.Sequential(
            nn.Linear(fused, hidden), nn.SiLU(), nn.Dropout(0.08),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Dropout(0.08),
            nn.Linear(hidden, hidden), nn.SiLU(),
        )

        self.r_head = nn.Linear(hidden, 2)
        self.z_head = nn.Linear(hidden, 2)
        self.pnuc_head = nn.Linear(hidden, 1)
        self.theta_head = nn.Linear(hidden, 2)

    def forward(self, context, context_mask, gene_idx, ct_idx, geom):
        h_ctx = self.context_encoder(context, context_mask)
        h_gene = self.gene_embed(gene_idx)
        h_ct = self.ct_embed(ct_idx)
        h_geom = self.geom_mlp(geom)

        h = self.backbone(torch.cat([h_ctx, h_gene, h_ct, h_geom], dim=-1))

        r_ab = F.softplus(self.r_head(h)) + 1.05
        z_ab = F.softplus(self.z_head(h)) + 1.05
        pnuc_logit = self.pnuc_head(h).squeeze(-1)
        theta_vec = self.theta_head(h)

        return {
            "r_alpha": r_ab[:, 0],
            "r_beta": r_ab[:, 1],
            "z_alpha": z_ab[:, 0],
            "z_beta": z_ab[:, 1],
            "pnuc_logit": pnuc_logit,
            "theta_vec": theta_vec,
        }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LocalizationDistributionModel(
    n_genes=len(shared_genes),
    n_cell_types=len(unique_cell_types),
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,} ({n_params * 4 / 1e6:.1f} MB)")
print(f"Device: {device}")

# ------------------------------------------------------------------------------
# 8. Load best 9E coordNN checkpoint
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("6. Loading best 9E coordNN checkpoint")
print("=" * 90)

print(f"Loading checkpoint: {BEST_COORD_MODEL_PATH}")
ckpt = torch.load(BEST_COORD_MODEL_PATH, map_location=device)

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded model_state_dict from full checkpoint payload.")
    if "best_coord_nn" in ckpt:
        print(f"Checkpoint best_coord_nn: {ckpt['best_coord_nn']}")
    if "selection_metric" in ckpt:
        print(f"Checkpoint selection_metric: {ckpt['selection_metric']}")
else:
    model.load_state_dict(ckpt)
    print("Loaded raw model state_dict.")

model.to(device)
model.eval()

print("\n" + "=" * 90)
print("9E RECOVERY COMPLETE")
print("=" * 90)
print("Ready for final large evaluation.")
print(f"Use validation_examples: {len(validation_examples):,}")
print(f"Use baseline_gene_pools: {len(baseline_gene_pools):,}")
print(f"Use baseline_ct_gene_pools: {len(baseline_ct_gene_pools):,}")
print(f"Use baseline_spatial_index: {len(baseline_spatial_index):,}")
print(f"Device for final eval: {device}")

In [ ]:
# ==============================================================================
# FINAL LARGE RECOVERY EVALUATION FOR 9E
# Run this after the 9E recovery cell.
# ==============================================================================

import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.distributions import Beta
from scipy.stats import wasserstein_distance
from scipy.spatial import cKDTree
from sklearn.neighbors import BallTree

print("=" * 90)
print("FINAL LARGE RECOVERY EVALUATION — 9E")
print("=" * 90)

# ------------------------------------------------------------------------------
# 0. Required variables from recovery cell
# ------------------------------------------------------------------------------

required_vars = [
    "model",
    "device",
    "validation_examples",
    "baseline_gene_pools",
    "baseline_ct_gene_pools",
    "baseline_spatial_index",
    "shared_genes",
    "unique_cell_types",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from recovery cell: {v}")

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
RUN_NAME = "attempt_9E_distribution_empirical_baselines"

BEST_COORD_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_best_coordNN_model.pt"
)

FINAL_EVAL_CSV_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_final_large_recovery_metrics.csv"
)

FINAL_EVAL_REPORT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_final_large_recovery_report.txt"
)

# Recommended:
#   10000 = safer/faster
#   20000 = stronger final estimate but slower
FINAL_RECOVERY_EVAL_MAX_ITEMS = 20000
FINAL_RECOVERY_EVAL_BATCH_SIZE = 256
FINAL_EVAL_SEED = 22345

print(f"Validation examples available: {len(validation_examples):,}")
print(f"Final eval max items       : {FINAL_RECOVERY_EVAL_MAX_ITEMS:,}")
print(f"Final eval batch size      : {FINAL_RECOVERY_EVAL_BATCH_SIZE}")
print(f"Device                     : {device}")

# ------------------------------------------------------------------------------
# 1. Load best coordNN checkpoint again for safety
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("1. Loading best coordNN checkpoint")
print("=" * 90)

if os.path.exists(BEST_COORD_MODEL_PATH):
    print(f"Loading: {BEST_COORD_MODEL_PATH}")
    ckpt = torch.load(BEST_COORD_MODEL_PATH, map_location=device)

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
        print("Loaded model_state_dict from checkpoint payload.")
        if "best_coord_nn" in ckpt:
            print(f"Checkpoint best_coord_nn: {ckpt['best_coord_nn']}")
        if "selection_metric" in ckpt:
            print(f"Selection metric: {ckpt['selection_metric']}")
    else:
        model.load_state_dict(ckpt)
        print("Loaded raw state_dict checkpoint.")
else:
    print("WARNING: best coordNN checkpoint not found.")
    print("Using currently loaded model from recovery cell.")

model.to(device)
model.eval()

# ------------------------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------------------------

def _features_np(vals):
    """
    Convert molecule values [r_norm, theta, z_rel, p_nuclear]
    into model context features:
      [r_norm, sin(theta), cos(theta), z_rel, p_nuclear]
    """
    r = vals[:, 0]
    theta = vals[:, 1]
    z = vals[:, 2]
    p_nuc = vals[:, 3]

    return np.stack(
        [r, np.sin(theta), np.cos(theta), z, p_nuc],
        axis=-1
    ).astype(np.float32)


def make_eval_item_from_example(ex, idx, max_context=96, max_target=24,
                                p_drop_min=0.15, p_drop_max=0.40,
                                seed=12345):
    """
    Recreates the same validation hide/recover format used by 9E.
    Important: geom has 9 features because 9E uses geom_dim=9.
    """
    vals = ex["values"]
    n = len(vals)

    rng = np.random.default_rng(seed + idx)

    p_drop = rng.uniform(p_drop_min, p_drop_max)

    k = int(rng.binomial(n, p_drop))
    k = max(1, min(n - 3, k, max_target))

    perm = rng.permutation(n)

    target_vals = vals[perm[:k]]
    context_vals = vals[perm[k:]]

    if len(context_vals) > max_context:
        ctx_idx = rng.choice(len(context_vals), size=max_context, replace=False)
        context_used = context_vals[ctx_idx]
    else:
        context_used = context_vals

    n_ctx = min(len(context_used), max_context)
    n_tgt = min(len(target_vals), max_target)

    ctx_padded = np.zeros((max_context, 5), dtype=np.float32)
    if n_ctx > 0:
        ctx_padded[:n_ctx] = _features_np(context_used[:n_ctx])

    ctx_mask = np.zeros(max_context, dtype=np.float32)
    ctx_mask[:n_ctx] = 1.0

    target_coords = np.zeros((max_target, 3), dtype=np.float32)
    target_pnuc = np.zeros(max_target, dtype=np.float32)
    target_mask = np.zeros(max_target, dtype=np.float32)

    target_coords[:n_tgt] = target_vals[:n_tgt, :3]
    target_pnuc[:n_tgt] = target_vals[:n_tgt, 3]
    target_mask[:n_tgt] = 1.0

    c_area = float(ex.get("cell_area", 100.0))
    n_area = float(ex.get("nuc_area", 30.0))

    # 9E context-summary features.
    if n_ctx > 0:
        ctx_features = ctx_padded[:n_ctx]

        ctx_r_mean = float(ctx_features[:, 0].mean())
        ctx_r_std = float(ctx_features[:, 0].std())

        ctx_z_mean = float(ctx_features[:, 3].mean())
        ctx_z_std = float(ctx_features[:, 3].std())

        ctx_pnuc_mean = float(ctx_features[:, 4].mean())
    else:
        ctx_r_mean = 0.5
        ctx_r_std = 0.0
        ctx_z_mean = 0.5
        ctx_z_std = 0.0
        ctx_pnuc_mean = 0.0

    geom = np.array([
        c_area / 500.0,
        n_area / 200.0,
        n_area / c_area if c_area > 0 else 0.3,
        n_ctx / 50.0,
        ctx_r_mean,
        ctx_r_std,
        ctx_z_mean,
        ctx_z_std,
        ctx_pnuc_mean,
    ], dtype=np.float32)

    return {
        "context": ctx_padded,
        "context_mask": ctx_mask,
        "target_coords": target_coords,
        "target_pnuc": target_pnuc,
        "target_mask": target_mask,
        "gene_idx": int(ex["gene_idx"]),
        "ct_idx": int(ex["ct_idx"]),
        "geom": geom,
        "ex": ex,
        "n_tgt": n_tgt,
    }


@torch.no_grad()
def sample_from_learned_distribution(outputs, target_mask, context, context_mask):
    """
    9E learned sampler:
      r_norm sampled from predicted Beta distribution
      z_rel sampled from predicted Beta distribution
      theta sampled around learned theta direction
    """
    dev = target_mask.device
    B, K = target_mask.shape

    r_dist = Beta(
        outputs["r_alpha"].clamp_min(1e-4),
        outputs["r_beta"].clamp_min(1e-4),
    )
    r = r_dist.sample((K,)).transpose(0, 1).to(dev)
    r = r.clamp(0.0, 1.0)

    z_dist = Beta(
        outputs["z_alpha"].clamp_min(1e-4),
        outputs["z_beta"].clamp_min(1e-4),
    )
    z = z_dist.sample((K,)).transpose(0, 1).to(dev)
    z = z.clamp(0.0, 1.0)

    theta_vec = F.normalize(outputs["theta_vec"], dim=-1)
    theta_mu = torch.atan2(theta_vec[:, 0], theta_vec[:, 1])

    theta = theta_mu.unsqueeze(1).expand(B, K).clone()

    theta_jitter = 0.20
    theta = theta + torch.randn((B, K), device=dev) * theta_jitter

    theta = ((theta + torch.pi) % (2 * torch.pi)) - torch.pi

    return torch.stack([r, theta, z], dim=-1)


def sample_gene_empirical(ex, n_tgt, rng):
    gene_idx = int(ex["gene_idx"])

    pool = baseline_gene_pools.get(gene_idx, None)

    if pool is None or len(pool) == 0:
        return fallback_sample_from_context_or_uniform(ex, n_tgt, rng)

    idx = rng.choice(len(pool), size=n_tgt, replace=True)
    return pool[idx, :3].astype(np.float32)


def sample_ct_gene_empirical(ex, n_tgt, rng):
    key = (int(ex["ct_idx"]), int(ex["gene_idx"]))

    pool = baseline_ct_gene_pools.get(key, None)

    if pool is None or len(pool) == 0:
        return sample_gene_empirical(ex, n_tgt, rng)

    idx = rng.choice(len(pool), size=n_tgt, replace=True)
    return pool[idx, :3].astype(np.float32)


def sample_spatial_knn_empirical(ex, n_tgt, rng, k_neighbors=80, min_pool=5):
    key = (int(ex["ct_idx"]), int(ex["gene_idx"]))

    info = baseline_spatial_index.get(key, None)

    if info is None or len(info["cell_ids"]) == 0:
        return sample_ct_gene_empirical(ex, n_tgt, rng)

    xy = info["xy"]
    values_list = info["values"]

    query_xy = np.asarray(ex["cell_xy"], dtype=np.float32).reshape(1, -1)

    k = min(k_neighbors, len(xy))

    if k < 1:
        return sample_ct_gene_empirical(ex, n_tgt, rng)

    tree = BallTree(xy)
    _, ind = tree.query(query_xy, k=k)

    candidate_vals = []
    for idx in ind[0]:
        vals = values_list[int(idx)]
        if vals is not None and len(vals) > 0:
            candidate_vals.append(vals)

    if len(candidate_vals) < min_pool:
        return sample_ct_gene_empirical(ex, n_tgt, rng)

    pool = np.concatenate(candidate_vals, axis=0)

    if len(pool) == 0:
        return sample_ct_gene_empirical(ex, n_tgt, rng)

    idx = rng.choice(len(pool), size=n_tgt, replace=True)
    return pool[idx, :3].astype(np.float32)


def fallback_sample_from_context_or_uniform(ex, n_tgt, rng):
    vals = ex.get("values", None)

    if vals is not None and len(vals) > 0:
        idx = rng.choice(len(vals), size=n_tgt, replace=True)
        return vals[idx, :3].astype(np.float32)

    r = rng.uniform(0, 1, size=n_tgt)
    theta = rng.uniform(-np.pi, np.pi, size=n_tgt)
    z = rng.uniform(0, 1, size=n_tgt)

    return np.stack([r, theta, z], axis=-1).astype(np.float32)


def circular_angle_distance(a, b):
    """
    Absolute circular distance between two angle arrays.
    Output is in radians, between 0 and pi.
    """
    d = np.abs(a - b)
    return np.minimum(d, 2 * np.pi - d)


def coord_features(coords):
    """
    Convert [r, theta, z] into a Euclidean-friendly representation:
      [r*cos(theta), r*sin(theta), z]
    """
    r = coords[:, 0]
    theta = coords[:, 1]
    z = coords[:, 2]

    return np.stack(
        [r * np.cos(theta), r * np.sin(theta), z],
        axis=-1
    ).astype(np.float32)


def coord_nn_distance(pred, true):
    """
    Mean nearest-neighbor distance from predicted molecules to true molecules
    in transformed coordinate space.
    """
    if len(pred) == 0 or len(true) == 0:
        return np.nan

    pred_xyz = coord_features(pred)
    true_xyz = coord_features(true)

    tree = cKDTree(true_xyz)
    d, _ = tree.query(pred_xyz, k=1)

    return float(np.mean(d))


def radial_hist_l1(pred_r, true_r, bins=20):
    hist_pred, _ = np.histogram(pred_r, bins=bins, range=(0, 1), density=True)
    hist_true, _ = np.histogram(true_r, bins=bins, range=(0, 1), density=True)

    hist_pred = hist_pred / (hist_pred.sum() + 1e-8)
    hist_true = hist_true / (hist_true.sum() + 1e-8)

    return float(np.abs(hist_pred - hist_true).sum())


def compute_recovery_metrics_np(preds, targets):
    """
    Computes recovery metrics across a list of predicted/true coordinate arrays.
    Lower is better for all metrics.
    """
    coord_nn_vals = []
    r_w_vals = []
    z_w_vals = []
    theta_vals = []
    radial_l1_vals = []

    for pred, true in zip(preds, targets):
        if pred is None or true is None:
            continue
        if len(pred) == 0 or len(true) == 0:
            continue

        pred = np.asarray(pred, dtype=np.float32)
        true = np.asarray(true, dtype=np.float32)

        coord_nn_vals.append(coord_nn_distance(pred, true))

        r_w_vals.append(float(wasserstein_distance(true[:, 0], pred[:, 0])))
        z_w_vals.append(float(wasserstein_distance(true[:, 2], pred[:, 2])))

        # Compare each predicted theta to nearest true theta.
        theta_d_mat = circular_angle_distance(
            pred[:, 1][:, None],
            true[:, 1][None, :]
        )
        theta_vals.append(float(theta_d_mat.min(axis=1).mean()))

        radial_l1_vals.append(radial_hist_l1(pred[:, 0], true[:, 0]))

    def safe_mean(x):
        x = [v for v in x if np.isfinite(v)]
        return float(np.mean(x)) if len(x) else np.nan

    return {
        "coord_nn": safe_mean(coord_nn_vals),
        "r_wasserstein": safe_mean(r_w_vals),
        "z_wasserstein": safe_mean(z_w_vals),
        "theta_distance": safe_mean(theta_vals),
        "radial_hist_l1": safe_mean(radial_l1_vals),
    }


@torch.no_grad()
def evaluate_heldout_recovery_empirical_large(model, validation_examples, device,
                                              max_items=10000,
                                              batch_size=256,
                                              seed=12345):
    """
    Large final evaluation of learned model against 3 empirical baselines.
    """
    model.eval()

    n_eval = min(max_items, len(validation_examples))

    learned_preds = []
    gene_preds = []
    ct_gene_preds = []
    spatial_knn_preds = []
    targets = []

    rng_eval = np.random.default_rng(seed)

    t0 = time.time()

    for start in range(0, n_eval, batch_size):
        end = min(start + batch_size, n_eval)

        items = [
            make_eval_item_from_example(
                validation_examples[i],
                idx=i,
                max_context=96,
                max_target=24,
                p_drop_min=0.15,
                p_drop_max=0.40,
                seed=seed,
            )
            for i in range(start, end)
        ]

        ctx = torch.tensor(
            np.stack([it["context"] for it in items]),
            dtype=torch.float32,
            device=device
        )
        ctx_mask = torch.tensor(
            np.stack([it["context_mask"] for it in items]),
            dtype=torch.float32,
            device=device
        )
        target_coords_t = torch.tensor(
            np.stack([it["target_coords"] for it in items]),
            dtype=torch.float32,
            device=device
        )
        target_mask_t = torch.tensor(
            np.stack([it["target_mask"] for it in items]),
            dtype=torch.float32,
            device=device
        )

        gene_idx_t = torch.tensor(
            [it["gene_idx"] for it in items],
            dtype=torch.long,
            device=device
        )
        ct_idx_t = torch.tensor(
            [it["ct_idx"] for it in items],
            dtype=torch.long,
            device=device
        )
        geom_t = torch.tensor(
            np.stack([it["geom"] for it in items]),
            dtype=torch.float32,
            device=device
        )

        outputs = model(ctx, ctx_mask, gene_idx_t, ct_idx_t, geom_t)
        learned_sample_t = sample_from_learned_distribution(
            outputs,
            target_mask_t,
            ctx,
            ctx_mask
        )

        learned_sample = learned_sample_t.detach().cpu().numpy()
        target_coords_np = target_coords_t.detach().cpu().numpy()
        target_mask_np = target_mask_t.detach().cpu().numpy()

        for local_i, it in enumerate(items):
            m = target_mask_np[local_i] > 0
            n_tgt = int(m.sum())

            if n_tgt <= 0:
                continue

            true_coords = target_coords_np[local_i, m, :]
            learned_coords = learned_sample[local_i, m, :]

            ex = it["ex"]

            gene_coords = sample_gene_empirical(ex, n_tgt, rng_eval)
            ct_gene_coords = sample_ct_gene_empirical(ex, n_tgt, rng_eval)
            spatial_coords = sample_spatial_knn_empirical(ex, n_tgt, rng_eval)

            targets.append(true_coords)
            learned_preds.append(learned_coords)
            gene_preds.append(gene_coords)
            ct_gene_preds.append(ct_gene_coords)
            spatial_knn_preds.append(spatial_coords)

        if (end % 2048 == 0) or (end == n_eval):
            elapsed = (time.time() - t0) / 60
            print(f"  evaluated {end:,}/{n_eval:,} examples "
                  f"({elapsed:.1f} min elapsed)")

    metrics = {}

    all_methods = {
        "learned": learned_preds,
        "gene_emp": gene_preds,
        "ct_gene_emp": ct_gene_preds,
        "spatial_knn_emp": spatial_knn_preds,
    }

    for name, preds in all_methods.items():
        m = compute_recovery_metrics_np(preds, targets)
        for k, v in m.items():
            metrics[f"{name}_{k}"] = v

    learned_coord = metrics.get("learned_coord_nn", np.nan)

    for base in ["gene_emp", "ct_gene_emp", "spatial_knn_emp"]:
        base_coord = metrics.get(f"{base}_coord_nn", np.nan)
        metrics[f"improvement_vs_{base}_coord_nn"] = float(
            (base_coord - learned_coord) / (abs(base_coord) + 1e-8)
        )

    metrics["n_eval_examples"] = int(n_eval)
    metrics["seed"] = int(seed)

    return metrics


# ------------------------------------------------------------------------------
# 3. Run final large evaluation
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("2. Running final large evaluation")
print("=" * 90)

t0 = time.time()

final_metrics = evaluate_heldout_recovery_empirical_large(
    model=model,
    validation_examples=validation_examples,
    device=device,
    max_items=FINAL_RECOVERY_EVAL_MAX_ITEMS,
    batch_size=FINAL_RECOVERY_EVAL_BATCH_SIZE,
    seed=FINAL_EVAL_SEED,
)

elapsed = time.time() - t0

print("\n" + "=" * 90)
print("FINAL LARGE EVALUATION COMPLETE")
print("=" * 90)
print(f"Elapsed: {elapsed / 60:.1f} min")

# ------------------------------------------------------------------------------
# 4. Display final metrics
# ------------------------------------------------------------------------------

rows = []

method_map = {
    "learned": "Learned 9E",
    "gene_emp": "Gene empirical",
    "ct_gene_emp": "Cell-type gene empirical",
    "spatial_knn_emp": "Spatial-kNN empirical",
}

for key, label in method_map.items():
    rows.append({
        "method": label,
        "coord_nn": final_metrics.get(f"{key}_coord_nn", np.nan),
        "r_wasserstein": final_metrics.get(f"{key}_r_wasserstein", np.nan),
        "z_wasserstein": final_metrics.get(f"{key}_z_wasserstein", np.nan),
        "theta_distance": final_metrics.get(f"{key}_theta_distance", np.nan),
        "radial_hist_l1": final_metrics.get(f"{key}_radial_hist_l1", np.nan),
    })

final_df = pd.DataFrame(rows)

print("\nFinal large recovery metrics:")
display(final_df)

print("\nFinal learned improvement over baselines using coordNN:")
for base in ["gene_emp", "ct_gene_emp", "spatial_knn_emp"]:
    imp = final_metrics.get(f"improvement_vs_{base}_coord_nn", np.nan)
    print(f"  learned vs {base}: {100 * imp:.2f}%")

# ------------------------------------------------------------------------------
# 5. Save final results
# ------------------------------------------------------------------------------

final_df.to_csv(FINAL_EVAL_CSV_PATH, index=False)

with open(FINAL_EVAL_REPORT_PATH, "w") as f:
    f.write("FINAL LARGE RECOVERY EVALUATION — 9E\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"RUN_NAME: {RUN_NAME}\n")
    f.write(f"n_eval_examples: {final_metrics['n_eval_examples']}\n")
    f.write(f"seed: {final_metrics['seed']}\n")
    f.write(f"elapsed_min: {elapsed / 60:.2f}\n\n")

    f.write("Metrics:\n")
    f.write(final_df.to_string(index=False))
    f.write("\n\n")

    f.write("CoordNN improvements:\n")
    for base in ["gene_emp", "ct_gene_emp", "spatial_knn_emp"]:
        imp = final_metrics.get(f"improvement_vs_{base}_coord_nn", np.nan)
        f.write(f"  learned vs {base}: {100 * imp:.2f}%\n")

print("\nSaved final large evaluation files:")
print(f"  CSV   : {FINAL_EVAL_CSV_PATH}")
print(f"  Report: {FINAL_EVAL_REPORT_PATH}")

======EVALUATION ENDS======

In [ ]:
# ==============================================================================
# CELL S5-9E-INFER-1 — Prepare 9E imputation targets + geometry/cache
# Run this after final large evaluation.
# ==============================================================================

import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

from shapely.geometry import Point, LineString, MultiPoint, GeometryCollection
from shapely.prepared import prep as shapely_prep

print("=" * 90)
print("SUBSTEP 5F-9E: Prepare 9E imputation targets and geometry")
print("=" * 90)

t0 = time.time()

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "mol",
    "model",
    "device",
    "X_denoised",
    "X_raw_counts",
    "was_corrected",
    "shared_genes",
    "cell_ids_step4",
    "gene_to_col",
    "step4_cell_to_row",
    "cell_type_indices",
    "unique_cell_types",
    "denoised_adata",
    "cell_polygons",
    "nuc_polygons",
    "cell_areas",
    "nuc_areas",
    "nuc_centroids",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable: {v}")

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RUN_NAME = "attempt_9E_distribution_empirical_baselines"

MAX_CONTEXT_9E = 96
INFER_BATCH_SIZE = 256

print(f"RUN_NAME: {RUN_NAME}")
print(f"MAX_CONTEXT_9E: {MAX_CONTEXT_9E}")
print(f"INFER_BATCH_SIZE: {INFER_BATCH_SIZE}")
print(f"Device: {device}")

# ------------------------------------------------------------------------------
# 1. Cell type labels
# ------------------------------------------------------------------------------

if "cell_type" in denoised_adata.obs.columns:
    cell_type_labels = denoised_adata.obs["cell_type"].astype(str).values
elif "Assigned_Xenium_Cell_Type" in denoised_adata.obs.columns:
    cell_type_labels = denoised_adata.obs["Assigned_Xenium_Cell_Type"].astype(str).values
else:
    raise KeyError("Could not find cell type column in denoised_adata.obs")

print(f"Cell type labels loaded: {len(cell_type_labels):,}")

# ------------------------------------------------------------------------------
# 2. Identify imputation targets from corrected Step 4 counts
# ------------------------------------------------------------------------------

print("\nIdentifying imputation targets from Step 4 denoised counts...")

X_den_round = np.rint(X_denoised).astype(np.int32)
X_raw_int = np.rint(X_raw_counts).astype(np.int32)

delta_counts = X_den_round - X_raw_int
delta_counts[delta_counts < 0] = 0

# Only impute pairs Step 4 actually corrected.
if was_corrected is not None:
    if was_corrected.shape == delta_counts.shape:
        delta_counts = delta_counts * was_corrected.astype(bool)
    else:
        print("WARNING: was_corrected shape does not match delta_counts; ignoring was_corrected filter.")

rows, cols = np.where(delta_counts > 0)

imputation_targets = []
total_to_generate = 0

for row, col in zip(rows, cols):
    n_imp = int(delta_counts[row, col])
    if n_imp <= 0:
        continue

    cid = int(cell_ids_step4[row])
    gid = str(shared_genes[col])

    # Require valid geometry for coordinate conversion.
    if cid not in cell_polygons:
        continue
    if cid not in nuc_centroids:
        continue

    target = {
        "row": int(row),
        "col": int(col),
        "cell_id": cid,
        "gene_id": gid,
        "gene_idx": int(col),
        "ct_idx": int(cell_type_indices[row]),
        "ct_label": str(cell_type_labels[row]),
        "n_impute": n_imp,
    }

    imputation_targets.append(target)
    total_to_generate += n_imp

print(f"Pairs needing imputation: {len(imputation_targets):,}")
print(f"Total molecules to generate: {total_to_generate:,}")

if len(imputation_targets) == 0:
    raise RuntimeError("No imputation targets found. Check X_denoised, X_raw_counts, and was_corrected.")

imputation_targets_df = pd.DataFrame(imputation_targets)
display(imputation_targets_df.head())

# Save target summary for traceability.
targets_path = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_imputation_targets.csv")
imputation_targets_df.to_csv(targets_path, index=False)
print(f"Saved target list: {targets_path}")

# ------------------------------------------------------------------------------
# 3. Build observed molecule context lookup for target pairs only
# ------------------------------------------------------------------------------

print("\nBuilding observed molecule context lookup for target pairs only...")

target_pairs = imputation_targets_df[["cell_id", "gene_id"]].drop_duplicates()
target_pairs["cell_id"] = target_pairs["cell_id"].astype(int)
target_pairs["gene_id"] = target_pairs["gene_id"].astype(str)

mol_obs_cols = ["cell_id", "gene_id", "r_norm", "theta", "z_rel", "p_nuclear", "status"]
mol_obs = mol[mol["status"].astype(str) == "observed"][mol_obs_cols].copy()
mol_obs["cell_id"] = mol_obs["cell_id"].astype(int)
mol_obs["gene_id"] = mol_obs["gene_id"].astype(str)

# Inner merge keeps only observed molecules for target cell-gene pairs.
mol_target_obs = mol_obs.merge(
    target_pairs,
    on=["cell_id", "gene_id"],
    how="inner",
)

print(f"Observed molecules in target pairs: {len(mol_target_obs):,}")

mol_context_lookup = {}

for (cid, gid), group in tqdm(
    mol_target_obs.groupby(["cell_id", "gene_id"], sort=False),
    desc="Building context lookup",
    mininterval=5,
):
    vals = group[["r_norm", "theta", "z_rel", "p_nuclear"]].values.astype(np.float32)
    mol_context_lookup[(int(cid), str(gid))] = vals

print(f"Context lookup entries: {len(mol_context_lookup):,}")

del mol_obs, mol_target_obs, target_pairs
gc.collect()

# ------------------------------------------------------------------------------
# 4. Build cell z-range lookup
# ------------------------------------------------------------------------------

print("\nBuilding cell z-range lookup...")

mol_active = mol[mol["status"].astype(str) != "pruned"][["cell_id", "z"]].copy()
mol_active["cell_id"] = mol_active["cell_id"].astype(int)

cell_z_stats = mol_active.groupby("cell_id")["z"].agg(["min", "max"]).to_dict("index")

print(f"Cell z-range entries: {len(cell_z_stats):,}")

del mol_active
gc.collect()

# ------------------------------------------------------------------------------
# 5. Geometry helper functions
# ------------------------------------------------------------------------------

def _centroid_to_xy(c):
    """
    Convert stored nucleus centroid to a 2D numpy array.
    Handles numpy arrays, lists, tuples, and shapely Points.
    """
    if hasattr(c, "x") and hasattr(c, "y"):
        return np.array([float(c.x), float(c.y)], dtype=np.float32)

    arr = np.asarray(c, dtype=np.float32).reshape(-1)
    if len(arr) < 2:
        raise ValueError(f"Invalid centroid object: {c}")

    return arr[:2].astype(np.float32)


def _collect_points_from_intersection(geom):
    """
    Extract candidate points from a shapely intersection geometry.
    """
    pts = []

    if geom.is_empty:
        return pts

    gtype = geom.geom_type

    if gtype == "Point":
        pts.append(geom)

    elif gtype == "MultiPoint":
        pts.extend(list(geom.geoms))

    elif gtype == "LineString":
        coords = list(geom.coords)
        if len(coords) > 0:
            pts.append(Point(coords[0]))
            pts.append(Point(coords[-1]))

    elif gtype == "MultiLineString":
        for line in geom.geoms:
            coords = list(line.coords)
            if len(coords) > 0:
                pts.append(Point(coords[0]))
                pts.append(Point(coords[-1]))

    elif gtype == "GeometryCollection":
        for sub in geom.geoms:
            pts.extend(_collect_points_from_intersection(sub))

    return pts


def _nearest_intersection_point(intersection, nc):
    """
    Pick nearest boundary intersection point along a ray.
    """
    pts = _collect_points_from_intersection(intersection)

    if not pts:
        return None

    dists = [np.hypot(float(p.x) - nc[0], float(p.y) - nc[1]) for p in pts]
    best = pts[int(np.argmin(dists))]

    return np.array([float(best.x), float(best.y)], dtype=np.float32)


# ------------------------------------------------------------------------------
# 6. Precompute ray-to-cell-boundary edge lookup
# ------------------------------------------------------------------------------

print("\nPrecomputing geometry for cells needing imputation...")

GEOM_CACHE_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_geometry_cache.pkl")

N_ANGLES = 72
precomputed_angles = np.linspace(-np.pi, np.pi, N_ANGLES, endpoint=False).astype(np.float32)

cells_needing_imputation = sorted(set(int(t["cell_id"]) for t in imputation_targets))

# If a previous failed run created a broken/partial cache, remove it.
if os.path.exists(GEOM_CACHE_PATH):
    try:
        with open(GEOM_CACHE_PATH, "rb") as f:
            test_cache = pickle.load(f)

        if "cell_edge_lookup" not in test_cache:
            print("Existing geometry cache is incomplete. Removing it.")
            os.remove(GEOM_CACHE_PATH)
        else:
            del test_cache

    except Exception as e:
        print(f"Existing geometry cache could not be loaded: {e}")
        print("Removing broken geometry cache.")
        os.remove(GEOM_CACHE_PATH)

if os.path.exists(GEOM_CACHE_PATH):
    print(f"Loading geometry cache: {GEOM_CACHE_PATH}")

    with open(GEOM_CACHE_PATH, "rb") as f:
        geom_cache = pickle.load(f)

    cell_edge_lookup = geom_cache["cell_edge_lookup"]
    N_ANGLES = int(geom_cache["N_ANGLES"])
    precomputed_angles = geom_cache["precomputed_angles"].astype(np.float32)

    print(f"Loaded cell_edge_lookup: {len(cell_edge_lookup):,}")
    print(f"N_ANGLES: {N_ANGLES}")

else:
    cell_edge_lookup = {}

    for cid in tqdm(cells_needing_imputation, desc="Precomputing cell geometry", mininterval=5):
        if cid not in cell_polygons:
            continue
        if cid not in nuc_centroids:
            continue

        nc = _centroid_to_xy(nuc_centroids[cid])
        cell_poly = cell_polygons[cid]
        boundary = cell_poly.boundary

        edge_points = np.zeros((N_ANGLES, 2), dtype=np.float32)

        for ai, angle in enumerate(precomputed_angles):
            ray_end_x = nc[0] + 1000.0 * np.cos(angle)
            ray_end_y = nc[1] + 1000.0 * np.sin(angle)

            ray = LineString([
                (float(nc[0]), float(nc[1])),
                (float(ray_end_x), float(ray_end_y)),
            ])

            intersection = ray.intersection(boundary)
            p = _nearest_intersection_point(intersection, nc)

            if p is None:
                # Fallback if boundary intersection fails.
                p = np.array([
                    nc[0] + 10.0 * np.cos(angle),
                    nc[1] + 10.0 * np.sin(angle),
                ], dtype=np.float32)

            edge_points[ai] = p

        cell_edge_lookup[int(cid)] = edge_points

    # IMPORTANT:
    # Only save picklable objects.
    # Do NOT save Shapely prepared geometries.
    geom_cache = {
        "cell_edge_lookup": cell_edge_lookup,
        "N_ANGLES": N_ANGLES,
        "precomputed_angles": precomputed_angles,
    }

    with open(GEOM_CACHE_PATH, "wb") as f:
        pickle.dump(geom_cache, f)

    print(f"Saved geometry cache: {GEOM_CACHE_PATH}")
    print(f"cell_edge_lookup: {len(cell_edge_lookup):,}")

# ------------------------------------------------------------------------------
# 6b. Build prepared nucleus polygons in memory only
# ------------------------------------------------------------------------------

print("\nBuilding prepared nucleus polygons in memory only...")

cell_nuc_prep = {}

for cid in tqdm(cells_needing_imputation, desc="Preparing nucleus polygons", mininterval=5):
    cid = int(cid)

    if cid in nuc_polygons:
        try:
            cell_nuc_prep[cid] = shapely_prep(nuc_polygons[cid])
        except Exception:
            pass

print(f"In-memory prepared nucleus polygons: {len(cell_nuc_prep):,}")
print("Note: prepared geometries are not saved because Shapely cannot pickle them.")


# ------------------------------------------------------------------------------
# 7. Context feature and absolute coordinate conversion
# ------------------------------------------------------------------------------

def context_features(vals):
    """
    Convert observed molecule values [r_norm, theta, z_rel, p_nuclear]
    into model context features [r_norm, sin(theta), cos(theta), z_rel, p_nuclear].
    """
    r = vals[:, 0]
    th = vals[:, 1]
    z = vals[:, 2]
    pn = vals[:, 3]

    return np.stack(
        [r, np.sin(th), np.cos(th), z, pn],
        axis=-1
    ).astype(np.float32)


def convert_to_absolute_fast(r_norms, thetas, z_rels, cid):
    """
    Convert normalized 9E molecule coordinates [r_norm, theta, z_rel]
    back to absolute Xenium-like x/y/z coordinates.
    """
    cid = int(cid)

    nc = _centroid_to_xy(nuc_centroids[cid])
    edge_pts = cell_edge_lookup[cid]

    zr = cell_z_stats.get(cid, {"min": 0.0, "max": 15.0})
    z_min_c = float(zr["min"])
    z_max_c = float(zr["max"])
    z_range_c = max(z_max_c - z_min_c, 0.1)

    r_norms = np.asarray(r_norms, dtype=np.float32)
    thetas = np.asarray(thetas, dtype=np.float32)
    z_rels = np.asarray(z_rels, dtype=np.float32)

    n = len(r_norms)

    abs_x = np.zeros(n, dtype=np.float32)
    abs_y = np.zeros(n, dtype=np.float32)
    abs_z = np.zeros(n, dtype=np.float32)
    p_nuc_geom = np.zeros(n, dtype=np.float32)

    angle_step = 2.0 * np.pi / N_ANGLES
    first_angle = float(precomputed_angles[0])

    for j in range(n):
        theta_w = ((float(thetas[j]) + np.pi) % (2.0 * np.pi)) - np.pi

        frac_pos = (theta_w - first_angle) / angle_step
        frac_pos = frac_pos % N_ANGLES

        idx_lo = int(np.floor(frac_pos)) % N_ANGLES
        idx_hi = (idx_lo + 1) % N_ANGLES
        frac = frac_pos - np.floor(frac_pos)

        edge_x = edge_pts[idx_lo, 0] * (1.0 - frac) + edge_pts[idx_hi, 0] * frac
        edge_y = edge_pts[idx_lo, 1] * (1.0 - frac) + edge_pts[idx_hi, 1] * frac

        abs_x[j] = nc[0] + r_norms[j] * (edge_x - nc[0])
        abs_y[j] = nc[1] + r_norms[j] * (edge_y - nc[1])
        abs_z[j] = z_min_c + z_rels[j] * z_range_c

        if cid in cell_nuc_prep:
            try:
                p_nuc_geom[j] = 1.0 if cell_nuc_prep[cid].contains(
                    Point(float(abs_x[j]), float(abs_y[j]))
                ) else 0.0
            except Exception:
                p_nuc_geom[j] = 0.0

    return abs_x, abs_y, abs_z, p_nuc_geom


elapsed = time.time() - t0

print("\n" + "=" * 90)
print("9E INFERENCE SETUP COMPLETE")
print("=" * 90)
print(f"Pairs needing imputation: {len(imputation_targets):,}")
print(f"Total molecules to generate: {total_to_generate:,}")
print(f"Context lookup entries: {len(mol_context_lookup):,}")
print(f"Cells with geometry cache: {len(cell_edge_lookup):,}")
print(f"Setup time: {elapsed / 60:.1f} min")

In [ ]:
# ==============================================================================
# CELL S5-9E-INFER-2 — Generate 9E imputed molecule records
# Corrected version
# GPU strongly recommended.
# ==============================================================================

import os
import gc
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.distributions import Beta
from tqdm import tqdm

print("=" * 90)
print("SUBSTEP 5G-9E: Generate imputed molecule records")
print("=" * 90)

t0 = time.time()

# ------------------------------------------------------------------------------
# 0. Required variable checks
# ------------------------------------------------------------------------------

required_vars = [
    "model",
    "device",
    "imputation_targets",
    "imputation_targets_df",
    "total_to_generate",
    "mol_context_lookup",
    "convert_to_absolute_fast",
    "cell_areas",
    "nuc_areas",
    "cell_type_labels",
    "cell_edge_lookup",
    "cell_nuc_prep",
    "nuc_centroids",
    "cell_z_stats",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from Infer-1: {v}")

# ------------------------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------------------------

RUN_NAME = "attempt_9E_distribution_empirical_baselines"

MAX_CONTEXT_9E = 96
INFER_BATCH_SIZE = 256
THETA_JITTER = 0.20

# For reproducible stochastic sampling.
SEED_INFER = 12345
np.random.seed(SEED_INFER)
torch.manual_seed(SEED_INFER)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_INFER)

model.to(device)
model.eval()

n_targets = len(imputation_targets)

print(f"RUN_NAME: {RUN_NAME}")
print(f"Targets / cell-gene pairs       : {n_targets:,}")
print(f"Expected imputed molecules      : {total_to_generate:,}")
print(f"Context lookup entries          : {len(mol_context_lookup):,}")
print(f"Cells with edge geometry cache  : {len(cell_edge_lookup):,}")
print(f"Cells with prepared nuclei      : {len(cell_nuc_prep):,}")
print(f"Device                          : {device}")
print(f"MAX_CONTEXT_9E                  : {MAX_CONTEXT_9E}")
print(f"INFER_BATCH_SIZE                : {INFER_BATCH_SIZE}")
print(f"THETA_JITTER                    : {THETA_JITTER}")
print(f"SEED_INFER                      : {SEED_INFER}")

if len(imputation_targets) == 0:
    raise RuntimeError("imputation_targets is empty. Run/fix Infer-1 first.")

if total_to_generate <= 0:
    raise RuntimeError("total_to_generate <= 0. Nothing to impute.")

# ------------------------------------------------------------------------------
# 2. Helper: context feature conversion
# ------------------------------------------------------------------------------

def make_context_features_9e(vals):
    """
    Convert observed molecule values [r_norm, theta, z_rel, p_nuclear]
    into 9E model context features:
      [r_norm, sin(theta), cos(theta), z_rel, p_nuclear]

    Important:
      This function name is intentionally not 'context_features' to avoid being
      overwritten by local variables inside the notebook loop.
    """
    vals = np.asarray(vals, dtype=np.float32)

    if vals.ndim != 2 or vals.shape[1] < 4:
        raise ValueError(f"Expected vals shape [N, 4], got {vals.shape}")

    r = vals[:, 0]
    th = vals[:, 1]
    z = vals[:, 2]
    pn = vals[:, 3]

    return np.stack(
        [r, np.sin(th), np.cos(th), z, pn],
        axis=-1
    ).astype(np.float32)

# ------------------------------------------------------------------------------
# 3. Main inference loop
# ------------------------------------------------------------------------------

imputed_records = []
n_generated_so_far = 0

n_batches = (n_targets + INFER_BATCH_SIZE - 1) // INFER_BATCH_SIZE

print("\nStarting 9E imputation generation...")

for batch_start in tqdm(
    range(0, n_targets, INFER_BATCH_SIZE),
    desc="Generating 9E imputed molecules",
    total=n_batches,
    mininterval=10,
):
    batch_targets = imputation_targets[batch_start:batch_start + INFER_BATCH_SIZE]
    B = len(batch_targets)

    # Fixed-size model inputs.
    ctx_batch = np.zeros((B, MAX_CONTEXT_9E, 5), dtype=np.float32)
    mask_batch = np.zeros((B, MAX_CONTEXT_9E), dtype=np.float32)

    gene_batch = np.zeros(B, dtype=np.int64)
    ct_batch = np.zeros(B, dtype=np.int64)

    # 9E uses geom_dim = 9.
    geom_batch = np.zeros((B, 9), dtype=np.float32)

    n_gen_list = []

    # --------------------------------------------------------------------------
    # Build batch tensors from target cell-gene pairs
    # --------------------------------------------------------------------------
    for b, target in enumerate(batch_targets):
        cid = int(target["cell_id"])
        gid = str(target["gene_id"])
        n_imp = int(target["n_impute"])

        if n_imp <= 0:
            n_gen_list.append(0)
            continue

        vals = mol_context_lookup.get((cid, gid), None)

        if vals is not None and len(vals) > 0:
            n_ctx = min(len(vals), MAX_CONTEXT_9E)

            # Deterministic context selection for reproducibility.
            # Use first n_ctx observed molecules for this target pair.
            vals_used = vals[:n_ctx]

            ctx_batch[b, :n_ctx] = make_context_features_9e(vals_used)
            mask_batch[b, :n_ctx] = 1.0
        else:
            # No same-gene observed context in this cell.
            # Model will rely on gene embedding, cell type, geometry, and defaults.
            n_ctx = 0

        gene_batch[b] = int(target["gene_idx"])
        ct_batch[b] = int(target["ct_idx"])

        c_area = float(cell_areas.get(cid, 100.0))
        n_area = float(nuc_areas.get(cid, 30.0))

        # ----------------------------------------------------------------------
        # 9E context-summary geometry features
        # Must match training/evaluation geom_dim=9:
        #   0: cell_area / 500
        #   1: nuc_area / 200
        #   2: nuc_area / cell_area
        #   3: n_context / 50
        #   4: context_r_mean
        #   5: context_r_std
        #   6: context_z_mean
        #   7: context_z_std
        #   8: context_pnuc_mean
        # ----------------------------------------------------------------------
        if n_ctx > 0:
            ctx_summary_features = ctx_batch[b, :n_ctx]

            ctx_r_mean = float(ctx_summary_features[:, 0].mean())
            ctx_r_std = float(ctx_summary_features[:, 0].std())

            ctx_z_mean = float(ctx_summary_features[:, 3].mean())
            ctx_z_std = float(ctx_summary_features[:, 3].std())

            ctx_pnuc_mean = float(ctx_summary_features[:, 4].mean())
        else:
            ctx_r_mean = 0.5
            ctx_r_std = 0.0
            ctx_z_mean = 0.5
            ctx_z_std = 0.0
            ctx_pnuc_mean = 0.0

        geom_batch[b] = np.array([
            c_area / 500.0,
            n_area / 200.0,
            n_area / c_area if c_area > 0 else 0.3,
            n_ctx / 50.0,
            ctx_r_mean,
            ctx_r_std,
            ctx_z_mean,
            ctx_z_std,
            ctx_pnuc_mean,
        ], dtype=np.float32)

        n_gen_list.append(n_imp)

    K = int(max(n_gen_list))

    if K <= 0:
        continue

    # --------------------------------------------------------------------------
    # Move batch to GPU/CPU
    # --------------------------------------------------------------------------
    ctx_t = torch.tensor(ctx_batch, dtype=torch.float32, device=device)
    mask_t = torch.tensor(mask_batch, dtype=torch.float32, device=device)
    gene_t = torch.tensor(gene_batch, dtype=torch.long, device=device)
    ct_t = torch.tensor(ct_batch, dtype=torch.long, device=device)
    geom_t = torch.tensor(geom_batch, dtype=torch.float32, device=device)

    # --------------------------------------------------------------------------
    # Model inference + sampling
    # --------------------------------------------------------------------------
    with torch.no_grad():
        outputs = model(ctx_t, mask_t, gene_t, ct_t, geom_t)

        # Sample r_norm from predicted Beta distribution.
        r_alpha = outputs["r_alpha"].clamp_min(1e-4).unsqueeze(1).expand(B, K)
        r_beta = outputs["r_beta"].clamp_min(1e-4).unsqueeze(1).expand(B, K)

        r_dist = Beta(r_alpha, r_beta)
        r_samples = r_dist.sample().clamp(0.0, 1.0)

        # Sample z_rel from predicted Beta distribution.
        z_alpha = outputs["z_alpha"].clamp_min(1e-4).unsqueeze(1).expand(B, K)
        z_beta = outputs["z_beta"].clamp_min(1e-4).unsqueeze(1).expand(B, K)

        z_dist = Beta(z_alpha, z_beta)
        z_samples = z_dist.sample().clamp(0.0, 1.0)

        # 9E learned theta direction.
        # theta_vec is trained as [sin(theta), cos(theta)].
        theta_vec = F.normalize(outputs["theta_vec"], dim=-1)
        theta_mu = torch.atan2(theta_vec[:, 0], theta_vec[:, 1])

        theta_samples = theta_mu.unsqueeze(1).expand(B, K).clone()

        # Add jitter to prevent all generated molecules from collapsing to one angle.
        theta_samples = theta_samples + torch.randn((B, K), device=device) * THETA_JITTER

        # Wrap theta back to [-pi, pi].
        theta_samples = ((theta_samples + torch.pi) % (2 * torch.pi)) - torch.pi

        # Model-predicted nuclear probability.
        p_nuc_pred = torch.sigmoid(outputs["pnuc_logit"])

        # Move to CPU numpy.
        r_samples_np = r_samples.detach().cpu().numpy()
        z_samples_np = z_samples.detach().cpu().numpy()
        theta_samples_np = theta_samples.detach().cpu().numpy()
        p_nuc_pred_np = p_nuc_pred.detach().cpu().numpy()

    # Free GPU tensors from this batch.
    del ctx_t, mask_t, gene_t, ct_t, geom_t
    del outputs, r_samples, z_samples, theta_samples, p_nuc_pred

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------------------------
    # Convert normalized coordinates to absolute x/y/z and build records
    # --------------------------------------------------------------------------
    for b, target in enumerate(batch_targets):
        cid = int(target["cell_id"])
        gid = str(target["gene_id"])
        row = int(target["row"])
        n_imp = int(target["n_impute"])

        if n_imp <= 0:
            continue

        if cid not in cell_edge_lookup:
            raise KeyError(f"Missing cell_edge_lookup for cell_id={cid}")

        if cid not in nuc_centroids:
            raise KeyError(f"Missing nucleus centroid for cell_id={cid}")

        r_norms = r_samples_np[b, :n_imp]
        z_rels = z_samples_np[b, :n_imp]
        thetas = theta_samples_np[b, :n_imp]

        abs_x, abs_y, abs_z, p_nuc_geom = convert_to_absolute_fast(
            r_norms,
            thetas,
            z_rels,
            cid,
        )

        ct_label = str(cell_type_labels[row])

        # One confidence value per cell-gene pair from the model's nuclear-probability head.
        # Keep bounded away from zero so downstream weighted analyses do not discard them.
        confidence = float(np.clip(p_nuc_pred_np[b], 0.1, 1.0))

        for k in range(n_imp):
            global_imp_idx = n_generated_so_far + k

            imputed_records.append({
                # Keep transcript_id numeric-compatible with old molecule table.
                # Add unique imputed_molecule_id for reliable downstream tracking.
                "transcript_id": -1,
                "imputed_molecule_id": f"imputed_9E_{global_imp_idx}",

                "cell_id": cid,
                "overlaps_nucleus": int(p_nuc_geom[k] > 0.5),
                "gene_id": gid,

                # Absolute Xenium-like coordinates.
                "x": float(abs_x[k]),
                "y": float(abs_y[k]),
                "z": float(abs_z[k]),

                # No raw Xenium quality for generated molecules.
                "quality": np.nan,

                "Assigned_Xenium_Cell_Type": ct_label,

                # Normalized learned coordinates.
                "r_norm": float(r_norms[k]),
                "theta": float(thetas[k]),
                "z_rel": float(z_rels[k]),

                # Geometry-derived nuclear assignment for generated absolute coordinate.
                "p_nuclear": float(p_nuc_geom[k]),

                # Model output probability for this cell-gene pair.
                "p_nuclear_model_prob": float(p_nuc_pred_np[b]),

                # Status flags.
                "status": "imputed",
                "weight": confidence,
                "is_imputed": True,
                "imputation_confidence": confidence,
                "imputation_model": "9E_best_coordNN",

                # Traceability.
                "source_run": RUN_NAME,
                "n_impute_for_pair": n_imp,
            })

        n_generated_so_far += n_imp

    # Optional progress every ~200k generated molecules.
    if n_generated_so_far > 0 and n_generated_so_far % 200_000 < max(n_gen_list):
        elapsed_min = (time.time() - t0) / 60
        print(
            f"  Generated {n_generated_so_far:,}/{total_to_generate:,} molecules "
            f"({elapsed_min:.1f} min elapsed)"
        )

    # Free CPU arrays.
    del ctx_batch, mask_batch, gene_batch, ct_batch, geom_batch
    del r_samples_np, z_samples_np, theta_samples_np, p_nuc_pred_np
    gc.collect()

# ------------------------------------------------------------------------------
# 4. Final checks
# ------------------------------------------------------------------------------

elapsed = time.time() - t0

print("\n" + "=" * 90)
print("9E IMPUTATION GENERATION COMPLETE")
print("=" * 90)
print(f"Generated imputed molecules: {len(imputed_records):,}")
print(f"Expected imputed molecules : {total_to_generate:,}")
print(f"Difference                 : {len(imputed_records) - total_to_generate:,}")
print(f"Elapsed                    : {elapsed / 60:.1f} min")

if len(imputed_records) != int(total_to_generate):
    raise RuntimeError(
        f"Generated molecule count does not match expected total: "
        f"{len(imputed_records):,} vs {total_to_generate:,}"
    )

if len(imputed_records) > 0:
    confs = np.array([r["imputation_confidence"] for r in imputed_records], dtype=np.float32)

    print("\nConfidence summary:")
    print(f"  Mean confidence: {np.mean(confs):.4f}")
    print(f"  Min confidence : {np.min(confs):.4f}")
    print(f"  Max confidence : {np.max(confs):.4f}")

    # Quick preview.
    preview_df = pd.DataFrame(imputed_records[:5])
    print("\nPreview of first 5 imputed records:")
    display(preview_df)

print("\nInfer-2 complete. Next run the corrected Infer-3 save/check cell.")

In [ ]:
# ==============================================================================
# CELL S5-9E-INFER-3 — Save 9E imputed records + count reconciliation
# Corrected version with transcript_id dtype fix
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd

print("=" * 90)
print("SUBSTEP 5H-9E: Save 9E imputed records and verify counts")
print("=" * 90)

# ------------------------------------------------------------------------------
# 0. Paths / configuration
# ------------------------------------------------------------------------------

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RUN_NAME = "attempt_9E_distribution_empirical_baselines"

IMPUTED_RECORDS_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_imputed_records.parquet"
)

COMPLETED_MOL_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_completed_molecule_table.parquet"
)

IMPUTATION_SUMMARY_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_imputation_summary.csv"
)

COUNT_CHECK_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_count_reconciliation.csv"
)

# Temporary files for safer overwrite.
IMPUTED_RECORDS_TMP_PATH = IMPUTED_RECORDS_PATH + ".tmp"
COMPLETED_MOL_TMP_PATH = COMPLETED_MOL_PATH + ".tmp"
IMPUTATION_SUMMARY_TMP_PATH = IMPUTATION_SUMMARY_PATH + ".tmp"
COUNT_CHECK_TMP_PATH = COUNT_CHECK_PATH + ".tmp"

print(f"RUN_NAME: {RUN_NAME}")
print(f"IMPUTED_RECORDS_PATH: {IMPUTED_RECORDS_PATH}")
print(f"COMPLETED_MOL_PATH: {COMPLETED_MOL_PATH}")
print(f"IMPUTATION_SUMMARY_PATH: {IMPUTATION_SUMMARY_PATH}")
print(f"COUNT_CHECK_PATH: {COUNT_CHECK_PATH}")

# ------------------------------------------------------------------------------
# 1. Required variable checks
# ------------------------------------------------------------------------------

required_vars = [
    "mol",
    "imputation_targets_df",
    "total_to_generate",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable: {v}")

# imputed_records may be in memory after Infer-2.
# If not, reload imputed_df from the already-saved parquet.
if "imputed_records" in globals() and len(imputed_records) > 0:
    print("\nBuilding imputed_df from imputed_records in memory...")
    imputed_df = pd.DataFrame(imputed_records)

elif "imputed_df" in globals() and len(imputed_df) > 0:
    print("\nUsing existing imputed_df from memory...")

elif os.path.exists(IMPUTED_RECORDS_PATH):
    print("\nimputed_records not found in memory. Loading saved imputed records:")
    print(f"  {IMPUTED_RECORDS_PATH}")
    imputed_df = pd.read_parquet(IMPUTED_RECORDS_PATH)

else:
    raise RuntimeError(
        "Could not find imputed_records, imputed_df, or saved imputed records parquet. "
        "Run Infer-2 first."
    )

print(f"imputed_df shape: {imputed_df.shape}")

if len(imputed_df) == 0:
    raise RuntimeError("imputed_df is empty. Do not continue.")

# ------------------------------------------------------------------------------
# 2. Basic imputed_df schema checks
# ------------------------------------------------------------------------------

expected_imputed_cols = [
    "cell_id",
    "gene_id",
    "x",
    "y",
    "z",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "status",
    "is_imputed",
]

missing_imputed_cols = [c for c in expected_imputed_cols if c not in imputed_df.columns]

if missing_imputed_cols:
    raise KeyError(f"imputed_df is missing expected columns: {missing_imputed_cols}")

# Ensure unique imputed molecule IDs exist.
if "imputed_molecule_id" not in imputed_df.columns:
    print("imputed_molecule_id missing; creating it.")
    imputed_df["imputed_molecule_id"] = [
        f"imputed_9E_{i}" for i in range(len(imputed_df))
    ]

# Add transcript_id if missing.
if "transcript_id" not in imputed_df.columns:
    print("transcript_id missing in imputed_df; creating from imputed_molecule_id.")
    imputed_df["transcript_id"] = imputed_df["imputed_molecule_id"].astype(str)

# Add inference_seed to imputed records if available.
if "SEED_INFER" in globals() and "inference_seed" not in imputed_df.columns:
    imputed_df["inference_seed"] = int(SEED_INFER)

# ------------------------------------------------------------------------------
# 3. Count reconciliation BEFORE saving huge completed table
# ------------------------------------------------------------------------------

print("\nRunning count reconciliation check...")

# Force consistent merge dtypes.
imputed_df["cell_id"] = imputed_df["cell_id"].astype(int)
imputed_df["gene_id"] = imputed_df["gene_id"].astype(str)

imputation_targets_df["cell_id"] = imputation_targets_df["cell_id"].astype(int)
imputation_targets_df["gene_id"] = imputation_targets_df["gene_id"].astype(str)

imputed_counts = (
    imputed_df
    .groupby(["cell_id", "gene_id"])
    .size()
    .reset_index(name="n_imputed_actual")
)

target_counts = imputation_targets_df[["cell_id", "gene_id", "n_impute"]].copy()
target_counts = target_counts.rename(columns={"n_impute": "n_imputed_expected"})

check_df = target_counts.merge(
    imputed_counts,
    on=["cell_id", "gene_id"],
    how="left",
)

check_df["n_imputed_actual"] = check_df["n_imputed_actual"].fillna(0).astype(int)
check_df["n_imputed_expected"] = check_df["n_imputed_expected"].astype(int)
check_df["diff"] = check_df["n_imputed_actual"] - check_df["n_imputed_expected"]

n_bad = int((check_df["diff"] != 0).sum())
max_abs_diff = int(check_df["diff"].abs().max()) if len(check_df) else 0

expected_sum = int(check_df["n_imputed_expected"].sum())
actual_sum = int(check_df["n_imputed_actual"].sum())

print(f"Target cell-gene pairs       : {len(check_df):,}")
print(f"Expected imputed molecules   : {expected_sum:,}")
print(f"Actual imputed molecules     : {actual_sum:,}")
print(f"Mismatched target pairs      : {n_bad:,}")
print(f"Max abs diff expected/actual : {max_abs_diff}")

# Save count reconciliation using temp file, then replace old file.
check_df.to_csv(COUNT_CHECK_TMP_PATH, index=False)
os.replace(COUNT_CHECK_TMP_PATH, COUNT_CHECK_PATH)

print(f"Saved count reconciliation:")
print(f"  {COUNT_CHECK_PATH}")

if n_bad > 0:
    print("\nWARNING: Some imputation target counts do not match generated records.")
    display(check_df[check_df["diff"] != 0].head(20))
    raise RuntimeError(
        "Count reconciliation failed. Do not save completed molecule table until this is fixed."
    )

if actual_sum != int(total_to_generate):
    raise RuntimeError(
        f"Total generated molecules mismatch: actual {actual_sum:,}, "
        f"expected total_to_generate {int(total_to_generate):,}"
    )

print("Count reconciliation passed: every target pair has expected number of imputed molecules.")

# ------------------------------------------------------------------------------
# 4. Save imputed records
# ------------------------------------------------------------------------------

print("\nSaving imputed records to:")
print(f"  {IMPUTED_RECORDS_PATH}")

# Clean any previous temp file.
if os.path.exists(IMPUTED_RECORDS_TMP_PATH):
    os.remove(IMPUTED_RECORDS_TMP_PATH)

imputed_df.to_parquet(IMPUTED_RECORDS_TMP_PATH, index=False)
os.replace(IMPUTED_RECORDS_TMP_PATH, IMPUTED_RECORDS_PATH)

print("Saved imputed records.")

# ------------------------------------------------------------------------------
# 5. Build observed molecule table
# ------------------------------------------------------------------------------

print("\nBuilding observed molecule table...")

observed_mol = mol[mol["status"].astype(str) == "observed"].copy()

print(f"observed_mol shape: {observed_mol.shape}")

if len(observed_mol) == 0:
    raise RuntimeError("observed_mol is empty. Check mol['status'].")

# ------------------------------------------------------------------------------
# 6. Fix transcript_id dtype before concatenation
# ------------------------------------------------------------------------------

print("\nFixing transcript_id dtype for Parquet compatibility...")

if "transcript_id" not in observed_mol.columns:
    print("WARNING: observed_mol has no transcript_id column. Creating transcript_id from row index.")
    observed_mol["transcript_id"] = observed_mol.index.astype(str)

if "transcript_id" not in imputed_df.columns:
    imputed_df["transcript_id"] = imputed_df["imputed_molecule_id"].astype(str)

# Detect observed transcript_id type.
non_null_tid = observed_mol["transcript_id"].dropna()

if len(non_null_tid) > 0:
    observed_tid_example = non_null_tid.iloc[0]
else:
    observed_tid_example = None

print(f"Observed transcript_id example: {observed_tid_example}")
print(f"Observed transcript_id type   : {type(observed_tid_example)}")

# Main fix:
# If observed transcript_id is bytes, imputed transcript_id must also be bytes.
# Otherwise, convert both observed and imputed transcript_id to string.
if isinstance(observed_tid_example, (bytes, bytearray)):
    print("Detected bytes-like observed transcript_id.")
    print("Converting imputed transcript_id to bytes using imputed_molecule_id.")

    imputed_df["transcript_id"] = imputed_df["imputed_molecule_id"].astype(str).map(
        lambda s: s.encode("utf-8")
    )

else:
    print("Detected non-bytes observed transcript_id.")
    print("Converting both observed and imputed transcript_id to string.")

    observed_mol["transcript_id"] = observed_mol["transcript_id"].astype(str)
    imputed_df["transcript_id"] = imputed_df["imputed_molecule_id"].astype(str)

# ------------------------------------------------------------------------------
# 7. Normalize common columns to avoid Arrow mixed-type errors
# ------------------------------------------------------------------------------

print("\nNormalizing common column dtypes...")

# Text-like columns.
text_cols = [
    "gene_id",
    "Assigned_Xenium_Cell_Type",
    "status",
    "imputed_molecule_id",
    "imputation_model",
    "source_run",
]

for col in text_cols:
    if col in observed_mol.columns:
        # Avoid forcing missing columns into literal "nan" if possible.
        observed_mol[col] = observed_mol[col].where(observed_mol[col].notna(), None)
        observed_mol[col] = observed_mol[col].astype("string")

    if col in imputed_df.columns:
        imputed_df[col] = imputed_df[col].where(imputed_df[col].notna(), None)
        imputed_df[col] = imputed_df[col].astype("string")

# Numeric columns.
numeric_cols = [
    "cell_id",
    "overlaps_nucleus",
    "x",
    "y",
    "z",
    "quality",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "p_nuclear_model_prob",
    "weight",
    "imputation_confidence",
    "n_impute_for_pair",
    "inference_seed",
]

for col in numeric_cols:
    if col in observed_mol.columns:
        observed_mol[col] = pd.to_numeric(observed_mol[col], errors="coerce")
    if col in imputed_df.columns:
        imputed_df[col] = pd.to_numeric(imputed_df[col], errors="coerce")

# Boolean column.
if "is_imputed" in observed_mol.columns:
    observed_mol["is_imputed"] = observed_mol["is_imputed"].fillna(False).astype(bool)
else:
    observed_mol["is_imputed"] = False

if "is_imputed" in imputed_df.columns:
    imputed_df["is_imputed"] = imputed_df["is_imputed"].fillna(True).astype(bool)
else:
    imputed_df["is_imputed"] = True

# ------------------------------------------------------------------------------
# 8. Align columns before concatenation
# ------------------------------------------------------------------------------

print("\nAligning observed and imputed table columns...")

all_cols = list(dict.fromkeys(list(observed_mol.columns) + list(imputed_df.columns)))

for col in all_cols:
    if col not in observed_mol.columns:
        observed_mol[col] = np.nan
    if col not in imputed_df.columns:
        imputed_df[col] = np.nan

observed_mol = observed_mol[all_cols]
imputed_df = imputed_df[all_cols]

print(f"Observed columns: {len(observed_mol.columns)}")
print(f"Imputed columns : {len(imputed_df.columns)}")

# ------------------------------------------------------------------------------
# 9. Build completed molecule table
# ------------------------------------------------------------------------------

print("\nBuilding completed molecule table...")

completed_mol = pd.concat([observed_mol, imputed_df], ignore_index=True)

print(f"observed molecules : {len(observed_mol):,}")
print(f"imputed molecules  : {len(imputed_df):,}")
print(f"completed molecules: {len(completed_mol):,}")

expected_completed = len(observed_mol) + len(imputed_df)

if len(completed_mol) != expected_completed:
    raise RuntimeError(
        f"Completed molecule table size mismatch: "
        f"{len(completed_mol):,} vs expected {expected_completed:,}"
    )

# Final sanity check.
if int(len(imputed_df)) != int(total_to_generate):
    raise RuntimeError(
        f"Imputed count mismatch: {len(imputed_df):,} vs expected {int(total_to_generate):,}"
    )

# ------------------------------------------------------------------------------
# 10. Save completed molecule table
# ------------------------------------------------------------------------------

print("\nSaving completed molecule table to:")
print(f"  {COMPLETED_MOL_PATH}")

if os.path.exists(COMPLETED_MOL_TMP_PATH):
    os.remove(COMPLETED_MOL_TMP_PATH)

completed_mol.to_parquet(COMPLETED_MOL_TMP_PATH, index=False)
os.replace(COMPLETED_MOL_TMP_PATH, COMPLETED_MOL_PATH)

print("Saved completed molecule table.")

# ------------------------------------------------------------------------------
# 11. Save summary
# ------------------------------------------------------------------------------

summary = {
    "run_name": RUN_NAME,
    "n_observed_molecules": int(len(observed_mol)),
    "n_imputed_molecules": int(len(imputed_df)),
    "n_completed_molecules": int(len(completed_mol)),
    "n_target_pairs": int(len(imputation_targets_df)),
    "expected_imputed_molecules": int(total_to_generate),
    "actual_imputed_molecules": int(len(imputed_df)),
    "count_mismatch_pairs": int(n_bad),
    "max_abs_count_diff": int(max_abs_diff),
    "imputed_records_path": IMPUTED_RECORDS_PATH,
    "completed_molecule_table_path": COMPLETED_MOL_PATH,
    "count_reconciliation_path": COUNT_CHECK_PATH,
    "transcript_id_fix": "fixed mixed transcript_id dtype before parquet save",
}

if "SEED_INFER" in globals():
    summary["inference_seed"] = int(SEED_INFER)

summary_df = pd.DataFrame([summary])

summary_df.to_csv(IMPUTATION_SUMMARY_TMP_PATH, index=False)
os.replace(IMPUTATION_SUMMARY_TMP_PATH, IMPUTATION_SUMMARY_PATH)

print(f"\nSaved imputation summary:")
print(f"  {IMPUTATION_SUMMARY_PATH}")

display(summary_df)

# ------------------------------------------------------------------------------
# 12. Optional preview / final checks
# ------------------------------------------------------------------------------

print("\nCompleted molecule table status counts:")
display(
    completed_mol["status"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "status", "status": "n_molecules"})
)

print("\nImputed molecule table preview:")
display(imputed_df.head())

# ------------------------------------------------------------------------------
# 13. Cleanup
# ------------------------------------------------------------------------------

gc.collect()

print("\n" + "=" * 90)
print("9E IMPUTATION SAVE/CHECK COMPLETE")
print("=" * 90)
print("Saved files:")
print(f"  Imputed records      : {IMPUTED_RECORDS_PATH}")
print(f"  Completed molecule table: {COMPLETED_MOL_PATH}")
print(f"  Count reconciliation : {COUNT_CHECK_PATH}")
print(f"  Summary              : {IMPUTATION_SUMMARY_PATH}")

In [ ]:
import os

paths = [
    "/content/drive/MyDrive/diffusion/latest_run/attempt_9E_distribution_empirical_baselines_imputed_records.parquet",
    "/content/drive/MyDrive/diffusion/latest_run/attempt_9E_distribution_empirical_baselines_completed_molecule_table.parquet",
    "/content/drive/MyDrive/diffusion/latest_run/attempt_9E_distribution_empirical_baselines_count_reconciliation.csv",
    "/content/drive/MyDrive/diffusion/latest_run/attempt_9E_distribution_empirical_baselines_imputation_summary.csv",
]

for p in paths:
    print(("✓" if os.path.exists(p) else "✗"), p,
          f"{os.path.getsize(p)/1e9:.2f} GB" if os.path.exists(p) else "")

==========IMPUTATION===========

======SAFE TO DISCONNECT RUNTIME======

In [ ]:
# ==============================================================================
# CELL S5-9E-BASELINE-1 — Generate 3 empirical baseline imputed records
# Baselines:
#   1. Gene-level empirical distribution
#   2. Cell-type gene empirical distribution
#   3. Spatial-kNN empirical distribution
#
# CPU/RAM-heavy. GPU not needed.
# ==============================================================================

import os
import gc
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.neighbors import BallTree

print("=" * 90)
print("SUBSTEP 5I-9E: Generate empirical baseline imputed records")
print("=" * 90)

t0_all = time.time()

# ------------------------------------------------------------------------------
# 0. Paths / configuration
# ------------------------------------------------------------------------------

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RUN_NAME = "attempt_9E_distribution_empirical_baselines"

BASELINE_SEED = 12345
SPATIAL_KNN_K = 80

rng_global = np.random.default_rng(BASELINE_SEED)

baseline_paths = {
    "gene_emp": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_gene_emp_imputed_records.parquet"
    ),
    "ct_gene_emp": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_ct_gene_emp_imputed_records.parquet"
    ),
    "spatial_knn_emp": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_spatial_knn_emp_imputed_records.parquet"
    ),
}

baseline_summary_path = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_baseline_imputation_summary.csv"
)

print(f"RUN_NAME: {RUN_NAME}")
print(f"BASELINE_SEED: {BASELINE_SEED}")
print(f"SPATIAL_KNN_K: {SPATIAL_KNN_K}")

for k, p in baseline_paths.items():
    print(f"{k:16s}: {p}")

# ------------------------------------------------------------------------------
# 1. Required variable checks
# ------------------------------------------------------------------------------

required_vars = [
    "imputation_targets",
    "imputation_targets_df",
    "total_to_generate",
    "baseline_gene_pools",
    "baseline_ct_gene_pools",
    "baseline_spatial_index",
    "convert_to_absolute_fast",
    "cell_type_labels",
    "cell_edge_lookup",
    "nuc_centroids",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable: {v}")

print(f"\nTarget cell-gene pairs: {len(imputation_targets):,}")
print(f"Expected molecules per baseline: {total_to_generate:,}")
print(f"Gene empirical pools: {len(baseline_gene_pools):,}")
print(f"Cell-type gene empirical pools: {len(baseline_ct_gene_pools):,}")
print(f"Spatial-kNN empirical groups: {len(baseline_spatial_index):,}")

# ------------------------------------------------------------------------------
# 2. Build spatial BallTrees once for speed
# ------------------------------------------------------------------------------

print("\nBuilding spatial BallTrees for baseline_spatial_index...")

spatial_tree_index = {}

for key, info in tqdm(
    baseline_spatial_index.items(),
    desc="Building spatial BallTrees",
    mininterval=5,
):
    xy = np.asarray(info["xy"], dtype=np.float32)

    if len(xy) == 0:
        continue

    spatial_tree_index[key] = {
        "tree": BallTree(xy),
        "xy": xy,
        "cell_ids": info["cell_ids"],
        "values": info["values"],
    }

print(f"Spatial BallTrees built: {len(spatial_tree_index):,}")

# ------------------------------------------------------------------------------
# 3. Sampling helpers
# ------------------------------------------------------------------------------

def fallback_sample_uniform(n_imp, rng):
    """
    Used only if no empirical pool is available.
    """
    r = rng.uniform(0.0, 1.0, size=n_imp).astype(np.float32)
    theta = rng.uniform(-np.pi, np.pi, size=n_imp).astype(np.float32)
    z = rng.uniform(0.0, 1.0, size=n_imp).astype(np.float32)
    p_nuc = np.zeros(n_imp, dtype=np.float32)

    return np.stack([r, theta, z, p_nuc], axis=-1).astype(np.float32)


def sample_gene_empirical_for_target(target, n_imp, rng):
    gene_idx = int(target["gene_idx"])

    pool = baseline_gene_pools.get(gene_idx, None)

    if pool is None or len(pool) == 0:
        return fallback_sample_uniform(n_imp, rng)

    idx = rng.choice(len(pool), size=n_imp, replace=True)
    return pool[idx].astype(np.float32)


def sample_ct_gene_empirical_for_target(target, n_imp, rng):
    key = (int(target["ct_idx"]), int(target["gene_idx"]))

    pool = baseline_ct_gene_pools.get(key, None)

    if pool is None or len(pool) == 0:
        return sample_gene_empirical_for_target(target, n_imp, rng)

    idx = rng.choice(len(pool), size=n_imp, replace=True)
    return pool[idx].astype(np.float32)


def sample_spatial_knn_empirical_for_target(target, n_imp, rng, k_neighbors=80, min_pool=5):
    key = (int(target["ct_idx"]), int(target["gene_idx"]))

    info = spatial_tree_index.get(key, None)

    if info is None:
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    xy = info["xy"]
    values_list = info["values"]

    if len(xy) == 0:
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    # Need target cell coordinate. If not present in target, use fallback.
    if "cell_xy" in target:
        query_xy = np.asarray(target["cell_xy"], dtype=np.float32).reshape(1, -1)
    elif "cell_xy_lookup" in globals() and int(target["cell_id"]) in cell_xy_lookup:
        query_xy = np.asarray(cell_xy_lookup[int(target["cell_id"])], dtype=np.float32).reshape(1, -1)
    else:
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    k = min(k_neighbors, len(xy))

    if k < 1:
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    _, ind = info["tree"].query(query_xy, k=k)

    candidate_vals = []

    for idx in ind[0]:
        vals = values_list[int(idx)]
        if vals is not None and len(vals) > 0:
            candidate_vals.append(vals)

    if len(candidate_vals) < min_pool:
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    pool = np.concatenate(candidate_vals, axis=0)

    if len(pool) == 0:
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    idx = rng.choice(len(pool), size=n_imp, replace=True)
    return pool[idx].astype(np.float32)


def sample_for_method(method_key, target, n_imp, rng):
    if method_key == "gene_emp":
        return sample_gene_empirical_for_target(target, n_imp, rng)

    if method_key == "ct_gene_emp":
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    if method_key == "spatial_knn_emp":
        return sample_spatial_knn_empirical_for_target(
            target,
            n_imp,
            rng,
            k_neighbors=SPATIAL_KNN_K,
        )

    raise ValueError(f"Unknown baseline method: {method_key}")


# ------------------------------------------------------------------------------
# 4. Generate one baseline at a time
# ------------------------------------------------------------------------------

def generate_baseline_imputed_records(method_key, method_label, seed_offset=0):
    """
    Generate imputed molecule records for one empirical baseline method.
    Saves one parquet file per baseline.
    """

    print("\n" + "=" * 90)
    print(f"Generating baseline: {method_key} — {method_label}")
    print("=" * 90)

    t0 = time.time()

    rng = np.random.default_rng(BASELINE_SEED + seed_offset)

    records = []
    generated = 0

    for target in tqdm(
        imputation_targets,
        desc=f"{method_key} imputation",
        mininterval=10,
    ):
        cid = int(target["cell_id"])
        gid = str(target["gene_id"])
        row = int(target["row"])
        n_imp = int(target["n_impute"])

        if n_imp <= 0:
            continue

        if cid not in cell_edge_lookup:
            raise KeyError(f"Missing cell_edge_lookup for cell_id={cid}")

        if cid not in nuc_centroids:
            raise KeyError(f"Missing nuc_centroids for cell_id={cid}")

        sampled_vals = sample_for_method(method_key, target, n_imp, rng)

        # sampled_vals expected shape: [n_imp, 4] = [r_norm, theta, z_rel, p_nuclear]
        if sampled_vals.ndim != 2 or sampled_vals.shape[0] != n_imp:
            raise ValueError(
                f"Bad sampled_vals shape for {method_key}, target {target}: "
                f"{sampled_vals.shape}, expected [{n_imp}, >=3]"
            )

        r_norms = np.clip(sampled_vals[:, 0].astype(np.float32), 0.0, 1.0)
        thetas = sampled_vals[:, 1].astype(np.float32)
        thetas = ((thetas + np.pi) % (2 * np.pi)) - np.pi
        z_rels = np.clip(sampled_vals[:, 2].astype(np.float32), 0.0, 1.0)

        # p_nuclear from sampled baseline pool, if available.
        if sampled_vals.shape[1] >= 4:
            p_nuc_sampled = sampled_vals[:, 3].astype(np.float32)
        else:
            p_nuc_sampled = np.zeros(n_imp, dtype=np.float32)

        abs_x, abs_y, abs_z, p_nuc_geom = convert_to_absolute_fast(
            r_norms,
            thetas,
            z_rels,
            cid,
        )

        ct_label = str(cell_type_labels[row])

        for k in range(n_imp):
            global_idx = generated + k

            records.append({
                "transcript_id": f"{method_key}_imputed_{global_idx}",
                "imputed_molecule_id": f"{method_key}_imputed_{global_idx}",

                "cell_id": cid,
                "overlaps_nucleus": int(p_nuc_geom[k] > 0.5),
                "gene_id": gid,

                "x": float(abs_x[k]),
                "y": float(abs_y[k]),
                "z": float(abs_z[k]),

                "quality": np.nan,
                "Assigned_Xenium_Cell_Type": ct_label,

                "r_norm": float(r_norms[k]),
                "theta": float(thetas[k]),
                "z_rel": float(z_rels[k]),

                # Geometry-derived nuclear assignment after absolute conversion.
                "p_nuclear": float(p_nuc_geom[k]),

                # Nuclear value sampled from empirical pool before geometry conversion.
                "p_nuclear_sampled": float(p_nuc_sampled[k]),

                "status": "imputed",
                "weight": 1.0,
                "is_imputed": True,
                "imputation_confidence": 1.0,

                "imputation_model": method_key,
                "source_run": RUN_NAME,
                "baseline_method": method_key,
                "baseline_label": method_label,
                "baseline_seed": int(BASELINE_SEED + seed_offset),
                "n_impute_for_pair": n_imp,
            })

        generated += n_imp

    elapsed = time.time() - t0

    print("\nConverting records to DataFrame...")
    baseline_df = pd.DataFrame(records)

    print(f"{method_key} generated molecules: {len(baseline_df):,}")
    print(f"Expected molecules          : {total_to_generate:,}")
    print(f"Difference                  : {len(baseline_df) - int(total_to_generate):,}")
    print(f"Elapsed                     : {elapsed / 60:.1f} min")

    if len(baseline_df) != int(total_to_generate):
        raise RuntimeError(
            f"{method_key} generated count mismatch: "
            f"{len(baseline_df):,} vs {int(total_to_generate):,}"
        )

    out_path = baseline_paths[method_key]

    print(f"Saving {method_key} imputed records:")
    print(f"  {out_path}")

    baseline_df.to_parquet(out_path, index=False)

    print(f"Saved {method_key} baseline records.")

    summary = {
        "run_name": RUN_NAME,
        "baseline_method": method_key,
        "baseline_label": method_label,
        "n_imputed_molecules": int(len(baseline_df)),
        "expected_imputed_molecules": int(total_to_generate),
        "seed": int(BASELINE_SEED + seed_offset),
        "output_path": out_path,
        "elapsed_min": float(elapsed / 60),
    }

    # Free memory before next baseline.
    del records
    del baseline_df
    gc.collect()

    return summary


baseline_summaries = []

baseline_summaries.append(
    generate_baseline_imputed_records(
        method_key="gene_emp",
        method_label="Gene-level empirical distribution",
        seed_offset=0,
    )
)

baseline_summaries.append(
    generate_baseline_imputed_records(
        method_key="ct_gene_emp",
        method_label="Cell-type gene empirical distribution",
        seed_offset=1000,
    )
)

baseline_summaries.append(
    generate_baseline_imputed_records(
        method_key="spatial_knn_emp",
        method_label="Spatial-kNN empirical distribution",
        seed_offset=2000,
    )
)

baseline_summary_df = pd.DataFrame(baseline_summaries)
baseline_summary_df.to_csv(baseline_summary_path, index=False)

print("\n" + "=" * 90)
print("9E EMPIRICAL BASELINE IMPUTATION COMPLETE")
print("=" * 90)
display(baseline_summary_df)

print(f"Saved baseline summary:")
print(f"  {baseline_summary_path}")

print(f"Total elapsed: {(time.time() - t0_all) / 60:.1f} min")

In [ ]:
# ==============================================================================
# CELL S5-9E-DOWNSTREAM-1 — Corrected downstream validation
# Raw / Step4 / Learned 9E / 3 empirical baselines
#
# This version matches the Step 4 cell-level denoising clustering validation:
#   - sample_size = 50,000
#   - random_state = 42
#   - Scanpy normalize_total + log1p
#   - Scanpy PCA with 30 PCs
#   - Scanpy neighbors
#   - Leiden clustering, resolution = 0.5
#   - ARI/NMI: true cell type labels vs Leiden clusters
#   - Silhouette: PCA coordinates vs true cell type labels
#   - n_clusters: number of Leiden clusters
#
# GPU not needed.
# ==============================================================================

import os
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import sparse

# ------------------------------------------------------------------------------
# 0. Imports
# ------------------------------------------------------------------------------

try:
    import scanpy as sc
    import anndata as ad
except Exception as e:
    print(f"Scanpy import failed: {e}")
    print("Installing scanpy/leiden dependencies...")
    !pip install -q scanpy leidenalg igraph anndata
    import scanpy as sc
    import anndata as ad

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
)

print("=" * 90)
print("SUBSTEP 5J-9E: Downstream validation using Step-4-identical clustering logic")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Paths / configuration
# ------------------------------------------------------------------------------

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
RUN_NAME = "attempt_9E_distribution_empirical_baselines"

LEARNED_IMPUTED_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_imputed_records.parquet"
)

BASELINE_PATHS = {
    "Gene empirical completed counts": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_gene_emp_imputed_records.parquet"
    ),
    "Cell-type gene empirical completed counts": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_ct_gene_emp_imputed_records.parquet"
    ),
    "Spatial-kNN empirical completed counts": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_spatial_knn_emp_imputed_records.parquet"
    ),
}

COUNT_METRICS_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_downstream_count_matrix_metrics_step4matched.csv"
)

CLUSTERING_METRICS_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_downstream_clustering_metrics_step4matched.csv"
)

REPORT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_downstream_validation_report_step4matched.txt"
)

# Must match Step 4 validation setup.
SAMPLE_SIZE = 50000
RANDOM_STATE = 42
N_PCS = 30
LEIDEN_RESOLUTION = 0.5

print(f"RUN_NAME: {RUN_NAME}")
print(f"SAMPLE_SIZE: {SAMPLE_SIZE:,}")
print(f"RANDOM_STATE: {RANDOM_STATE}")
print(f"N_PCS: {N_PCS}")
print(f"LEIDEN_RESOLUTION: {LEIDEN_RESOLUTION}")

# ------------------------------------------------------------------------------
# 2. Required variable checks
# ------------------------------------------------------------------------------

required_vars = [
    "denoised_adata",
    "X_raw_counts",
    "X_denoised",
    "shared_genes",
    "cell_ids_step4",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable: {v}")

if "was_corrected" not in globals():
    print("WARNING: was_corrected not found. Corrected-pair metrics will be skipped.")
    was_corrected = None

# ------------------------------------------------------------------------------
# 3. Helper functions copied/adapted from Step 4 validation
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def get_cell_type_column(adata):
    """
    Finds the correct cell-type column.
    Same logic as Step 4 validation cell.
    """
    if "cell_type" in adata.obs.columns:
        return "cell_type"
    elif "Assigned_Xenium_Cell_Type" in adata.obs.columns:
        return "Assigned_Xenium_Cell_Type"
    else:
        raise KeyError(
            "No cell-type column found. Expected either 'cell_type' or "
            "'Assigned_Xenium_Cell_Type' in adata.obs."
        )


ct_col = get_cell_type_column(denoised_adata)
print(f"Cell-type column: {ct_col}")

# ------------------------------------------------------------------------------
# 4. Build cell/gene maps
# ------------------------------------------------------------------------------

cell_idx_map = {int(cid): i for i, cid in enumerate(cell_ids_step4)}
gene_idx_map = {str(g): i for i, g in enumerate(shared_genes)}

print(f"Cells in Step4 matrix: {len(cell_idx_map):,}")
print(f"Genes in Step4 matrix: {len(gene_idx_map):,}")

# ------------------------------------------------------------------------------
# 5. Convert base matrices
# ------------------------------------------------------------------------------

print("\nLoading/confirming raw and Step4 matrices...")

X_raw_full = ensure_dense(X_raw_counts).astype(np.float32)
X_step4_full = ensure_dense(X_denoised).astype(np.float32)

if X_raw_full.shape != X_step4_full.shape:
    raise ValueError(
        f"Shape mismatch: raw {X_raw_full.shape}, Step4 {X_step4_full.shape}"
    )

if X_raw_full.shape[0] != denoised_adata.n_obs:
    raise ValueError(
        f"Raw matrix rows {X_raw_full.shape[0]} do not match denoised_adata.n_obs {denoised_adata.n_obs}"
    )

if X_raw_full.shape[1] != len(shared_genes):
    raise ValueError(
        f"Raw matrix cols {X_raw_full.shape[1]} do not match shared_genes {len(shared_genes)}"
    )

print(f"Raw matrix shape: {X_raw_full.shape}")
print(f"Step4 matrix shape: {X_step4_full.shape}")
print(f"Raw matrix sum: {X_raw_full.sum(dtype=np.float64):,.0f}")
print(f"Step4 matrix sum: {X_step4_full.sum(dtype=np.float64):,.2f}")

# ------------------------------------------------------------------------------
# 6. Fixed Step-4-identical sampling
# ------------------------------------------------------------------------------

print("\nCreating fixed 50k-cell sample using Step 4 logic...")

n_cells = X_raw_full.shape[0]
n_sample = min(SAMPLE_SIZE, n_cells)

rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(n_cells, size=n_sample, replace=False)

obs_sub = denoised_adata.obs.iloc[sample_idx].copy()
var_sub = denoised_adata.var.copy()
true_labels = obs_sub[ct_col].astype(str).values

print(f"Sampled cells: {n_sample:,} / {n_cells:,}")
print(f"Sampling seed: {RANDOM_STATE}")
print(f"Unique true labels in sample: {pd.Series(true_labels).nunique()}")

# This sample_idx is reused for raw, Step4, learned 9E, and all baselines.
SAMPLE_INDEX_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_downstream_step4matched_sample_idx.npy"
)
np.save(SAMPLE_INDEX_PATH, sample_idx)
print(f"Saved fixed sample index:")
print(f"  {SAMPLE_INDEX_PATH}")

# ------------------------------------------------------------------------------
# 7. Count matrix helper functions
# ------------------------------------------------------------------------------

def add_imputed_counts_from_file(base_counts, imputed_path, label):
    """
    Builds completed count matrix:
        completed = X_raw_counts + counts(imputed_records)

    This reads only cell_id/gene_id from parquet to reduce memory.
    """
    if not os.path.exists(imputed_path):
        raise FileNotFoundError(f"Missing imputed records for {label}: {imputed_path}")

    print(f"\nLoading imputed counts for {label}:")
    print(f"  {imputed_path}")

    # Only need cell_id/gene_id for count matrix construction.
    imp = pd.read_parquet(imputed_path, columns=["cell_id", "gene_id"])

    print(f"  Loaded imputed rows: {len(imp):,}")

    imp["cell_id"] = imp["cell_id"].astype(int)
    imp["gene_id"] = imp["gene_id"].astype(str)

    counts = (
        imp
        .groupby(["cell_id", "gene_id"])
        .size()
        .reset_index(name="count")
    )

    print(f"  Unique imputed cell-gene pairs: {len(counts):,}")

    rr = counts["cell_id"].map(cell_idx_map)
    cc = counts["gene_id"].map(gene_idx_map)

    ok = rr.notna() & cc.notna()

    if ok.sum() < len(counts):
        print(
            f"  WARNING: dropped {len(counts) - int(ok.sum()):,} unmapped cell-gene count rows."
        )

    X = np.asarray(base_counts, dtype=np.float32).copy()

    X[
        rr[ok].astype(int).to_numpy(),
        cc[ok].astype(int).to_numpy(),
    ] += counts.loc[ok, "count"].to_numpy(dtype=np.float32)

    del imp, counts, rr, cc, ok
    gc.collect()

    return X


def count_matrix_metrics(name, X, X_target, X_raw, corrected_mask=None):
    """
    Count-level comparison against Step4 denoised target and raw counts.
    """
    X = np.asarray(X, dtype=np.float32)
    X_target = np.asarray(X_target, dtype=np.float32)
    X_raw = np.asarray(X_raw, dtype=np.float32)

    diff_target = X - X_target
    diff_raw = X - X_raw

    row = {
        "dataset": name,
        "total_counts": float(X.sum(dtype=np.float64)),
        "mean_counts_per_cell": float(X.sum(axis=1, dtype=np.float64).mean()),
        "nonzero_entries": int((X > 0).sum()),
        "mae_vs_step4_all": float(np.mean(np.abs(diff_target))),
        "rmse_vs_step4_all": float(np.sqrt(np.mean(diff_target ** 2))),
        "total_abs_diff_vs_step4": float(np.abs(diff_target).sum(dtype=np.float64)),
        "total_added_vs_raw": float(np.maximum(diff_raw, 0).sum(dtype=np.float64)),
    }

    if corrected_mask is not None and corrected_mask.shape == X.shape:
        m = corrected_mask.astype(bool)
        if m.sum() > 0:
            row["mae_vs_step4_corrected_pairs"] = float(np.mean(np.abs(diff_target[m])))
            row["rmse_vs_step4_corrected_pairs"] = float(np.sqrt(np.mean(diff_target[m] ** 2)))
            row["n_corrected_pairs"] = int(m.sum())
        else:
            row["mae_vs_step4_corrected_pairs"] = np.nan
            row["rmse_vs_step4_corrected_pairs"] = np.nan
            row["n_corrected_pairs"] = 0
    else:
        row["mae_vs_step4_corrected_pairs"] = np.nan
        row["rmse_vs_step4_corrected_pairs"] = np.nan
        row["n_corrected_pairs"] = np.nan

    return row


# ------------------------------------------------------------------------------
# 8. Step-4-matched clustering evaluation
# ------------------------------------------------------------------------------

def evaluate_clustering_quality_one_matrix(
    dataset_name,
    X_full,
    sample_idx,
    obs_sub,
    var_sub,
    true_labels,
    random_state=42,
    n_pcs=30,
    leiden_resolution=0.5,
):
    """
    Same method as Step 4 evaluate_clustering_quality(), generalized to one matrix.

    Exact shared setup:
      - same fixed sample_idx
      - same obs_sub / true_labels
      - normalize_total + log1p
      - PCA with n_pcs
      - neighbors
      - Leiden resolution=0.5
      - ARI/NMI vs true labels
      - silhouette over PCA coordinates vs true labels
      - n_clusters = number of Leiden clusters
    """

    print("\n" + "-" * 80)
    print(f"Evaluating clustering quality: {dataset_name}")
    print("-" * 80)

    X_full = np.asarray(X_full, dtype=np.float32)

    adata_tmp = ad.AnnData(
        X=X_full[sample_idx, :].copy(),
        obs=obs_sub.copy(),
        var=var_sub.copy(),
    )

    adata_tmp.var_names_make_unique()

    n_pcs_use = min(n_pcs, adata_tmp.shape[1] - 1)

    print(f"  AnnData shape: {adata_tmp.shape}")
    print(f"  Total counts in sampled matrix: {adata_tmp.X.sum(dtype=np.float64):,.2f}")

    # Same as Step 4.
    sc.pp.normalize_total(adata_tmp, target_sum=1e4)
    sc.pp.log1p(adata_tmp)

    sc.pp.pca(
        adata_tmp,
        n_comps=n_pcs_use,
        random_state=random_state,
    )

    sc.pp.neighbors(
        adata_tmp,
        n_pcs=n_pcs_use,
    )

    sc.tl.leiden(
        adata_tmp,
        resolution=leiden_resolution,
        random_state=random_state,
        key_added="leiden",
    )

    clusters = adata_tmp.obs["leiden"].astype(str).values

    ari = adjusted_rand_score(true_labels, clusters)
    nmi = normalized_mutual_info_score(true_labels, clusters)

    print("  Computing silhouette on full sampled set, matching Step 4 validation...")
    sil = silhouette_score(
        adata_tmp.obsm["X_pca"][:, :n_pcs_use],
        true_labels,
        metric="euclidean",
    )

    n_clusters = pd.Series(clusters).nunique()

    result = {
        "dataset": dataset_name,
        "ARI": float(ari),
        "NMI": float(nmi),
        "silhouette": float(sil),
        "n_clusters": int(n_clusters),
        "n_cells_used": int(adata_tmp.n_obs),
        "n_pcs": int(n_pcs_use),
        "sample_size": int(len(sample_idx)),
        "random_state": int(random_state),
        "leiden_resolution": float(leiden_resolution),
    }

    print(
        f"  ARI={ari:.4f}, NMI={nmi:.4f}, "
        f"silhouette={sil:.4f}, n_clusters={n_clusters}"
    )

    del adata_tmp
    gc.collect()

    return result


# ------------------------------------------------------------------------------
# 9. Build/evaluate matrices one by one
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("Building and evaluating downstream count matrices")
print("=" * 90)

count_metric_rows = []
clustering_rows = []

# Raw observed counts.
count_metric_rows.append(
    count_matrix_metrics(
        name="Raw observed counts",
        X=X_raw_full,
        X_target=X_step4_full,
        X_raw=X_raw_full,
        corrected_mask=was_corrected,
    )
)

clustering_rows.append(
    evaluate_clustering_quality_one_matrix(
        dataset_name="Raw observed counts",
        X_full=X_raw_full,
        sample_idx=sample_idx,
        obs_sub=obs_sub,
        var_sub=var_sub,
        true_labels=true_labels,
        random_state=RANDOM_STATE,
        n_pcs=N_PCS,
        leiden_resolution=LEIDEN_RESOLUTION,
    )
)

# Step4 denoised counts.
count_metric_rows.append(
    count_matrix_metrics(
        name="Step4 denoised counts",
        X=X_step4_full,
        X_target=X_step4_full,
        X_raw=X_raw_full,
        corrected_mask=was_corrected,
    )
)

clustering_rows.append(
    evaluate_clustering_quality_one_matrix(
        dataset_name="Step4 denoised counts",
        X_full=X_step4_full,
        sample_idx=sample_idx,
        obs_sub=obs_sub,
        var_sub=var_sub,
        true_labels=true_labels,
        random_state=RANDOM_STATE,
        n_pcs=N_PCS,
        leiden_resolution=LEIDEN_RESOLUTION,
    )
)

# Learned 9E completed counts.
X_learned_9E = add_imputed_counts_from_file(
    base_counts=X_raw_full,
    imputed_path=LEARNED_IMPUTED_PATH,
    label="Learned 9E completed counts",
)

count_metric_rows.append(
    count_matrix_metrics(
        name="Learned 9E completed counts",
        X=X_learned_9E,
        X_target=X_step4_full,
        X_raw=X_raw_full,
        corrected_mask=was_corrected,
    )
)

clustering_rows.append(
    evaluate_clustering_quality_one_matrix(
        dataset_name="Learned 9E completed counts",
        X_full=X_learned_9E,
        sample_idx=sample_idx,
        obs_sub=obs_sub,
        var_sub=var_sub,
        true_labels=true_labels,
        random_state=RANDOM_STATE,
        n_pcs=N_PCS,
        leiden_resolution=LEIDEN_RESOLUTION,
    )
)

del X_learned_9E
gc.collect()

# Baseline completed counts.
for dataset_name, path in BASELINE_PATHS.items():
    X_base = add_imputed_counts_from_file(
        base_counts=X_raw_full,
        imputed_path=path,
        label=dataset_name,
    )

    count_metric_rows.append(
        count_matrix_metrics(
            name=dataset_name,
            X=X_base,
            X_target=X_step4_full,
            X_raw=X_raw_full,
            corrected_mask=was_corrected,
        )
    )

    clustering_rows.append(
        evaluate_clustering_quality_one_matrix(
            dataset_name=dataset_name,
            X_full=X_base,
            sample_idx=sample_idx,
            obs_sub=obs_sub,
            var_sub=var_sub,
            true_labels=true_labels,
            random_state=RANDOM_STATE,
            n_pcs=N_PCS,
            leiden_resolution=LEIDEN_RESOLUTION,
        )
    )

    del X_base
    gc.collect()

# ------------------------------------------------------------------------------
# 10. Save metrics
# ------------------------------------------------------------------------------

count_metrics_df = pd.DataFrame(count_metric_rows)
clustering_metrics_df = pd.DataFrame(clustering_rows)

count_metrics_df.to_csv(COUNT_METRICS_PATH, index=False)
clustering_metrics_df.to_csv(CLUSTERING_METRICS_PATH, index=False)

print("\n" + "=" * 90)
print("COUNT MATRIX METRICS")
print("=" * 90)
display(count_metrics_df)

print("\n" + "=" * 90)
print("CLUSTERING METRICS — STEP4-MATCHED METHOD")
print("=" * 90)
display(clustering_metrics_df)

print(f"\nSaved count matrix metrics:")
print(f"  {COUNT_METRICS_PATH}")

print(f"Saved clustering metrics:")
print(f"  {CLUSTERING_METRICS_PATH}")

# ------------------------------------------------------------------------------
# 11. Print Step4-style detailed comparison
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("STEP4-MATCHED CLUSTERING COMPARISON")
print("=" * 90)

baseline_raw = clustering_metrics_df[
    clustering_metrics_df["dataset"] == "Raw observed counts"
].iloc[0]

print(f"{'Dataset':45s} {'ARI':>10s} {'NMI':>10s} {'Silhouette':>12s} {'Clusters':>10s}")
print("-" * 95)

for _, row in clustering_metrics_df.iterrows():
    print(
        f"{row['dataset'][:45]:45s} "
        f"{row['ARI']:10.4f} "
        f"{row['NMI']:10.4f} "
        f"{row['silhouette']:12.4f} "
        f"{int(row['n_clusters']):10d}"
    )

print("\nChange relative to Raw observed counts:")
print(f"{'Dataset':45s} {'ΔARI':>10s} {'ΔNMI':>10s} {'ΔSilhouette':>14s} {'ΔClusters':>10s}")
print("-" * 100)

for _, row in clustering_metrics_df.iterrows():
    if row["dataset"] == "Raw observed counts":
        continue

    print(
        f"{row['dataset'][:45]:45s} "
        f"{row['ARI'] - baseline_raw['ARI']:10.4f} "
        f"{row['NMI'] - baseline_raw['NMI']:10.4f} "
        f"{row['silhouette'] - baseline_raw['silhouette']:14.4f} "
        f"{int(row['n_clusters']) - int(baseline_raw['n_clusters']):10d}"
    )

# ------------------------------------------------------------------------------
# 12. Save report
# ------------------------------------------------------------------------------

print("\nWriting downstream validation report...")

with open(REPORT_PATH, "w") as f:
    f.write("9E DOWNSTREAM VALIDATION REPORT — STEP4-MATCHED CLUSTERING METHOD\n")
    f.write("=" * 90 + "\n\n")

    f.write("This report uses the same clustering-quality setup as the Step 4 denoising validation cell:\n")
    f.write(f"- sample_size: {SAMPLE_SIZE}\n")
    f.write(f"- random_state: {RANDOM_STATE}\n")
    f.write("- normalization: scanpy.pp.normalize_total(target_sum=1e4) + scanpy.pp.log1p\n")
    f.write(f"- PCA: scanpy.pp.pca, n_pcs={N_PCS}\n")
    f.write(f"- Leiden: resolution={LEIDEN_RESOLUTION}\n")
    f.write("- ARI/NMI: true cell type labels vs Leiden clusters\n")
    f.write("- Silhouette: PCA coordinates vs true cell type labels\n\n")

    f.write(f"RUN_NAME: {RUN_NAME}\n")
    f.write(f"Cell-type column: {ct_col}\n")
    f.write(f"Sample index path: {SAMPLE_INDEX_PATH}\n\n")

    f.write("COUNT MATRIX METRICS\n")
    f.write("-" * 90 + "\n")
    f.write(count_metrics_df.to_string(index=False))
    f.write("\n\n")

    f.write("CLUSTERING METRICS\n")
    f.write("-" * 90 + "\n")
    f.write(clustering_metrics_df.to_string(index=False))
    f.write("\n\n")

    f.write("CHANGE RELATIVE TO RAW OBSERVED COUNTS\n")
    f.write("-" * 90 + "\n")

    for _, row in clustering_metrics_df.iterrows():
        if row["dataset"] == "Raw observed counts":
            continue

        f.write(
            f"{row['dataset']}: "
            f"ΔARI={row['ARI'] - baseline_raw['ARI']:.4f}, "
            f"ΔNMI={row['NMI'] - baseline_raw['NMI']:.4f}, "
            f"Δsilhouette={row['silhouette'] - baseline_raw['silhouette']:.4f}, "
            f"Δclusters={int(row['n_clusters']) - int(baseline_raw['n_clusters'])}\n"
        )

print(f"Saved downstream validation report:")
print(f"  {REPORT_PATH}")

print("\n" + "=" * 90)
print("9E DOWNSTREAM VALIDATION COMPLETE — STEP4-MATCHED")
print("=" * 90)

print("Saved outputs:")
print(f"  Count matrix metrics   : {COUNT_METRICS_PATH}")
print(f"  Clustering metrics     : {CLUSTERING_METRICS_PATH}")
print(f"  Fixed sample index     : {SAMPLE_INDEX_PATH}")
print(f"  Report                 : {REPORT_PATH}")

In [ ]:
# ==============================================================================
# FINAL SAFETY CHECKPOINT CELL — 9E Step 5 / imputation / downstream
# Purpose:
#   1. Verify all important files are saved in Google Drive.
#   2. Check key row counts / count reconciliation.
#   3. Save a complete manifest of files, sizes, row counts, and runtime variables.
#   4. Create a final "safe to disconnect" report.
#
# GPU not needed.
# ==============================================================================

import os
import gc
import json
import time
import glob
import platform
from datetime import datetime

import numpy as np
import pandas as pd

print("=" * 100)
print("FINAL SAFETY CHECKPOINT — STEP 5 9E")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Paths / run name
# ------------------------------------------------------------------------------

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"
STEP4_EXPORT_DIR = "/content/drive/MyDrive/diffusion/step4_exports"

RUN_NAME = "attempt_9E_distribution_empirical_baselines"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

MANIFEST_JSON_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_FINAL_MANIFEST_{timestamp}.json"
)

MANIFEST_CSV_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_FINAL_FILE_MANIFEST_{timestamp}.csv"
)

SAFE_REPORT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_SAFE_TO_DISCONNECT_REPORT_{timestamp}.txt"
)

LATEST_SAFE_REPORT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_SAFE_TO_DISCONNECT_REPORT_LATEST.txt"
)

print(f"CHECKPOINT_DIR : {CHECKPOINT_DIR}")
print(f"STEP4_EXPORT_DIR: {STEP4_EXPORT_DIR}")
print(f"RUN_NAME       : {RUN_NAME}")
print(f"Timestamp      : {timestamp}")

# ------------------------------------------------------------------------------
# 1. File helpers
# ------------------------------------------------------------------------------

def file_size_gb(path):
    return os.path.getsize(path) / 1e9 if os.path.exists(path) else np.nan


def get_parquet_metadata(path):
    """
    Fast Parquet metadata reader. Does not load full table into RAM.
    """
    meta = {
        "parquet_rows": np.nan,
        "parquet_columns": np.nan,
        "parquet_error": "",
    }

    if not os.path.exists(path):
        meta["parquet_error"] = "file_missing"
        return meta

    try:
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(path)
        meta["parquet_rows"] = int(pf.metadata.num_rows)
        meta["parquet_columns"] = int(pf.metadata.num_columns)
    except Exception as e:
        meta["parquet_error"] = str(e)

    return meta


def check_file(path, category, required=True):
    exists = os.path.exists(path)

    row = {
        "category": category,
        "required": bool(required),
        "exists": bool(exists),
        "path": path,
        "size_gb": file_size_gb(path) if exists else np.nan,
        "parquet_rows": np.nan,
        "parquet_columns": np.nan,
        "notes": "",
    }

    if exists and path.endswith(".parquet"):
        meta = get_parquet_metadata(path)
        row["parquet_rows"] = meta["parquet_rows"]
        row["parquet_columns"] = meta["parquet_columns"]
        row["notes"] = meta["parquet_error"]

    return row


# ------------------------------------------------------------------------------
# 2. Important expected files
# ------------------------------------------------------------------------------

critical_files = []

# Step 4 exports needed to reproduce / reload Step 5.
for fname in [
    "molecules.parquet",
    "cell_data.npz",
    "denoised_adata.h5ad",
    "was_corrected.npy",
    "step4_config.json",
]:
    critical_files.append((
        os.path.join(STEP4_EXPORT_DIR, fname),
        "Step4 corrected exports",
        True,
    ))

# Step 5 processed molecule checkpoints.
for fname in [
    "checkpoint_mol_after_5A.parquet",
    "cell_polygons.pkl",
    "nuc_polygons.pkl",
    "cell_areas.pkl",
    "nuc_areas.pkl",
    "nuc_centroids.pkl",
    "step5_mol_processed.parquet",
]:
    critical_files.append((
        os.path.join(CHECKPOINT_DIR, fname),
        "Step5 preprocessing/checkpoints",
        True,
    ))

# 9E model / training / evaluation artifacts.
for fname in [
    f"{RUN_NAME}_best_coordNN_model.pt",
    f"{RUN_NAME}_final_large_recovery_metrics.csv",
    f"{RUN_NAME}_final_large_recovery_report.txt",
]:
    critical_files.append((
        os.path.join(CHECKPOINT_DIR, fname),
        "9E trained model/final evaluation",
        True,
    ))

# 9E learned imputation outputs.
for fname in [
    f"{RUN_NAME}_imputation_targets.csv",
    f"{RUN_NAME}_geometry_cache.pkl",
    f"{RUN_NAME}_imputed_records.parquet",
    f"{RUN_NAME}_completed_molecule_table.parquet",
    f"{RUN_NAME}_count_reconciliation.csv",
    f"{RUN_NAME}_imputation_summary.csv",
]:
    critical_files.append((
        os.path.join(CHECKPOINT_DIR, fname),
        "9E learned imputation outputs",
        True,
    ))

# 3 empirical baseline imputation outputs.
for fname in [
    f"{RUN_NAME}_gene_emp_imputed_records.parquet",
    f"{RUN_NAME}_ct_gene_emp_imputed_records.parquet",
    f"{RUN_NAME}_spatial_knn_emp_imputed_records.parquet",
    f"{RUN_NAME}_baseline_imputation_summary.csv",
]:
    critical_files.append((
        os.path.join(CHECKPOINT_DIR, fname),
        "Empirical baseline imputation outputs",
        True,
    ))

# Downstream outputs.
for fname in [
    f"{RUN_NAME}_downstream_count_matrix_metrics_step4matched.csv",
    f"{RUN_NAME}_downstream_clustering_metrics_step4matched.csv",
    f"{RUN_NAME}_downstream_step4matched_sample_idx.npy",
    f"{RUN_NAME}_downstream_validation_report_step4matched.txt",
]:
    critical_files.append((
        os.path.join(CHECKPOINT_DIR, fname),
        "Downstream validation outputs",
        True,
    ))

# Optional artifacts: include if present, do not fail if absent.
optional_globs = [
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*training*curve*.png"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*curve*.png"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*checkpoint*.pt"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*report*.json"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*report*.txt"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*metrics*.csv"),
]

optional_files = []
for pattern in optional_globs:
    optional_files.extend(glob.glob(pattern))

optional_files = sorted(set(optional_files))

# Avoid duplicate required files.
required_paths_set = set(p for p, _, _ in critical_files)

for path in optional_files:
    if path not in required_paths_set:
        critical_files.append((path, "Optional discovered 9E artifacts", False))

# ------------------------------------------------------------------------------
# 3. Check file existence / sizes / row counts
# ------------------------------------------------------------------------------

print("\nChecking important files in Drive...")

file_rows = []

for path, category, required in critical_files:
    file_rows.append(check_file(path, category, required=required))

file_manifest_df = pd.DataFrame(file_rows)

required_missing = file_manifest_df[
    (file_manifest_df["required"] == True)
    & (file_manifest_df["exists"] == False)
].copy()

print("\nFile manifest:")
display(file_manifest_df)

file_manifest_df.to_csv(MANIFEST_CSV_PATH, index=False)

print(f"\nSaved file manifest CSV:")
print(f"  {MANIFEST_CSV_PATH}")

# ------------------------------------------------------------------------------
# 4. Check important count reconciliation files
# ------------------------------------------------------------------------------

print("\nChecking count reconciliation and summaries...")

check_results = {
    "count_reconciliation_ok": None,
    "count_reconciliation_mismatched_pairs": None,
    "count_reconciliation_max_abs_diff": None,
    "imputation_summary_ok": None,
    "baseline_summary_ok": None,
    "downstream_metrics_ok": None,
}

COUNT_CHECK_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_count_reconciliation.csv"
)

IMPUTATION_SUMMARY_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_imputation_summary.csv"
)

BASELINE_SUMMARY_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_baseline_imputation_summary.csv"
)

DOWNSTREAM_CLUSTERING_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_downstream_clustering_metrics_step4matched.csv"
)

DOWNSTREAM_COUNT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_downstream_count_matrix_metrics_step4matched.csv"
)

FINAL_RECOVERY_METRICS_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_final_large_recovery_metrics.csv"
)

# Count reconciliation
if os.path.exists(COUNT_CHECK_PATH):
    count_check = pd.read_csv(COUNT_CHECK_PATH)

    if "diff" in count_check.columns:
        n_bad = int((count_check["diff"] != 0).sum())
        max_abs_diff = int(count_check["diff"].abs().max()) if len(count_check) else 0
        check_results["count_reconciliation_ok"] = bool(n_bad == 0 and max_abs_diff == 0)
        check_results["count_reconciliation_mismatched_pairs"] = n_bad
        check_results["count_reconciliation_max_abs_diff"] = max_abs_diff

        print(f"Count reconciliation mismatched pairs: {n_bad:,}")
        print(f"Count reconciliation max abs diff    : {max_abs_diff}")
    else:
        check_results["count_reconciliation_ok"] = False
        print("WARNING: count_reconciliation.csv exists but has no 'diff' column.")
else:
    check_results["count_reconciliation_ok"] = False
    print("WARNING: count_reconciliation.csv missing.")

# Imputation summary
if os.path.exists(IMPUTATION_SUMMARY_PATH):
    imp_summary = pd.read_csv(IMPUTATION_SUMMARY_PATH)
    check_results["imputation_summary_ok"] = True

    print("\nImputation summary:")
    display(imp_summary)
else:
    check_results["imputation_summary_ok"] = False
    print("WARNING: imputation_summary.csv missing.")

# Baseline summary
if os.path.exists(BASELINE_SUMMARY_PATH):
    baseline_summary = pd.read_csv(BASELINE_SUMMARY_PATH)
    check_results["baseline_summary_ok"] = True

    print("\nBaseline imputation summary:")
    display(baseline_summary)
else:
    check_results["baseline_summary_ok"] = False
    print("WARNING: baseline_imputation_summary.csv missing.")

# Downstream outputs
downstream_ok = os.path.exists(DOWNSTREAM_CLUSTERING_PATH) and os.path.exists(DOWNSTREAM_COUNT_PATH)
check_results["downstream_metrics_ok"] = bool(downstream_ok)

if os.path.exists(DOWNSTREAM_CLUSTERING_PATH):
    clustering_df = pd.read_csv(DOWNSTREAM_CLUSTERING_PATH)
    print("\nDownstream clustering metrics:")
    display(clustering_df)
else:
    print("WARNING: downstream clustering metrics missing.")

if os.path.exists(DOWNSTREAM_COUNT_PATH):
    count_metrics_df = pd.read_csv(DOWNSTREAM_COUNT_PATH)
    print("\nDownstream count matrix metrics:")
    display(count_metrics_df)
else:
    print("WARNING: downstream count matrix metrics missing.")

# Final large recovery metrics
if os.path.exists(FINAL_RECOVERY_METRICS_PATH):
    final_recovery_df = pd.read_csv(FINAL_RECOVERY_METRICS_PATH)
    print("\nFinal large recovery metrics:")
    display(final_recovery_df)
else:
    final_recovery_df = None
    print("WARNING: final large recovery metrics missing.")

# ------------------------------------------------------------------------------
# 5. Fast row-count sanity checks for important parquet files
# ------------------------------------------------------------------------------

print("\nRunning fast Parquet row-count sanity checks...")

expected_rows = {
    f"{RUN_NAME}_imputed_records.parquet": 4_638_214,
    f"{RUN_NAME}_gene_emp_imputed_records.parquet": 4_638_214,
    f"{RUN_NAME}_ct_gene_emp_imputed_records.parquet": 4_638_214,
    f"{RUN_NAME}_spatial_knn_emp_imputed_records.parquet": 4_638_214,
    f"{RUN_NAME}_completed_molecule_table.parquet": 27_809_317,
    "step5_mol_processed.parquet": 23_171_103,
}

row_count_checks = []

for fname, expected in expected_rows.items():
    path = os.path.join(CHECKPOINT_DIR, fname)

    meta = get_parquet_metadata(path)
    actual = meta["parquet_rows"]

    ok = bool(os.path.exists(path) and pd.notna(actual) and int(actual) == int(expected))

    row_count_checks.append({
        "file": fname,
        "path": path,
        "expected_rows": int(expected),
        "actual_rows": int(actual) if pd.notna(actual) else np.nan,
        "ok": ok,
        "notes": meta["parquet_error"],
    })

row_count_df = pd.DataFrame(row_count_checks)

print("\nParquet row-count checks:")
display(row_count_df)

# ------------------------------------------------------------------------------
# 6. Runtime variable snapshot
# ------------------------------------------------------------------------------

print("\nCapturing runtime variable snapshot...")

def describe_var(name):
    if name not in globals():
        return {
            "name": name,
            "exists_in_runtime": False,
            "type": None,
            "shape_or_len": None,
            "notes": "",
        }

    obj = globals()[name]

    info = {
        "name": name,
        "exists_in_runtime": True,
        "type": type(obj).__name__,
        "shape_or_len": None,
        "notes": "",
    }

    try:
        if hasattr(obj, "shape"):
            info["shape_or_len"] = str(obj.shape)
        elif hasattr(obj, "__len__"):
            info["shape_or_len"] = str(len(obj))
    except Exception as e:
        info["notes"] = str(e)

    return info


important_runtime_vars = [
    "mol",
    "completed_mol",
    "imputed_df",
    "imputed_records",
    "imputation_targets_df",
    "imputation_targets",
    "X_raw_counts",
    "X_denoised",
    "was_corrected",
    "denoised_adata",
    "shared_genes",
    "cell_ids_step4",
    "cell_polygons",
    "nuc_polygons",
    "cell_areas",
    "nuc_areas",
    "nuc_centroids",
    "baseline_gene_pools",
    "baseline_ct_gene_pools",
    "baseline_spatial_index",
    "model",
    "device",
]

runtime_snapshot = [describe_var(v) for v in important_runtime_vars]
runtime_snapshot_df = pd.DataFrame(runtime_snapshot)

print("\nRuntime variable snapshot:")
display(runtime_snapshot_df)

# ------------------------------------------------------------------------------
# 7. Build final manifest JSON
# ------------------------------------------------------------------------------

print("\nWriting final manifest JSON/report...")

safe_required_files = bool(len(required_missing) == 0)
safe_count_reconciliation = bool(check_results["count_reconciliation_ok"] is True)
safe_row_counts = bool(row_count_df["ok"].fillna(False).all())
safe_downstream = bool(check_results["downstream_metrics_ok"] is True)

safe_to_disconnect = bool(
    safe_required_files
    and safe_count_reconciliation
    and safe_row_counts
    and safe_downstream
)

manifest = {
    "run_name": RUN_NAME,
    "timestamp": timestamp,
    "checkpoint_dir": CHECKPOINT_DIR,
    "step4_export_dir": STEP4_EXPORT_DIR,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "safe_to_disconnect": safe_to_disconnect,
    "safety_checks": {
        "required_files_present": safe_required_files,
        "count_reconciliation_ok": safe_count_reconciliation,
        "parquet_row_counts_ok": safe_row_counts,
        "downstream_metrics_present": safe_downstream,
    },
    "check_results": check_results,
    "required_missing_files": required_missing.to_dict(orient="records"),
    "file_manifest_csv": MANIFEST_CSV_PATH,
    "safe_report_path": SAFE_REPORT_PATH,
    "latest_safe_report_path": LATEST_SAFE_REPORT_PATH,
    "row_count_checks": row_count_df.to_dict(orient="records"),
    "runtime_snapshot": runtime_snapshot,
}

with open(MANIFEST_JSON_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

# ------------------------------------------------------------------------------
# 8. Write human-readable safety report
# ------------------------------------------------------------------------------

report_lines = []

report_lines.append("=" * 100)
report_lines.append("FINAL SAFETY CHECKPOINT REPORT — STEP 5 9E")
report_lines.append("=" * 100)
report_lines.append(f"Run name: {RUN_NAME}")
report_lines.append(f"Timestamp: {timestamp}")
report_lines.append(f"Checkpoint dir: {CHECKPOINT_DIR}")
report_lines.append(f"Step4 export dir: {STEP4_EXPORT_DIR}")
report_lines.append("")
report_lines.append("SAFETY CHECKS")
report_lines.append("-" * 100)
report_lines.append(f"Required files present      : {safe_required_files}")
report_lines.append(f"Count reconciliation OK     : {safe_count_reconciliation}")
report_lines.append(f"Parquet row counts OK       : {safe_row_counts}")
report_lines.append(f"Downstream metrics present  : {safe_downstream}")
report_lines.append(f"SAFE TO DISCONNECT          : {'YES' if safe_to_disconnect else 'NO'}")
report_lines.append("")

report_lines.append("KEY EXPECTED OUTPUTS")
report_lines.append("-" * 100)
for fname in [
    f"{RUN_NAME}_best_coordNN_model.pt",
    f"{RUN_NAME}_final_large_recovery_metrics.csv",
    f"{RUN_NAME}_imputed_records.parquet",
    f"{RUN_NAME}_completed_molecule_table.parquet",
    f"{RUN_NAME}_gene_emp_imputed_records.parquet",
    f"{RUN_NAME}_ct_gene_emp_imputed_records.parquet",
    f"{RUN_NAME}_spatial_knn_emp_imputed_records.parquet",
    f"{RUN_NAME}_downstream_clustering_metrics_step4matched.csv",
    f"{RUN_NAME}_downstream_count_matrix_metrics_step4matched.csv",
]:
    p = os.path.join(CHECKPOINT_DIR, fname)
    report_lines.append(
        f"{'✓' if os.path.exists(p) else '✗'} {fname} "
        f"({file_size_gb(p):.3f} GB)" if os.path.exists(p) else f"✗ {fname}"
    )

report_lines.append("")
report_lines.append("ROW COUNT CHECKS")
report_lines.append("-" * 100)
report_lines.append(row_count_df.to_string(index=False))

if len(required_missing) > 0:
    report_lines.append("")
    report_lines.append("MISSING REQUIRED FILES")
    report_lines.append("-" * 100)
    report_lines.append(required_missing.to_string(index=False))

report_lines.append("")
report_lines.append("MANIFEST FILES")
report_lines.append("-" * 100)
report_lines.append(f"Manifest JSON: {MANIFEST_JSON_PATH}")
report_lines.append(f"File manifest CSV: {MANIFEST_CSV_PATH}")
report_lines.append(f"Safety report: {SAFE_REPORT_PATH}")
report_lines.append("")

report_text = "\n".join(report_lines)

with open(SAFE_REPORT_PATH, "w") as f:
    f.write(report_text)

with open(LATEST_SAFE_REPORT_PATH, "w") as f:
    f.write(report_text)

print(report_text)

print("\nSaved final manifest files:")
print(f"  JSON manifest      : {MANIFEST_JSON_PATH}")
print(f"  File manifest CSV  : {MANIFEST_CSV_PATH}")
print(f"  Safety report      : {SAFE_REPORT_PATH}")
print(f"  Latest report copy : {LATEST_SAFE_REPORT_PATH}")

# ------------------------------------------------------------------------------
# 9. Final verdict
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)

if safe_to_disconnect:
    print("SAFE TO DISCONNECT: YES")
    print("All critical files, row counts, count reconciliation, and downstream outputs are saved.")
    print("You can disconnect/delete the runtime now.")
else:
    print("SAFE TO DISCONNECT: NO")
    print("One or more required checks failed. Review the warnings above before disconnecting.")

print("=" * 100)

gc.collect()